# Sugestão de endereço canônico por CPF — base `tb_endereco` (10 bi de linhas) com apoio da base dos Correios

> **Leia esta célula inteira antes de mexer em qualquer coisa.** Ela foi escrita para que outra IA (ou outra pessoa) consiga entender, revisar, corrigir e estender o notebook sem precisar de contexto adicional. Tudo que depende do ambiente (catálogo, schemas, nomes de tabelas e colunas, provedor/modelo/segredo do LLM) está **numa única célula, ⚙️ Configuração de ambiente**, em blocos `dev`/`prod` selecionados por `AMBIENTE_ATIVO`; os parâmetros operacionais ficam na célula **Parâmetros**. Ambos marcados com `# >>> AJUSTAR`.

---

## 1. Problema

A tabela `cadastro.tb_endereco` (~10 bilhões de linhas) guarda, **linha a linha**, os endereços cadastrados dos clientes. Para um mesmo `idCPF` existem dezenas de linhas que, na maior parte dos casos, são **o mesmo endereço escrito de formas diferentes**:

| Sintoma observado na base | Exemplo |
|---|---|
| Endereço inteiro dentro de `logradouro`, outras linhas com os campos separados | `"RUA ADINEI EMIDIO DE ALMEIDA, 123 APTO 45 BL B"` vs `logradouro="R ADINEI E DE ALMEIDA"`, `numero="123"`, `complemento="AP 45 BLOCO B"` |
| Linhas idênticas com CEP diferente (aleatório, mal fatiado ou sem zero à esquerda) | `cep_parte1="1310"`, `cep_parte2="100"` (era `01310-100`); `cep_parte1="01310100"`, `cep_parte2=NULL` |
| Tipo de logradouro presente/omitido/abreviado | `"RUA X"`, `"R X"`, `"R. X"`, `"X"` |
| Nomes abreviados no meio (padrão "reduzido" dos Correios) | `"ADINEI EMIDIO DE ALMEIDA"` vs `"ADINEI E DE ALMEIDA"` |
| Campos nulos em uma linha e preenchidos em outra | `complemento=NULL` vs `complemento="APTO 45"` |
| Acentos, caixa, pontuação, espaços duplicados | `"São João"`, `"SAO JOAO"`, `"S. JOAO"` |
| Número embutido no logradouro, com ou sem marcador | `"RUA X N 123"`, `"RUA X, 123"`, `"RUA X Nº123"`, `"RUA X 123"` |
| Sem número | `"S/N"`, `"SN"`, `"SEM NUMERO"`, `"0"` |
| Bairro/cidade/CEP embutidos no logradouro | `"RUA X 123 - CENTRO - SAO PAULO SP 01310-100"` |

**Objetivo:** para **cada linha original** produzir uma sugestão do endereço "correto" (canônico), com base (a) na massa de variantes do próprio CPF e (b) na base de referência dos Correios (`tb_correios`), entregando a tabela original **mais** as mesmas colunas com sufixo `_sugestao`, além de colunas de diagnóstico (cluster, score de confiança, método, flags).

## 2. Entradas

### 2.1 `tb_endereco` (10 bi de linhas)

| Coluna | Tipo esperado | Observação |
|---|---|---|
| `idCPF` | string/long | identificador do cliente |
| `logradouro` | string livre | pode conter tipo, nome, número, complemento, bairro, cidade, UF e CEP misturados |
| `numero` | string | pode vir `"123"`, `"123A"`, `"S/N"`, `NULL`, `"0"` |
| `complemento` | string | livre |
| `bairro`, `cidade`, `uf` | string | livres |
| `cep_parte1` | string | 5 primeiros dígitos do CEP (pode vir errado: 8 dígitos, 4 dígitos, etc.) |
| `cep_parte2` | string | 3 últimos dígitos (pode vir nulo, com menos dígitos, etc.) |
| *(opcional)* coluna de data | timestamp/date | se existir (`COL_DATA`), é usada como peso de recência; se não existir, deixe `None` |

O nome real das colunas é mapeado em `COLS_ENDERECO` (célula Parâmetros). Ex.: se a coluna se chama `cep_parte 1` (com espaço), ajuste lá — o notebook renomeia tudo para nomes internos padronizados.

### 2.2 `tb_correios` (base de apoio — DNE dos Correios ou equivalente)

| Coluna | Observação |
|---|---|
| `cep_parte1`, `cep_parte2` | CEP fatiado |
| `logradouro_correios` | nome do logradouro **sem** o tipo (ex.: `"ADINEI EMIDIO DE ALMEIDA"`) — se vier com tipo na frente o notebook trata |
| `logradouro_reduzido` *(opcional)* | forma reduzida oficial (ex.: `"AV ADINEI E DE ALMEIDA"`) |
| `tipo` | abreviação do tipo (`R`, `AV`, `TV`, ...) — existem ~360 tipos; os mais comuns estão mapeados em `MAPA_TIPOS_CORREIOS`, os demais ficam com a própria abreviação como canônico (a lista dos não mapeados é impressa na etapa E3) |
| `lado` | `P` (par), `I` (ímpar), outros/nulo = ambos os lados |
| `uf`, `municipio`, `bairro` | referência oficial |
| *(opcionais)* `numero_inicial`, `numero_final` | faixa numérica do CEP, se sua extração do DNE tiver |

**Como a base dos Correios é usada corretamente:**
1. **Nunca** como "verdade absoluta" sobre o cliente: ela valida/corrige **CEP ↔ logradouro ↔ bairro ↔ município ↔ UF**, mas número e complemento só vêm do próprio cliente.
2. Um CEP pode ter **mais de uma linha** (lado par/ímpar, faixas numéricas). O notebook agrega por CEP (`_dim_correios_cep`) e usa a paridade do número do cliente para escolher o lado quando existe mais de um.
3. Municípios pequenos têm **CEP único de localidade** (linhas sem logradouro). Nesses casos o CEP é validado, bairro/município/UF são preenchidos, e o logradouro fica por conta do consenso do CPF.
4. O logradouro dos Correios é comparado em **três representações**: chave por tokens (sem stopwords, ordenada), chave fonética (algoritmo próprio para pt-BR), e forma **reduzida** (1º e último nome inteiros, nomes do meio abreviados para a inicial — mesma regra do DNE). Isso resolve o caso `"ADINEI EMIDIO DE ALMEIDA"` ⇄ `"ADINEI E DE ALMEIDA"`.
5. Quando o cliente tem o nome **completo** e os Correios só o reduzido, a sugestão mantém o nome completo do cliente (mais informativo) se a forma reduzida de ambos coincidir.

## 3. Saída

Tabela `<PREFIXO_SAIDA>_final` (Delta, Liquid Clustering por `idCPF`) com:

- **Todas as colunas originais**, intactas, com os nomes originais.
- `logradouro_sugestao`, `numero_sugestao`, `complemento_sugestao`, `bairro_sugestao`, `cidade_sugestao`, `uf_sugestao`, `cep_parte1_sugestao`, `cep_parte2_sugestao`.
- Diagnóstico: `endereco_sugestao_completo` (string única legível), `cluster_id` (grupo "mesmo endereço" dentro do CPF), `unidade` (pares de complemento que definem o cluster), `n_unidades_local` (quantos endereços distintos este cliente tem no mesmo prédio/lote), `qtd_variantes_cluster`, `qtd_ocorrencias_cluster`, `score_confianca_sugestao` (0–100), `metodo_sugestao` (`CORREIOS_CEP`, `CORREIOS_FUZZY`, `CONSENSO_INTERNO`, `VARIANTE_UNICA`, `LLM`, `SEM_SUGESTAO`), **`grau_certeza`** (`A_ALTA` / `B_MEDIA` / `C_BAIXA` / `D_REVISAO_HUMANA`), **`requer_revisao_humana`** (booleano = grau D), **`motivos_revisao`** (array com as razões que rebaixaram o grau), `flags_sugestao` (array de strings com tudo que foi corrigido/detectado), `flag_alterou_<campo>` (booleanos).

### Escala de grau de certeza (para decidir o que aplicar automaticamente e o que mandar para revisão humana)

| Grau | Significado | Uso sugerido |
|---|---|---|
| `A_ALTA` | score ≥ `LIMIAR_GRAU_ALTA` (85), CEP respaldado pelos Correios, sem nenhuma restrição abaixo | aplicar automaticamente |
| `B_MEDIA` | score ≥ `LIMIAR_GRAU_MEDIA` (65) **ou** rebaixado por: prédio com 2+ unidades deste cliente, sem número (S/N), número desta linha preenchido a partir do cluster (a linha era S/N), consenso interno forte, LLM aplicado | aplicar com amostragem de conferência |
| `C_BAIXA` | score < 65 **ou** rebaixado por: CEP sem validação nos Correios, CEP ausente, conflito de número nas variantes, variante única sem Correios, consenso interno fraco, LLM consultado com baixa confiança | não aplicar em processos críticos; fila de revisão de baixa prioridade |
| `D_REVISAO_HUMANA` | qualquer condição crítica: complemento ambíguo (linha sem complemento num prédio com 2+ unidades), fuzzy ambíguo sem CEP validado, CEP que contradiz o endereço, concordância muito baixa de logradouro ou número, sem sugestão, **ou, no nível da linha**: a sugestão troca o número desta linha, troca o logradouro sem respaldo dos Correios/LLM, ou troca o CEP por outro não validado | fila de revisão humana obrigatória |

O grau é calculado por cluster em E7 (e recalculado em E8 se o LLM sobrescreveu) e **rebaixado por linha em E9** quando a sugestão contradiz o que aquela linha específica dizia. O pior nível vence. `motivos_revisao` lista as razões (ex.: `["MULTIPLAS_UNIDADES_NO_LOCAL", "NUMERO_ALTERADO"]`). `_metricas` guarda a distribuição por grau e `pct_revisao_humana`.

Tabelas intermediárias (todas com o prefixo `PREFIXO_SAIDA`), úteis para auditoria e para **reiniciar de uma etapa** sem refazer as anteriores:

| Sufixo | Conteúdo | Granularidade |
|---|---|---|
| `_01_variantes` | tuplas distintas (idCPF + campos brutos) com contagem de ocorrências | variante |
| `_02_normalizado` | variantes com todos os campos parseados/normalizados + chaves | variante |
| `_dim_correios_cep` | 1 linha por CEP (8 dígitos) com atributos agregados | CEP |
| `_dim_correios_logradouro` | 1 linha por (UF, município, tipo, logradouro) com lista de CEPs | logradouro |
| `_03_validado` | variantes enriquecidas com Correios via CEP + score de qualidade | variante |
| `_04_fuzzy_correios` | consultas fuzzy (UF+município+logradouro) → melhor logradouro Correios | consulta distinta |
| `_04b_variantes_enriquecidas` | variantes + resultado do fuzzy | variante |
| `_05_grupos_locais` | variantes com `grupo_local` (E6, nível 1: rua+número) | variante |
| `_05_clusters` | variantes com `cluster_id` = `grupo_local#unidade` (E6b, nível 2: complemento compatível) | variante |
| `_05b_clusters_consolidados` | mapa linha→cluster após a consolidação E7b (**é o que E9 usa**) | variante |
| `_06a_sugestao_pass1` / `_06a_consolidacao` | 1ª eleição e mapa de clusters fundidos (auditoria de E7b) | cluster |
| `_06_sugestao_cluster` | 1 sugestão por cluster (final, antes do LLM) | cluster |
| `_06b_sugestao_cluster_llm` | sugestões após sobrescrita pelo LLM (só se `USAR_LLM`) | cluster |
| `_07_llm_cache` | cache de chamadas ao LLM (chave = hash do prompt) | prompt |
| `_final` | saída (linhas originais + `_sugestao`) | linha original |
| `_metricas` | métricas da execução | execução |

Quando `N_LOTES > 1`, as intermediárias recebem o sufixo `_loteNN` e a `_final` recebe `append` a partir do lote 1.

## 4. Arquitetura em camadas (ordem de execução)

```
 tb_endereco (10 bi)                                   tb_correios
      |                                                    |
      v E1  amostragem / lotes por hash(idCPF)             v E3  normalização + dims (CEP / logradouro)
      v E1  dedup -> variantes (idCPF + campos brutos, qtd) |
      v E2  normalização e PARSING (Spark nativo, sem UDF)  |
      |      . acentos/caixa/pontuação                      |
      |      . CEP: remonta, corrige zero à esquerda, valida faixa por UF
      |      . logradouro: tira CEP/bairro/cidade embutidos, extrai número e complemento
      |      . tipo de logradouro canônico, títulos expandidos (DR->DOUTOR...)
      |      . chaves: tokens ordenados, fonética pt-BR, forma reduzida
      |      . número: dígitos / SN / sufixo de letra; complemento: pares (APTO 45, BLOCO B)
      v E4  validação via CEP (join com dim_correios_cep) + similaridade de logradouro + score de qualidade
      v E5  fallback FUZZY contra Correios (blocking por UF+município+fonética/1º token/reduzido; rapidfuzz nos pares)
      v E6  CLUSTERING intra-CPF nível 1 = grupo_local (rua+número): união por múltiplas chaves, propagação de rótulo mínimo
      v E6b CLUSTERING nível 2 = unidade: divide o grupo_local por compatibilidade de complemento (APTO 45 x APTO 46 = clusters distintos)
      v E7  ELEIÇÃO do endereço canônico por cluster (voto ponderado por campo + Correios como autoridade de CEP/bairro/município/UF)
      v E7b CONSOLIDAÇÃO: grupos locais com a mesma sugestão de local são fundidos, as unidades refeitas e os afetados reeleitos
      v E8  fallback LLM (amostra de clusters de baixa confiança; cache em Delta; Anthropic / OpenAI / Databricks)
      v E9  montagem final (join de volta nas linhas originais) + métricas
```

### Por que assim (decisões de projeto)

- **Tudo que roda nos 10 bi é Spark nativo** (`regexp_*`, `translate`, `split`, `transform`, `array_*`, `xxhash64`, janelas). Nenhuma UDF Python toca a base completa. `rapidfuzz` (pandas UDF) só roda sobre **pares candidatos distintos** da etapa fuzzy (milhões, não bilhões). O LLM só roda numa **amostra** limitada por `LIMITE_LLM`.
- **Dedup antes de tudo**: as 10 bi de linhas são reduzidas a variantes distintas por CPF (tupla dos campos brutos) com `qtd_ocorrencias`. Todo o processamento pesado acontece nas variantes; no final, um join por `hash_linha` devolve a sugestão para cada linha original.
- **Cada etapa persiste em Delta** (`PERSISTIR_INTERMEDIARIOS=True`) e a próxima lê da tabela persistida: é possível reexecutar só a etapa que quebrou (`ETAPAS_A_EXECUTAR`) e auditar cada camada.
- **Lotes por hash do CPF** (`N_LOTES`/`LOTE_ATUAL`): permite processar 10 bi em, por exemplo, 10 rodadas de 1 bi (append na saída), dentro da capacidade de um cluster de workers 32 GB / 8 cores.
- **Clustering por união de chaves** em vez de comparar todas as variantes entre si: chaves `CEP+número`, `logradouro(chave)+número`, `fonética+número`, `reduzido+número`, `logradouro Correios canônico+número`, `CEP5+número+1º token`, `anagrama do nome+UF+número`. Se duas variantes compartilham qualquer chave, entram no mesmo cluster (propagação iterativa do menor rótulo — equivalente a componentes conexas). O **número** faz parte de todas as chaves para não fundir endereços diferentes na mesma rua; variantes sem número (nulo, `S/N`, `0`) são anexadas depois, só quando o CPF tem um único cluster numerado com aquele logradouro (ou aquele CEP).
- **Resultado do teste sintético** (300 CPFs, ~6.200 linhas, ~390 endereços verdadeiros, 66 CPFs com dois apartamentos no mesmo prédio, todas as patologias): fusões indevidas entre linhas numeradas 0; pureza dos clusters 99,8 %; completude excluindo linhas ambíguas 99,3 %; acerto de CEP/logradouro/bairro/cidade/UF 100 %, número 99,7 %, complemento (linhas não ambíguas) 100 %; 8,5 % das linhas ficaram `COMPLEMENTO_AMBIGUO` (sem complemento num prédio com 2 unidades — decisão conservadora). Grau de certeza: ~62 % `A_ALTA`, ~29 % `B_MEDIA` (quase tudo por "prédio com 2+ unidades" e "número preenchido a partir do cluster", proporções exageradas no sintético), ~8,5 % `D_REVISAO_HUMANA` (as ambíguas). A única anexação S/N errada (linha `S/N` de outro endereço do cliente na mesma rua) sai como `B_MEDIA` com `NUMERO_PREENCHIDO_PELO_CLUSTER`. Não é garantia para a base real — é a prova de que o encanamento funciona de ponta a ponta.
- **Unidades dentro do mesmo local (E6b)**: rua+número identificam o **prédio/lote**, não o endereço. Um cliente pode ter dois apartamentos no mesmo prédio (`APTO 45` e `APTO 46`), duas salas, dois lotes na mesma quadra, a casa da frente e a dos fundos. Por isso o `grupo_local` de E6 é dividido por **compatibilidade de complemento**: pares distintivos (`APTO`, `BLOCO`, `TORRE`, `CASA`, `SALA`, `LOTE`, `QUADRA`, `KM`, posição `FUNDOS`/`FRENTE`...) com valores diferentes no mesmo tipo **nunca** se fundem; ausência é compatível; linhas sem complemento num prédio com 2+ unidades ficam num cluster próprio **sem complemento sugerido** e com a flag `COMPLEMENTO_AMBIGUO` (o processo não chuta o apartamento). O cluster final é `grupo_local#unidade`.
- **Eleição por campo** (não "a melhor linha inteira"): cada campo é votado separadamente com peso `qtd_ocorrencias × (1 + score_qualidade/100) × recência`, permitindo juntar o CEP correto de uma linha com o complemento de outra.
- **Correios é autoridade para CEP/bairro/município/UF quando o logradouro bate**; para número/complemento a autoridade é o consenso do CPF.
- **Fonética própria em pt-BR** (regex nativo, inspirada no Metaphone-BR): trata `PH→F`, `CH/SH/SCH→X`, `SS/Z→S`, `LH→L`, `NH→N`, `QU→K`, `C(E,I)→S`, `C→K`, `G(E,I)→J`, `W→V`, `Y→I`, `H` mudo, vogais internas removidas. Não usar `soundex()` do Spark (é para inglês).

### Garantia de uniformidade (mesmo endereço ⇒ mesma sugestão em todas as linhas do cliente)

A sugestão **nunca é calculada linha a linha**. O caminho é: linha → `hash_linha` → variante → `cluster_id` → **uma** sugestão por cluster (E7) → distribuída de volta para cada linha por `cluster_id` (E9). Logo, todas as linhas de um cluster recebem valores idênticos nas colunas `_sugestao` por construção (as `flag_alterou_*` variam por linha porque comparam cada original com a sugestão). A única forma de duas linhas do mesmo endereço divergirem é caírem em clusters diferentes; contra isso existem três camadas:
1. **E6** liga variantes por 7 chaves (CEP, nome, fonética, reduzido, id Correios, CEP5+1º token, anagrama) e anexa as sem número; **E6b** separa unidades diferentes no mesmo local (complementos conflitantes nunca se fundem).
2. **E7b** funde grupos locais do mesmo CPF cuja **sugestão eleita** descreve o mesmo local, refaz as unidades e reelege (mesmo local + mesma unidade ⇒ mesmo `cluster_id`).
3. **E9** mede e grava em `_metricas` dois checks que têm de ser 0: clusters com mais de uma sugestão e CPFs com clusters distintos e sugestão idêntica. No modo sintético mede também `fusoes_indevidas` (clusters não ambíguos misturando dois endereços verdadeiros — tem de ser 0).
Toda a eleição é determinística (desempates por rank → peso → ordem alfabética; arrays ordenados), então reexecutar produz a mesma sugestão. Recomendação: rode E1 e E9 sobre a **mesma versão** da tabela de origem (ou fixe `VERSION AS OF` no Delta), porque E9 relê a origem e linhas novas sem cluster saem como `SEM_SUGESTAO`.

### Trocar de tabelas/colunas/API (dev → prod)

Tudo que é nome físico está na célula **⚙️ Configuração de ambiente** (`AMBIENTES["dev"]`, `AMBIENTES["prod"]`, ...). Para migrar: preencha o bloco `prod` e mude `AMBIENTE_ATIVO` (ou widget/env `END_AMBIENTE_ATIVO=prod`). `verificar_ambiente()` roda antes de qualquer etapa e interrompe cedo se uma tabela ou coluna mapeada não existir.

## 5. Como executar

| Cenário | Como |
|---|---|
| **Databricks (cluster DBR 18 LTS, Spark 4.1)** | Importe o `.ipynb` no workspace ou faça deploy via **Databricks Asset Bundle** (VS Code). Preencha os widgets (ou deixe os defaults) e execute célula a célula ou como job. O `%pip install` da 2ª célula instala `rapidfuzz`, `unidecode`, `anthropic`, `openai` no escopo do notebook. |
| **Bundle / VS Code (Python 3.12)** | O notebook detecta `dbutils`/`spark` quando executado no cluster via extensão. Os parâmetros podem vir por widget (`dbutils.widgets`), por variáveis de ambiente `END_<NOME_DO_PARAMETRO>` (ex.: `END_MODO_AMOSTRA=true`) ou por um JSON no widget `PARAMETROS_JSON`. |
| **Local (PySpark 4.x + Java 17)** | Sem Databricks: `spark` local é criado automaticamente; use `PERSISTIR_INTERMEDIARIOS=False` e `ESCREVER_SAIDA=False` (no Windows, escrita de tabelas exige winutils). Para um smoke test, `MODO_TESTE_SINTETICO=True` gera `tb_endereco`/`tb_correios` sintéticas com todas as patologias descritas acima e roda o pipeline inteiro sobre elas. |

### Ordem recomendada na primeira execução (base real)

1. `MODO_AMOSTRA=true`, `FRACAO_AMOSTRA=0.001` (0,1 % dos CPFs ≈ 10 mi de linhas), `USAR_LLM=false`. Rode tudo, olhe `_metricas` e as amostras de auditoria.
2. Ajuste dicionários (célula **Referências**) com o que aparecer nas listas "não mapeados" (tipos dos Correios, tokens de complemento, UFs).
3. Ative `USAR_LLM=true` com `LIMITE_LLM=500`, avalie custo/qualidade do fallback.
4. Rode a base completa em lotes: `MODO_AMOSTRA=false`, `N_LOTES=10`, `LOTE_ATUAL=0..9` (a saída usa `append` a partir do lote 1).

### Dimensionamento (referência, não regra)

- Etapas E2/E4 (normalização/validação): CPU-bound, escalam linearmente; 1 bi de variantes/lote ≈ 20–40 workers de 8 cores por 1–3 h.
- Etapa E6 (clustering com janelas por `idCPF`): a mais pesada em shuffle. `spark.sql.shuffle.partitions` é definido como `N_PARTICOES_SHUFFLE` (default 4000; ajuste para ≈ 2–3× o total de cores do cluster, ou deixe o AQE coalescer).
- `tb_correios` (≈1–2 mi de linhas) é **broadcast** nos joins por CEP; se sua versão for maior que ~500 MB, desligue `BROADCAST_CORREIOS`.

## 6. Parâmetros (resumo — detalhes e defaults na célula Parâmetros)

`CATALOGO`, `SCHEMA_ORIGEM`, `TB_ENDERECO`, `TB_CORREIOS`, `SCHEMA_SAIDA`, `PREFIXO_SAIDA`, `COLS_ENDERECO`, `COLS_CORREIOS`, `COL_DATA`, `MODO_AMOSTRA`, `FRACAO_AMOSTRA`, `LISTA_CPFS_AMOSTRA`, `N_LOTES`, `LOTE_ATUAL`, `PERSISTIR_INTERMEDIARIOS`, `ESCREVER_SAIDA`, `ETAPAS_A_EXECUTAR`, `CALCULAR_CONTAGENS`, `USAR_FUZZY_CORREIOS`, `USAR_RAPIDFUZZ`, `LIMIAR_SIM_LOGRADOURO`, `LIMIAR_FUZZY_CORREIOS`, `MAX_CANDIDATOS_FUZZY`, `USAR_LLM`, `LIMITE_LLM`, `LIMIAR_CONFIANCA_PARA_LLM`, `LIMIAR_CONFIANCA_LLM`, `PROVEDOR_LLM`, `MODELO_LLM`, `SEGREDO_SCOPE`, `SEGREDO_CHAVE_LLM`, `ENDPOINT_LLM`, `N_PARTICOES_SHUFFLE`, `BROADCAST_CORREIOS`, `MAX_ITER_CLUSTER`, `CHAVES_CLUSTER_DESATIVADAS`, `LIMIAR_GRAU_ALTA`, `LIMIAR_GRAU_MEDIA`, `MODO_TESTE_SINTETICO`.

## 7. Glossário de colunas internas (para ler o código)

- `*_raw`: valor original (renomeado). `*_n`: maiúsculas, sem acento/pontuação, espaços únicos.
- `log_core`: logradouro após remover CEP/bairro/cidade/UF embutidos, número e complemento.
- `tipo_canon`: tipo de logradouro canônico por extenso (`RUA`, `AVENIDA`, ...). `nome_log`: nome sem o tipo, com títulos expandidos (`DR`→`DOUTOR`).
- `nome_chave`: tokens de `nome_log` sem stopwords, ordenados, unidos por espaço (chave de igualdade).
- `nome_fon`: chave fonética pt-BR de `nome_log`. `nome_red`: forma reduzida (regra DNE); `nome_red_chave`: chave de tokens da forma reduzida.
- `numero_final`: número (string de dígitos, sem zeros à esquerda) ou `SN`; `numero_int` numérico (nulo se `SN`). `numero_sufixo`: letra colada ao número (`123A`→`A`).
- `compl_canon`: complemento com palavras-chave canonizadas; `compl_pares`: array de `"TIPO VALOR"` (`APTO 45`, `BLOCO B`, `FUNDOS`); `compl_livre`: sobra textual.
- `cep8`: CEP remontado com 8 dígitos ou nulo; `cep_origem`: de onde veio (`CAMPOS`, `P1_COMPLETO`, `ZEROS_CORRIGIDOS`, `LOGRADOURO`, ...). `cep_faixa_uf_ok`: CEP dentro da faixa da UF informada.
- `uf_n` pode ter sido **inferida** (flag `UF_INFERIDA_PELA_CIDADE`) quando a UF veio vazia e a cidade é um nome/sigla de UF ou um município de UF única no país (lista tirada da `tb_correios`).
- Flags importantes na saída: `CEP_SUGERIDO_NAO_VALIDADO` (o CEP sugerido é o majoritário bruto, sem respaldo dos Correios), `LLM_APLICADO`, `FUZZY_CEP_AMBIGUO` (logradouro achado nos Correios mas com mais de um CEP possível e sem número para decidir o lado).
- Prefixo `c_`: atributos dos Correios obtidos via CEP. Prefixo `fz_`: atributos dos Correios obtidos via fuzzy.
- `*_efetivo`: valor do campo após aplicar a autoridade dos Correios (o que entra na votação).
- `peso`: `qtd_ocorrencias × (1 + score_qualidade/100) × fator_recencia`.
- `rotulo`: rótulo de componente conexa dentro do CPF; `grupo_local = idCPF || '_' || rotulo` (mesma rua + mesmo número).
- `unidade`: pares distintivos de complemento que definem a unidade dentro do grupo (`APTO 45 | BLOCO B`; vazio = sem complemento); `cluster_id = grupo_local || '#' || hash(unidade)`; `flag_compl_ambiguo`: linha sem complemento (ou com complemento parcial) num grupo com 2+ unidades.

## 8. Limitações conhecidas / pontos de atenção para quem revisar

1. **Nomes de rua com números** (`RUA 15 DE NOVEMBRO`, `RUA 7`, `AVENIDA 9 DE JULHO`, `QUADRA 3 CONJUNTO 5`): protegidos por heurísticas (dia+mês colado com `_`, número que sobra sozinho depois do tipo fica no nome). Casos de Brasília/Goiânia (`SQN 308 BL A`, `QD 12 LT 5`) são tratados de forma conservadora (não se extrai número/complemento se o nome ficaria vazio).
2. **Municípios × distritos**: o cliente pode escrever o distrito (`SANTO AMARO`) e os Correios o município (`SAO PAULO`). Quando o CEP bate com o logradouro, a sugestão usa o município dos Correios; quando não bate, prevalece o consenso do CPF.
3. **Correios desatualizados**: ruas novas/loteamentos não constam; o método `CONSENSO_INTERNO` cobre isso, com score menor.
4. **Duas moradias reais no mesmo CPF** (histórico de mudanças, ou dois apartamentos no mesmo prédio): viram clusters diferentes — a sugestão é **por cluster**, não "um endereço por CPF". Se você quiser 1 endereço por CPF, a última célula tem o pseudocódigo para escolher o cluster mais recente/mais frequente.
5. **Clustering é transitivo**: `A~B` e `B~C` ⇒ `A,B,C` juntos. O número em todas as chaves limita fusões erradas, mas números iguais em ruas com nomes parecidos no mesmo CEP podem se fundir (raro).
5b. **Unidades indistinguíveis**: `APTO 45` e `APTO 45 BLOCO A` são tratados como a mesma unidade (o segundo só completa o primeiro). Se o cliente tem de fato o apto 45 do bloco A **e** o apto 45 do bloco B, só linhas que citem os dois blocos separam as unidades. Linhas sem complemento num prédio com 2+ unidades ficam ambíguas (sem complemento sugerido) — é a decisão conservadora; o LLM (E8) pode ser usado nesses clusters se você quiser tentar resolver.
6. **LLM**: só roda em amostra, com cache, e só sobrescreve a sugestão se `confianca_llm ≥ LIMIAR_CONFIANCA_LLM`. **Nunca** enviar CPF ao LLM (o prompt envia só as variantes de endereço e candidatos dos Correios). `PROVEDOR_LLM="mock"` testa o encanamento sem API.
7. **Custos**: `regexp` em bilhões de linhas é CPU; as janelas do clustering são shuffle. Prefira executar em lotes.
8. **Teste**: este notebook foi validado de ponta a ponta com dados sintéticos em PySpark 4.1 local (sem Delta). No Databricks, revise as células de escrita (`persistir`, `escrever_saida`) se seu workspace exigir `LOCATION` externo ou permissões específicas de Unity Catalog.
9. **ANSI mode** (default no Spark 4): o código usa `try_cast`, `try_element_at`, `try_divide` e `get()` para nunca lançar erro por conversão/índice; se você adicionar expressões, mantenha o padrão.

## 9. Checklist para a IA revisora

- [ ] Bloco `AMBIENTES[AMBIENTE_ATIVO]`: catálogo/schema/tabelas/prefixo e `COLS_ENDERECO`/`COLS_CORREIOS` batem com os nomes reais? (`cep_parte 1` com espaço, `municipio` vs `cidade` etc.) — `verificar_ambiente()` lista o que faltar.
- [ ] Em `_metricas`, `check_clusters_nao_uniformes` e `check_sugestoes_iguais_em_clusters_distintos` estão em 0?
- [ ] Na amostra de auditoria de E7 ("prédios/locais com 2+ unidades"), apartamentos/salas diferentes do mesmo cliente estão em clusters distintos com complementos distintos? Quantas linhas ficaram `COMPLEMENTO_AMBIGUO`?
- [ ] `tb_correios.logradouro_correios` vem com ou sem o tipo na frente? (o código trata ambos, mas confira em `_dim_correios_logradouro`)
- [ ] Faixas de CEP por UF (`FAIXAS_CEP_UF`) — conferir DF/GO/RO/AM.
- [ ] Listas "não mapeados" impressas em E3 (tipos dos Correios) e E2 (tokens de complemento frequentes fora do dicionário).
- [ ] `LIMIAR_SIM_LOGRADOURO` (default 0.80) e `LIMIAR_FUZZY_CORREIOS` (default 88 de 100) — calibrar com a amostra auditada.
- [ ] Segredos: `dbutils.secrets.get(SEGREDO_SCOPE, SEGREDO_CHAVE_LLM)` → nunca colocar chave em texto no notebook.
- [ ] Tamanho do cluster e `N_PARTICOES_SHUFFLE` para a base completa.

In [ ]:
%pip install -q --upgrade rapidfuzz unidecode anthropic openai

In [ ]:
# Reinicia o interpretador Python para enxergar as libs instaladas pelo %pip (só faz sentido no Databricks).
try:
    dbutils.library.restartPython()  # noqa: F821 - objeto injetado pelo Databricks
except Exception:
    pass

## ⚙️ Configuração de ambiente — o ÚNICO lugar para trocar tabelas, colunas e API

Para apontar o notebook para outras tabelas (ex.: produtivas) você mexe **só nesta célula**:

1. Preencha (ou crie) um bloco em `AMBIENTES` com catálogo, schemas, nomes das tabelas, prefixo de saída, mapeamento de colunas e configuração do LLM.
2. Troque `AMBIENTE_ATIVO` (ou passe o widget / variável de ambiente `END_AMBIENTE_ATIVO`, ou a chave `AMBIENTE_ATIVO` no `PARAMETROS_JSON`).

Nada mais no notebook referencia nomes físicos: todas as etapas usam `NOME_TB_ENDERECO`, `NOME_TB_CORREIOS`, `COLS_ENDERECO`, `COLS_CORREIOS`, `nome_intermediario(...)` e as variáveis de LLM, que são derivadas do bloco ativo na célula **Parâmetros**. A célula seguinte ao bloco de dados sintéticos executa `verificar_ambiente()`, que confere se as tabelas existem e se **cada coluna mapeada existe de fato**, listando o que faltar antes de qualquer processamento pesado.

Regras do mapeamento de colunas: chave = nome interno usado no código (não mude), valor = nome real da coluna na sua tabela (`None` = a coluna não existe; permitido só nas opcionais: `logradouro_reduzido`, `numero_inicial`, `numero_final`).

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE — tabelas, colunas e API em UM lugar              # >>> AJUSTAR
# ============================================================================
import os

AMBIENTE_ATIVO = os.environ.get("END_AMBIENTE_ATIVO", "dev")   # <<< "dev" | "prod" | qualquer chave de AMBIENTES

# nomes INTERNOS (chaves) -> nomes REAIS das colunas (valores). Só altere os valores.
_COLS_ENDERECO_PADRAO = {
    "idCPF": "idCPF",
    "logradouro": "logradouro",
    "numero": "numero",
    "complemento": "complemento",
    "bairro": "bairro",
    "cidade": "cidade",
    "uf": "uf",
    "cep_parte1": "cep_parte1",
    "cep_parte2": "cep_parte2",
}
_COLS_CORREIOS_PADRAO = {
    "cep_parte1": "cep_parte1",
    "cep_parte2": "cep_parte2",
    "logradouro": "logradouro_correios",
    "logradouro_reduzido": None,      # opcional: ex. "logradouro_reduzido"
    "tipo": "tipo",
    "lado": "lado",
    "uf": "uf",
    "municipio": "municipio",
    "bairro": "bairro",
    "numero_inicial": None,           # opcional: faixa numérica do DNE
    "numero_final": None,
}

AMBIENTES = {
    # ---------------------------------------------------------------- DEV / homologação
    "dev": {
        "CATALOGO": "cadastro",                    # catálogo Unity (None ou "" = hive_metastore / sessão local)
        "SCHEMA_ORIGEM": "default",
        "TB_ENDERECO": "tb_endereco",
        "TB_CORREIOS": "tb_correios",
        "SCHEMA_SAIDA": "default",
        "PREFIXO_SAIDA": "tb_endereco_sugestao",   # todas as tabelas de saída começam com isto
        "COL_DATA": None,                          # coluna de data/recência em tb_endereco (opcional)
        "COLS_ENDERECO": dict(_COLS_ENDERECO_PADRAO),
        "COLS_CORREIOS": dict(_COLS_CORREIOS_PADRAO),
        # ---- LLM (fallback em amostra) ----
        "PROVEDOR_LLM": "anthropic",               # anthropic | openai | azure_openai | databricks | mock
        "MODELO_LLM": "claude-sonnet-5",
        "ENDPOINT_LLM": None,                      # base_url (databricks: https://<workspace>/serving-endpoints)
        "SEGREDO_SCOPE": "cadastro",               # dbutils.secrets.get(scope, chave)
        "SEGREDO_CHAVE_LLM": "llm_api_key",
    },
    # ---------------------------------------------------------------- PRODUÇÃO           # >>> AJUSTAR
    "prod": {
        "CATALOGO": "cadastro",
        "SCHEMA_ORIGEM": "producao",
        "TB_ENDERECO": "tb_endereco",
        "TB_CORREIOS": "tb_correios",
        "SCHEMA_SAIDA": "producao_higienizacao",
        "PREFIXO_SAIDA": "tb_endereco_sugestao",
        "COL_DATA": None,
        # exemplo de coluna com nome diferente/espacos: basta trocar o VALOR
        "COLS_ENDERECO": {**_COLS_ENDERECO_PADRAO, "cep_parte1": "cep_parte1", "cep_parte2": "cep_parte2"},
        "COLS_CORREIOS": {**_COLS_CORREIOS_PADRAO, "logradouro_reduzido": None},
        "PROVEDOR_LLM": "anthropic",
        "MODELO_LLM": "claude-sonnet-5",
        "ENDPOINT_LLM": None,
        "SEGREDO_SCOPE": "cadastro",
        "SEGREDO_CHAVE_LLM": "llm_api_key",
    },
}

_CHAVES_OBRIGATORIAS = ["CATALOGO", "SCHEMA_ORIGEM", "TB_ENDERECO", "TB_CORREIOS", "SCHEMA_SAIDA", "PREFIXO_SAIDA", "COL_DATA",
                        "COLS_ENDERECO", "COLS_CORREIOS", "PROVEDOR_LLM", "MODELO_LLM", "ENDPOINT_LLM", "SEGREDO_SCOPE", "SEGREDO_CHAVE_LLM"]
for _nome, _amb in AMBIENTES.items():
    _faltam = [k for k in _CHAVES_OBRIGATORIAS if k not in _amb]
    assert not _faltam, f"AMBIENTES['{_nome}'] sem as chaves {_faltam}"
    assert set(_amb["COLS_ENDERECO"]) == set(_COLS_ENDERECO_PADRAO), f"AMBIENTES['{_nome}']['COLS_ENDERECO']: não mude as chaves internas"
    assert set(_amb["COLS_CORREIOS"]) == set(_COLS_CORREIOS_PADRAO), f"AMBIENTES['{_nome}']['COLS_CORREIOS']: não mude as chaves internas"
print(f"ambientes definidos: {list(AMBIENTES)} | AMBIENTE_ATIVO (antes de widgets) = {AMBIENTE_ATIVO}")

## Parâmetros

Precedência de cada parâmetro: **widget `PARAMETROS_JSON`** (JSON com chaves = nomes dos parâmetros) → **widget individual** (só os operacionais) → **variável de ambiente `END_<NOME>`** → **default no código**.

Os nomes de tabelas, colunas e API **não** são definidos aqui: vêm do bloco ativo de `AMBIENTES` (célula anterior). Esta célula só lê o bloco, aplica sobrescritas pontuais (widget/env/JSON) e define os parâmetros **operacionais** (amostra, lotes, limiares, LLM ligado/desligado, performance), além dos helpers de I/O (`persistir`, `obter`, `verificar_ambiente`).

In [ ]:
# ============================================================================
# PARÂMETROS — tudo que depende do ambiente está aqui.               # >>> AJUSTAR
# ============================================================================
import os, sys, json, re, time, math, datetime, hashlib, logging, traceback
from typing import Optional

# --------------------------------------------------------------------------
# 0. Sessão Spark / dbutils / display — funcionam em Databricks, bundle e local
# --------------------------------------------------------------------------
def _obter_spark():
    """Reaproveita `spark` se já existir (Databricks); senão tenta Databricks Connect; senão cria sessão local."""
    g = globals()
    if "spark" in g and g["spark"] is not None:
        return g["spark"]
    try:
        from databricks.connect import DatabricksSession  # bundle / VS Code com Databricks Connect
        return DatabricksSession.builder.getOrCreate()
    except Exception:
        pass
    from pyspark.sql import SparkSession
    return (
        SparkSession.builder.master("local[*]")
        .appName("sugestao_endereco")
        .config("spark.driver.memory", os.environ.get("END_SPARK_DRIVER_MEMORY", "6g"))
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.session.timeZone", "America/Sao_Paulo")
        .getOrCreate()
    )

spark = _obter_spark()

def _obter_dbutils():
    g = globals()
    if "dbutils" in g and g["dbutils"] is not None:
        return g["dbutils"]
    try:
        from pyspark.dbutils import DBUtils  # type: ignore
        return DBUtils(spark)
    except Exception:
        return None

dbutils = _obter_dbutils()
EM_DATABRICKS = dbutils is not None and ("DATABRICKS_RUNTIME_VERSION" in os.environ)

try:
    display  # noqa: F821 - existe no Databricks
except NameError:
    def display(df, n: int = 20):  # fallback local
        try:
            df.show(n, truncate=False)
        except Exception:
            print(df)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", stream=sys.stdout, force=True)
log = logging.getLogger("sugestao_endereco")
def info(msg: str):
    print(f"[{datetime.datetime.now():%H:%M:%S}] {msg}", flush=True)

# --------------------------------------------------------------------------
# 1. Leitura de parâmetros (PARAMETROS_JSON > widget > env END_<NOME> > default)
# --------------------------------------------------------------------------
_PARAMS_JSON = {}
if dbutils is not None:
    try:
        dbutils.widgets.text("PARAMETROS_JSON", "", "PARAMETROS_JSON (opcional, sobrescreve tudo)")
        _txt = dbutils.widgets.get("PARAMETROS_JSON")
        if _txt and _txt.strip():
            _PARAMS_JSON = json.loads(_txt)
    except Exception as e:  # widgets indisponíveis (ex.: job sem widgets) -> segue com env/default
        info(f"widgets indisponíveis ({e}); usando env/default")
if os.environ.get("END_PARAMETROS_JSON"):
    _PARAMS_JSON.update(json.loads(os.environ["END_PARAMETROS_JSON"]))

def _converter(valor, default):
    if valor is None:
        return default
    if isinstance(default, bool):
        return str(valor).strip().lower() in ("1", "true", "t", "sim", "s", "yes", "y")
    if isinstance(default, int) and not isinstance(default, bool):
        return int(float(str(valor).strip()))
    if isinstance(default, float):
        return float(str(valor).strip())
    if default is None or isinstance(default, str):
        v = str(valor).strip()
        return None if v.lower() in ("", "none", "null") else v
    return valor

def param(nome: str, default, widget: bool = False, descricao: str = ""):
    """Lê um parâmetro seguindo a precedência documentada acima."""
    if nome in _PARAMS_JSON:
        return _converter(_PARAMS_JSON[nome], default)
    if widget and dbutils is not None:
        try:
            dbutils.widgets.text(nome, "" if default is None else str(default), descricao or nome)
            v = dbutils.widgets.get(nome)
            if v is not None and str(v) != "":
                return _converter(v, default)
        except Exception:
            pass
    if f"END_{nome}" in os.environ:
        return _converter(os.environ[f"END_{nome}"], default)
    return default

# --------------------------------------------------------------------------
# 2. Origem / destino — vem do bloco ativo de AMBIENTES (célula anterior); widget/env/JSON só sobrescrevem
# --------------------------------------------------------------------------
AMBIENTE_ATIVO  = param("AMBIENTE_ATIVO", AMBIENTE_ATIVO, widget=True, descricao="bloco de AMBIENTES a usar (dev | prod ...)")
assert AMBIENTE_ATIVO in AMBIENTES, f"AMBIENTE_ATIVO='{AMBIENTE_ATIVO}' não existe em AMBIENTES {list(AMBIENTES)}"
AMB = AMBIENTES[AMBIENTE_ATIVO]

CATALOGO        = param("CATALOGO", AMB["CATALOGO"], widget=True, descricao="Catálogo Unity (vazio = hive_metastore/local)")
SCHEMA_ORIGEM   = param("SCHEMA_ORIGEM", AMB["SCHEMA_ORIGEM"], widget=True)
TB_ENDERECO     = param("TB_ENDERECO", AMB["TB_ENDERECO"], widget=True)
TB_CORREIOS     = param("TB_CORREIOS", AMB["TB_CORREIOS"], widget=True)
SCHEMA_SAIDA    = param("SCHEMA_SAIDA", AMB["SCHEMA_SAIDA"], widget=True)
PREFIXO_SAIDA   = param("PREFIXO_SAIDA", AMB["PREFIXO_SAIDA"], widget=True)

COLS_ENDERECO = dict(AMB["COLS_ENDERECO"])   # nome interno -> nome real (definido na célula de ambiente)
COLS_CORREIOS = dict(AMB["COLS_CORREIOS"])
_json_cols_end = param("COLS_ENDERECO_JSON", None)   # sobrescrita pontual via JSON, ex.: {"cep_parte1": "cep_parte 1"}
if _json_cols_end:
    COLS_ENDERECO.update(json.loads(_json_cols_end))
_json_cols_cor = param("COLS_CORREIOS_JSON", None)
if _json_cols_cor:
    COLS_CORREIOS.update(json.loads(_json_cols_cor))

COL_DATA = param("COL_DATA", AMB["COL_DATA"], widget=True, descricao="Coluna de data/recência em tb_endereco (opcional)")

CAMPOS_ENDERECO = ["logradouro", "numero", "complemento", "bairro", "cidade", "uf", "cep_parte1", "cep_parte2"]

# --------------------------------------------------------------------------
# 3. Escopo da execução
# --------------------------------------------------------------------------
MODO_AMOSTRA        = param("MODO_AMOSTRA", True, widget=True, descricao="true = só uma fração dos CPFs")
FRACAO_AMOSTRA      = param("FRACAO_AMOSTRA", 0.001, widget=True, descricao="fração de CPFs (hash) quando MODO_AMOSTRA")
LISTA_CPFS_AMOSTRA  = param("LISTA_CPFS_AMOSTRA", None, widget=True, descricao="CPFs separados por vírgula (opcional)")
N_LOTES             = param("N_LOTES", 1, widget=True, descricao="nº de lotes por hash(idCPF) para a base completa")
LOTE_ATUAL          = param("LOTE_ATUAL", 0, widget=True, descricao="lote a processar nesta execução (0..N_LOTES-1)")
PERSISTIR_INTERMEDIARIOS = param("PERSISTIR_INTERMEDIARIOS", True, widget=True, descricao="grava cada etapa em Delta")
ESCREVER_SAIDA      = param("ESCREVER_SAIDA", True, widget=True)
ETAPAS_A_EXECUTAR   = param("ETAPAS_A_EXECUTAR", "E1,E2,E3,E4,E5,E6,E7,E8,E9", widget=True, descricao="etapas a rodar (as demais são lidas das tabelas intermediárias)")
CALCULAR_CONTAGENS  = param("CALCULAR_CONTAGENS", True, widget=True, descricao="false = pula counts caros na base completa")
MODO_TESTE_SINTETICO = param("MODO_TESTE_SINTETICO", False, widget=True, descricao="true = gera tb_endereco/tb_correios sintéticas em temp view")

# --------------------------------------------------------------------------
# 4. Matching / fuzzy
# --------------------------------------------------------------------------
USAR_FUZZY_CORREIOS    = param("USAR_FUZZY_CORREIOS", True)
USAR_RAPIDFUZZ         = param("USAR_RAPIDFUZZ", True)          # pandas UDF sobre pares candidatos (não sobre a base)
LIMIAR_SIM_LOGRADOURO  = param("LIMIAR_SIM_LOGRADOURO", 0.80)   # 0..1 — logradouro do cliente "bate" com o do CEP
LIMIAR_FUZZY_CORREIOS  = param("LIMIAR_FUZZY_CORREIOS", 88.0)   # 0..100 — aceitar melhor candidato fuzzy
MARGEM_FUZZY           = param("MARGEM_FUZZY", 3.0)             # distância mínima para o 2º candidato
MAX_CANDIDATOS_FUZZY   = param("MAX_CANDIDATOS_FUZZY", 200)     # limite de candidatos por consulta em blocos grandes
MAX_ITER_CLUSTER       = param("MAX_ITER_CLUSTER", 8)
CHAVES_CLUSTER_DESATIVADAS = param("CHAVES_CLUSTER_DESATIVADAS", None)  # csv p/ testes de ablação, ex.: "k_ana,k_p5" (desliga chaves de E6)
# --- grau de certeza (escala A_ALTA / B_MEDIA / C_BAIXA / D_REVISAO_HUMANA) ---
LIMIAR_GRAU_ALTA  = param("LIMIAR_GRAU_ALTA", 85.0)    # score_confianca >= isto (e sem restrições) -> A_ALTA
LIMIAR_GRAU_MEDIA = param("LIMIAR_GRAU_MEDIA", 65.0)   # score_confianca >= isto -> B_MEDIA; abaixo -> C_BAIXA

# --------------------------------------------------------------------------
# 5. LLM (fallback em amostra)                                          # >>> AJUSTAR
# --------------------------------------------------------------------------
USAR_LLM                  = param("USAR_LLM", False, widget=True)
LIMITE_LLM                = param("LIMITE_LLM", 500, widget=True, descricao="máx. de clusters enviados ao LLM nesta execução")
LIMIAR_CONFIANCA_PARA_LLM = param("LIMIAR_CONFIANCA_PARA_LLM", 60.0)   # clusters com score < isto vão ao LLM
LIMIAR_CONFIANCA_LLM      = param("LIMIAR_CONFIANCA_LLM", 0.80)        # confiança mínima da resposta para sobrescrever
PROVEDOR_LLM              = param("PROVEDOR_LLM", AMB["PROVEDOR_LLM"])            # anthropic | openai | databricks | azure_openai | mock
MODELO_LLM                = param("MODELO_LLM", AMB["MODELO_LLM"])                # ex.: claude-sonnet-5, claude-opus-5, gpt-4.1, databricks-claude-sonnet-4
SEGREDO_SCOPE             = param("SEGREDO_SCOPE", AMB["SEGREDO_SCOPE"])          # dbutils.secrets scope
SEGREDO_CHAVE_LLM         = param("SEGREDO_CHAVE_LLM", AMB["SEGREDO_CHAVE_LLM"])  # dbutils.secrets key
ENDPOINT_LLM              = param("ENDPOINT_LLM", AMB["ENDPOINT_LLM"])            # base_url p/ databricks/azure
LLM_CONCORRENCIA          = param("LLM_CONCORRENCIA", 4)
LLM_MAX_VARIANTES_PROMPT  = param("LLM_MAX_VARIANTES_PROMPT", 12)

# --------------------------------------------------------------------------
# 6. Performance
# --------------------------------------------------------------------------
N_PARTICOES_SHUFFLE = param("N_PARTICOES_SHUFFLE", 4000 if EM_DATABRICKS else 16)
BROADCAST_CORREIOS  = param("BROADCAST_CORREIOS", True)

# --------------------------------------------------------------------------
# 7. Derivados / helpers de nomes e I/O
# --------------------------------------------------------------------------
def nome_tabela(schema: Optional[str], tabela: str) -> str:
    partes = [p for p in (CATALOGO, schema, tabela) if p]
    return ".".join(f"`{p}`" if not p.startswith("`") else p for p in partes)

NOME_TB_ENDERECO = nome_tabela(SCHEMA_ORIGEM, TB_ENDERECO)
NOME_TB_CORREIOS = nome_tabela(SCHEMA_ORIGEM, TB_CORREIOS)
SUFIXO_LOTE = f"_lote{LOTE_ATUAL:02d}" if N_LOTES > 1 else ""
ETAPAS = {e.strip().upper() for e in ETAPAS_A_EXECUTAR.split(",") if e.strip()}
ID_EXECUCAO = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def etapa_ativa(e: str) -> bool:
    return e.upper() in ETAPAS

def nome_intermediario(sufixo: str, com_lote: bool = True) -> str:
    return nome_tabela(SCHEMA_SAIDA, f"{PREFIXO_SAIDA}_{sufixo}{SUFIXO_LOTE if com_lote else ''}")

ARTEFATOS = {}  # cache em memória dos DataFrames de cada etapa (usado quando não se persiste)

def persistir(df, sufixo: str, cluster_by=None, com_lote: bool = True):
    """Grava a etapa em Delta e devolve o DataFrame lido de volta (corta a linhagem).
    Sem persistência: cacheia em memória. Sempre registra em ARTEFATOS."""
    if PERSISTIR_INTERMEDIARIOS:
        nome = nome_intermediario(sufixo, com_lote)
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if cluster_by:
            try:
                w = w.clusterBy(*([cluster_by] if isinstance(cluster_by, str) else cluster_by))  # Liquid Clustering (DBR >= 15.2)
            except Exception:
                pass
        w.saveAsTable(nome)
        out = spark.table(nome)
        info(f"persistido {nome}")
    else:
        # sem tabela: materializa e CORTA A LINHAGEM (planos com centenas de Projects/regex estouram o driver)
        out = df.localCheckpoint(eager=True)
    ARTEFATOS[sufixo] = out
    return out

def obter(sufixo: str, com_lote: bool = True):
    """Devolve o DataFrame de uma etapa: memória (ARTEFATOS) ou tabela Delta persistida."""
    if sufixo in ARTEFATOS:
        return ARTEFATOS[sufixo]
    nome = nome_intermediario(sufixo, com_lote)
    info(f"lendo {nome}")
    return spark.table(nome)

def tabela_existe(nome: str) -> bool:
    try:
        return spark.catalog.tableExists(nome.replace("`", ""))
    except Exception:
        return False

def contar(df, rotulo: str):
    if CALCULAR_CONTAGENS:
        n = df.count()
        info(f"{rotulo}: {n:,}".replace(",", "."))
        return n
    return None


def verificar_ambiente(lancar_erro: bool = True) -> bool:
    """Confere se as tabelas do ambiente ativo existem e se TODAS as colunas mapeadas existem nelas.
    Imprime um relatório; com lancar_erro=True interrompe o notebook se algo faltar (evita rodar horas e quebrar no fim)."""
    problemas = []
    for rotulo, nome, mapa, opcionais in [
        ("tb_endereco", NOME_TB_ENDERECO, COLS_ENDERECO, set()),
        ("tb_correios", NOME_TB_CORREIOS, COLS_CORREIOS, {"logradouro_reduzido", "numero_inicial", "numero_final"}),
    ]:
        try:
            cols = {c.lower() for c in spark.table(nome).columns}
        except Exception as e:
            problemas.append(f"{rotulo}: tabela {nome} não encontrada ({type(e).__name__})")
            continue
        for interno, real in mapa.items():
            if real is None:
                if interno not in opcionais:
                    problemas.append(f"{rotulo}: coluna interna obrigatória '{interno}' mapeada como None")
            elif real.lower() not in cols:
                problemas.append(f"{rotulo}: coluna '{real}' (interna '{interno}') não existe em {nome}; existentes: {sorted(cols)[:40]}")
        if rotulo == "tb_endereco" and COL_DATA and COL_DATA.lower() not in cols:
            problemas.append(f"tb_endereco: COL_DATA='{COL_DATA}' não existe em {nome}")
    info(f"verificar_ambiente [{AMBIENTE_ATIVO}] origem={NOME_TB_ENDERECO}, {NOME_TB_CORREIOS} | saída={nome_intermediario('final', com_lote=False)}")
    if problemas:
        for p in problemas:
            info("  PROBLEMA: " + p)
        if lancar_erro:
            raise ValueError(f"Ambiente '{AMBIENTE_ATIVO}' inválido: {len(problemas)} problema(s) — corrija a célula de CONFIGURAÇÃO DE AMBIENTE")
        return False
    info("  ambiente OK: tabelas e colunas mapeadas encontradas")
    return True

# --------------------------------------------------------------------------
# 8. Configurações Spark
# --------------------------------------------------------------------------
for k, v in {
    "spark.sql.shuffle.partitions": str(N_PARTICOES_SHUFFLE),
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.autoBroadcastJoinThreshold": str(256 * 1024 * 1024),
    "spark.databricks.delta.optimizeWrite.enabled": "true",
    "spark.databricks.delta.autoCompact.enabled": "true",
}.items():
    try:
        spark.conf.set(k, v)
    except Exception:
        pass  # confs Databricks não existem em Spark local

info(f"Spark {spark.version} | Databricks={EM_DATABRICKS} | AMBIENTE_ATIVO={AMBIENTE_ATIVO} | origem={NOME_TB_ENDERECO} / {NOME_TB_CORREIOS}")
info(f"saída prefixo={PREFIXO_SAIDA}{SUFIXO_LOTE} | amostra={MODO_AMOSTRA} ({FRACAO_AMOSTRA}) | lotes={N_LOTES}/{LOTE_ATUAL} | etapas={sorted(ETAPAS)}")
info(f"persistir={PERSISTIR_INTERMEDIARIOS} | escrever_saida={ESCREVER_SAIDA} | fuzzy={USAR_FUZZY_CORREIOS} (rapidfuzz={USAR_RAPIDFUZZ}) | llm={USAR_LLM} ({PROVEDOR_LLM}/{MODELO_LLM}, limite {LIMITE_LLM})")

## Referências (dicionários pt-BR)

Tudo que é "conhecimento de domínio" fica aqui, em listas Python simples, para ser fácil de estender:

- `MAPA_TIPOS_CLIENTE`: variantes escritas pelo cliente para o **tipo de logradouro** (só é aplicado ao 1º token do logradouro). Conservador: abreviações ambíguas com títulos (ex.: `GAL` = General × Galeria) **não** entram aqui.
- `MAPA_TIPOS_CORREIOS`: abreviações do DNE (`tipo` da `tb_correios`) → nome por extenso. Os ~360 tipos oficiais não estão todos aqui; os não mapeados são impressos em E3 e ficam com a própria abreviação como canônico (funciona, só não casa com a variante por extenso do cliente).
- `MAPA_TITULOS`: abreviações dentro do **nome** (`DR`→`DOUTOR`, `PROF`→`PROFESSOR`, ...). Nunca expandimos letras únicas (`D`, `S`) dentro do nome, porque colidem com iniciais da forma reduzida — exceção tratada por regex: `S` como 1º token do nome (`S JOAO`→`SAO JOAO`) e `N S`/`NSA`/`N SRA` → `NOSSA SENHORA`.
- `MAPA_COMPL`: palavras-chave de **complemento** e sua forma canônica; `ORDEM_COMPL` define a ordem de montagem da sugestão.
- `MESES`: protege nomes como `15 DE NOVEMBRO` de serem lidos como número.
- `UFS`, `MAPA_UF`: siglas, nomes por extenso e erros comuns.
- `FAIXAS_CEP_UF`: faixas oficiais de CEP por UF (validação barata, independente da `tb_correios`).
- `ALIASES_CIDADE`: apelidos/abreviações de cidades muito frequentes (`SP`, `BH`, `POA`, `S PAULO`...).

In [ ]:
# ============================================================================
# REFERÊNCIAS — dicionários de domínio (pt-BR / endereços do Brasil)
# ============================================================================

# --- Tipos de logradouro: canônico -> variantes escritas por clientes (aplicado só ao 1º token) ------------
_TIPOS_CLIENTE = {
    "RUA": ["R", "RU", "RUA", "RUAS", "RA"],
    "AVENIDA": ["AV", "AVE", "AVN", "AVDA", "AVENIDA", "AVEN", "AVD", "AVENDIA", "AVENIDAS"],
    "TRAVESSA": ["TV", "TRAV", "TRV", "TRAVESSA", "TRA", "TRAVES"],
    "ALAMEDA": ["AL", "ALM", "ALAMEDA", "ALA"],
    "PRACA": ["PC", "PCA", "PRC", "PRACA", "PR"],
    "RODOVIA": ["ROD", "RODOVIA", "RDV", "RDVIA"],
    "ESTRADA": ["EST", "ESTR", "ESTRADA", "ETR", "ESTRD"],
    "LARGO": ["LGO", "LG", "LARGO"],
    "VIA": ["VIA"],
    "VIELA": ["VIELA", "VLA"],
    "VILA": ["VL", "VILA"],
    "BECO": ["BC", "BECO"],
    "QUADRA": ["QD", "QDA", "QUADRA", "Q", "QDR"],
    "CONJUNTO": ["CJ", "CONJ", "CONJUNTO", "CJTO"],
    "CONDOMINIO": ["COND", "CONDOMINIO", "CONDOM"],
    "SITIO": ["SIT", "SITIO"],
    "FAZENDA": ["FAZ", "FAZENDA", "FZD"],
    "CHACARA": ["CHAC", "CHACARA"],
    "LOTEAMENTO": ["LOT", "LOTEAMENTO", "LTM", "LOTEAM"],
    "PARQUE": ["PQ", "PQE", "PRQ", "PARQUE"],
    "JARDIM": ["JD", "JDM", "JARDIM"],
    "SERVIDAO": ["SERV", "SERVIDAO", "SVD"],
    "ACESSO": ["ACESSO"],
    "LADEIRA": ["LAD", "LADEIRA"],
    "MARGINAL": ["MARG", "MARGINAL"],
    "PASSAGEM": ["PSG", "PASSAGEM"],
    "PASSEIO": ["PSE", "PASSEIO"],
    "PATIO": ["PATIO"],
    "RAMAL": ["RAMAL"],
    "RECANTO": ["REC", "RECANTO"],
    "RETA": ["RETA"],
    "ROTULA": ["ROT", "ROTULA"],
    "SETOR": ["SETOR"],
    "TRECHO": ["TRECHO"],
    "TREVO": ["TREVO"],
    "TUNEL": ["TUN", "TUNEL"],
    "VALE": ["VLE", "VALE"],
    "VEREDA": ["VEREDA"],
    "AEROPORTO": ["AER", "AEROPORTO"],
    "BALNEARIO": ["BAL", "BALNEARIO"],
    "CAMINHO": ["CAM", "CAMINHO"],
    "COLONIA": ["COL", "COLONIA"],
    "DISTRITO": ["DISTRITO"],
    "ESPLANADA": ["ESP", "ESPLANADA"],
    "ESTACAO": ["ESTACAO"],
    "FAVELA": ["FAV", "FAVELA"],
    "FERROVIA": ["FERROVIA"],
    "GALERIA": ["GALERIA", "GLR"],
    "GRANJA": ["GJA", "GRANJA"],
    "ILHA": ["ILHA"],
    "LAGO": ["LAGO"],
    "LAGOA": ["LAGOA"],
    "MORRO": ["MRO", "MORRO"],
    "NUCLEO": ["NUC", "NUCLEO"],
    "PONTE": ["PTE", "PONTE"],
    "PORTO": ["PTO", "PORTO"],
    "PRAIA": ["PRAIA"],
    "PROLONGAMENTO": ["PRL", "PROLONGAMENTO", "PROL"],
    "RESIDENCIAL": ["RES", "RESIDENCIAL", "RESID"],
    "SUBIDA": ["SUBIDA"],
    "TERMINAL": ["TERMINAL"],
    "VARIANTE": ["VARIANTE"],
    "VIADUTO": ["VD", "VIADUTO"],
    "ZONA": ["ZONA"],
    "ESCADARIA": ["ESC", "ESCADARIA"],
    "PISTA": ["PST", "PISTA"],
    "RETORNO": ["RTN", "RETORNO"],
    "CIRCULAR": ["CIRC", "CIRCULAR"],
    "DESVIO": ["DSV", "DESVIO"],
    "ELEVADA": ["ELEVADA"],
    "ENTREQUADRA": ["EQ", "ENTREQUADRA"],
    "MODULO": ["MODULO"],
    "MONTE": ["MTE", "MONTE"],
    "PARADA": ["PDA", "PARADA"],
    "PARALELA": ["PARALELA"],
    "QUINTA": ["QTA", "QUINTA"],
    "QUINTAS": ["QTS", "QUINTAS"],
    "TRINCHEIRA": ["TCH", "TRINCHEIRA"],
    "UNIDADE": ["UNIDADE"],
    "VALA": ["VALA"],
    "ESTANCIA": ["ESTANCIA"],
    "BOSQUE": ["BSQ", "BOSQUE"],
    "BOULEVARD": ["BLV", "BOULEVARD", "BOULEVAR", "BULEVAR"],
    "CALCADA": ["CALCADA"],
    "CANAL": ["CANAL"],
    "AREA": ["AREA"],
    "CAMPO": ["CPO", "CAMPO"],
    "CORREDOR": ["CORREDOR"],
    "ESTACIONAMENTO": ["ESTACIONAMENTO"],
    "FONTE": ["FNT", "FONTE"],
    "FORTE": ["FORTE"],
    "JARDINETE": ["JDE", "JARDINETE"],
    "LOTE": ["LT", "LOTE"],
    "BLOCO": ["BL", "BLOCO"],
    "AREA ESPECIAL": ["AE"],
    "COMUNIDADE": ["COM", "COMUNIDADE"],
    "ASSENTAMENTO": ["ASSENTAMENTO", "ASSENT"],
    "POVOADO": ["POV", "POVOADO"],
    "LINHA": ["LINHA", "LNH"],
    "GLEBA": ["GLEBA", "GLB"],
}
MAPA_TIPOS_CLIENTE = {v: k for k, vs in _TIPOS_CLIENTE.items() for v in vs}

# --- Abreviações oficiais do DNE (campo `tipo` da tb_correios) -> por extenso ------------------------------
# Cobre os tipos mais frequentes; os demais ficam com a própria abreviação (lista impressa na etapa E3).
MAPA_TIPOS_CORREIOS = dict(MAPA_TIPOS_CLIENTE)
MAPA_TIPOS_CORREIOS.update({
    "R": "RUA", "AV": "AVENIDA", "TV": "TRAVESSA", "AL": "ALAMEDA", "PC": "PRACA", "PCA": "PRACA", "ROD": "RODOVIA",
    "EST": "ESTRADA", "LGO": "LARGO", "VIA": "VIA", "VL": "VILA", "BC": "BECO", "Q": "QUADRA", "CJ": "CONJUNTO",
    "COND": "CONDOMINIO", "SIT": "SITIO", "FAZ": "FAZENDA", "CH": "CHACARA", "LOT": "LOTEAMENTO", "PQ": "PARQUE",
    "JD": "JARDIM", "SERV": "SERVIDAO", "AC": "ACESSO", "LAD": "LADEIRA", "MARG": "MARGINAL", "PSG": "PASSAGEM",
    "PSE": "PASSEIO", "PAT": "PATIO", "RAM": "RAMAL", "REC": "RECANTO", "ROT": "ROTULA", "ST": "SETOR", "TR": "TRECHO",
    "TRV": "TREVO", "TUN": "TUNEL", "VLE": "VALE", "VER": "VEREDA", "AER": "AEROPORTO", "AREA": "AREA", "BAL": "BALNEARIO",
    "BL": "BLOCO", "CAM": "CAMINHO", "CPO": "CAMPO", "COL": "COLONIA", "COR": "CORREDOR", "DT": "DISTRITO",
    "ESP": "ESPLANADA", "ETC": "ESTACAO", "ETN": "ESTACIONAMENTO", "FAV": "FAVELA", "FRA": "FEIRA", "FER": "FERROVIA",
    "FNT": "FONTE", "FTE": "FORTE", "GAL": "GALERIA", "GJA": "GRANJA", "ILHA": "ILHA", "JDE": "JARDINETE",
    "LAGO": "LAGO", "LAGOA": "LAGOA", "LT": "LOTE", "MRO": "MORRO", "NUC": "NUCLEO", "PTE": "PONTE", "PTO": "PORTO",
    "PR": "PRAIA", "PRL": "PROLONGAMENTO", "RES": "RESIDENCIAL", "SUB": "SUBIDA", "TER": "TERMINAL", "VAR": "VARIANTE",
    "VD": "VIADUTO", "ZON": "ZONA", "ESC": "ESCADARIA", "PST": "PISTA", "RTN": "RETORNO", "CIRC": "CIRCULAR",
    "DSV": "DESVIO", "EVA": "ELEVADA", "EQ": "ENTREQUADRA", "MOD": "MODULO", "MTE": "MONTE", "PDA": "PARADA",
    "PAR": "PARALELA", "QTA": "QUINTA", "QTS": "QUINTAS", "TCH": "TRINCHEIRA", "UNI": "UNIDADE", "VALA": "VALA",
    "ETA": "ESTANCIA", "BSQ": "BOSQUE", "BLV": "BOULEVARD", "CAL": "CALCADA", "CAN": "CANAL", "AE": "AREA ESPECIAL",
    "COM": "COMUNIDADE", "POV": "POVOADO", "LNH": "LINHA", "GLB": "GLEBA", "O": "OUTROS", "ANT": "ANTIGA ESTRADA",
    "ART": "ARTERIA", "ATL": "ATALHO", "BX": "BAIXA", "BLQ": "BLOQUEIO", "BVD": "BOULEVARD", "CHA": "CHAPADAO",
    "CMP": "COMPLEXO VIARIO", "CON": "CONTORNO", "DSC": "DESCIDA", "ENT": "ENTRADA PARTICULAR", "ESD": "ESTADIO",
    "ETD": "ESTADIO", "ETV": "ESTIVA", "EVD": "ELEVADO", "GRJ": "GRANJA", "HAB": "HABITACIONAL", "JRD": "JARDIM",
    "LRG": "LARGO", "MNA": "MARINA", "MRG": "MARGEM", "PAS": "PASSAGEM", "PAV": "PAVILHAO", "PRO": "PROLONGAMENTO",
    "QDA": "QUADRA", "RCT": "RECANTO", "RER": "RETIRO", "RMP": "RAMPA", "RPR": "RAMPA", "RUA": "RUA", "SEG": "SEGUNDA AVENIDA",
    "TRR": "TERRA", "TRC": "TRECHO", "VAL": "VALE", "VCO": "VIA COLETORA",
    "VEV": "VIA ELEVADO", "VEX": "VIA EXPRESSA", "VLA": "VIELA", "VLT": "VIA LITORANEA", "VPE": "VIA DE PEDESTRE",
    "VRT": "VARIANTE", "ZIG": "ZIGUE-ZAGUE",
})

# --- Títulos / abreviações dentro do nome (token inteiro, nunca letra única) --------------------------------
MAPA_TITULOS = {
    "DR": "DOUTOR", "DRA": "DOUTORA", "PROF": "PROFESSOR", "PROFA": "PROFESSORA", "CEL": "CORONEL", "CORONEL": "CORONEL",
    "MAL": "MARECHAL", "GAL": "GENERAL", "GEN": "GENERAL", "BRIG": "BRIGADEIRO", "CAP": "CAPITAO", "TEN": "TENENTE",
    "SGT": "SARGENTO", "SARG": "SARGENTO", "MAJ": "MAJOR", "ALM": "ALMIRANTE", "CMTE": "COMANDANTE", "CMT": "COMANDANTE",
    "COMTE": "COMANDANTE", "PE": "PADRE", "PDE": "PADRE", "FR": "FREI", "MONS": "MONSENHOR", "SR": "SENHOR", "SRA": "SENHORA",
    "STO": "SANTO", "STA": "SANTA", "ENG": "ENGENHEIRO", "ENGO": "ENGENHEIRO", "DEP": "DEPUTADO", "SEN": "SENADOR",
    "VER": "VEREADOR", "PRES": "PRESIDENTE", "GOV": "GOVERNADOR", "PREF": "PREFEITO", "MIN": "MINISTRO",
    "DES": "DESEMBARGADOR", "CONS": "CONSELHEIRO", "VISC": "VISCONDE", "BAR": "BARAO", "MQS": "MARQUES", "CDE": "CONDE",
    "DQ": "DUQUE", "IMP": "IMPERADOR", "INF": "INFANTE", "EMB": "EMBAIXADOR", "JORN": "JORNALISTA", "EXP": "EXPEDICIONARIO",
    "PROC": "PROCURADOR", "FARM": "FARMACEUTICO", "ADV": "ADVOGADO", "ARQ": "ARQUITETO", "CB": "CABO", "SD": "SOLDADO",
    "SOLD": "SOLDADO", "SUBTEN": "SUBTENENTE", "ASP": "ASPIRANTE", "COMEND": "COMENDADOR", "CMDR": "COMENDADOR",
    "JD": "JARDIM", "PQ": "PARQUE", "PQE": "PARQUE", "VL": "VILA", "CJ": "CONJUNTO", "COND": "CONDOMINIO",
    "RES": "RESIDENCIAL", "LOT": "LOTEAMENTO", "NSA": "NOSSA SENHORA", "NSRA": "NOSSA SENHORA", "SANT": "SANTO",
    "PRINC": "PRINCESA", "MAE": "MAESTRO", "PIL": "PILOTO", "AVIAD": "AVIADOR", "COMP": "COMPOSITOR", "ESCR": "ESCRITOR",
    "PINT": "PINTOR", "POETA": "POETA", "HIST": "HISTORIADOR", "REV": "REVERENDO", "BISPO": "BISPO", "CAPELAO": "CAPELAO",
}

# --- Complemento ---------------------------------------------------------------------------------------------
_COMPL = {
    "APTO": ["APTO", "AP", "APT", "APART", "APARTAMENTO", "APTOS", "APARTAMENTOS", "APTP", "APTº"],
    "BLOCO": ["BL", "BLC", "BLO", "BLOCO", "BLK", "BLOC", "BLOCOS"],
    "CASA": ["CS", "CASA", "CASAS"],
    "FUNDOS": ["FUNDOS", "FDS", "FUNDO", "FND", "FDOS"],
    "LOTE": ["LT", "LOTE", "LTE", "LOTES"],
    "QUADRA": ["QD", "QDA", "QUADRA", "QDR", "Q"],
    "SALA": ["SL", "SALA", "SLA", "SALAS"],
    "CONJ": ["CJ", "CONJ", "CONJUNTO", "CJTO"],
    "ANDAR": ["AND", "ANDAR", "ANDR"],
    "TORRE": ["TR", "TORRE", "TOR", "TRR"],
    "EDIFICIO": ["ED", "EDF", "EDIF", "EDIFICIO", "PREDIO", "PRED", "EDIFICO"],
    "COND": ["COND", "CONDOMINIO", "CONDOM"],
    "LOJA": ["LJ", "LOJA", "LJA", "LOJAS"],
    "KM": ["KM", "QUILOMETRO", "KMS"],
    "GALPAO": ["GALPAO", "GLP"],
    "SOBRELOJA": ["SOBRELOJA", "SLJ"],
    "TERREO": ["TERREO"],
    "COBERTURA": ["COB", "COBERTURA"],
    "SOBRADO": ["SOBRADO"],
    "FRENTE": ["FRENTE", "FTE"],
    "BOX": ["BOX"],
    "VAGA": ["VAGA", "VG"],
    "UNIDADE": ["UNID", "UNIDADE", "UN"],
    "ESCRITORIO": ["ESCR", "ESCRITORIO"],
    "PAVIMENTO": ["PAV", "PAVIMENTO"],
    "PORTAO": ["PORTAO"],
    "ALA": ["ALA"],
    "MODULO": ["MOD", "MODULO"],
    "ETAPA": ["ETAPA"],
    "SUBSOLO": ["SUBSOLO"],
    "LETRA": ["LETRA"],
    "SETOR": ["SETOR"],
    "CHACARA": ["CHACARA"],
    "SITIO": ["SITIO"],
    "GLEBA": ["GLEBA"],
    "BARRACO": ["BARRACO"],
    "QUITINETE": ["QUITINETE", "KITNET", "KIT", "KITINETE"],
    "PARTE": ["PARTE"],
}
MAPA_COMPL = {v: k for k, vs in _COMPL.items() for v in vs}
# palavras-chave que carregam VALOR (APTO 45) x que são sozinhas (FUNDOS)
COMPL_COM_VALOR = ["APTO", "BLOCO", "CASA", "LOTE", "QUADRA", "SALA", "CONJ", "ANDAR", "TORRE", "EDIFICIO", "COND", "LOJA",
                   "KM", "GALPAO", "BOX", "VAGA", "UNIDADE", "ESCRITORIO", "PAVIMENTO", "PORTAO", "ALA", "MODULO", "ETAPA",
                   "LETRA", "SETOR", "CHACARA", "SITIO", "GLEBA", "BARRACO", "QUITINETE", "PARTE", "SOBRELOJA", "SUBSOLO"]
COMPL_SEM_VALOR = ["FUNDOS", "TERREO", "COBERTURA", "SOBRADO", "FRENTE", "SOBRELOJA", "SUBSOLO"]
ORDEM_COMPL = ["QUADRA", "LOTE", "GLEBA", "SETOR", "ETAPA", "MODULO", "COND", "EDIFICIO", "BLOCO", "TORRE", "ALA", "ANDAR",
               "PAVIMENTO", "APTO", "CASA", "SALA", "CONJ", "LOJA", "SOBRELOJA", "SUBSOLO", "GALPAO", "BOX", "VAGA", "UNIDADE",
               "ESCRITORIO", "PORTAO", "LETRA", "CHACARA", "SITIO", "BARRACO", "QUITINETE", "PARTE", "KM", "FUNDOS", "FRENTE",
               "SOBRADO", "TERREO", "COBERTURA"]

# --- Marcadores de número / sem número -----------------------------------------------------------------------
MARCADORES_NUMERO = ["N", "NO", "NR", "NRO", "NUM", "NUMERO", "NUN"]
RE_SEM_NUMERO = r"^(S\s*/?\s*N[O.]?|SN|S\s*N|SEM\s*N(UMERO|RO|R|O)?|S\s*/\s*NUMERO|0+)$"

# --- Meses (para proteger "15 DE NOVEMBRO") ------------------------------------------------------------------
MESES = ["JANEIRO", "FEVEREIRO", "MARCO", "ABRIL", "MAIO", "JUNHO", "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO",
         "DEZEMBRO", "JAN", "FEV", "MAR", "ABR", "MAI", "JUN", "JUL", "AGO", "SET", "OUT", "NOV", "DEZ"]

# --- Stopwords de nomes de logradouro ------------------------------------------------------------------------
STOPWORDS = ["DE", "DA", "DO", "DAS", "DOS", "E", "D"]

# --- UFs -----------------------------------------------------------------------------------------------------
UFS = ["AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ",
       "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"]
MAPA_UF = {u: u for u in UFS}
MAPA_UF.update({
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP", "AMAZONAS": "AM", "BAHIA": "BA", "CEARA": "CE", "DISTRITO FEDERAL": "DF",
    "BRASILIA": "DF", "ESPIRITO SANTO": "ES", "GOIAS": "GO", "MARANHAO": "MA", "MATO GROSSO": "MT", "MATO GROSSO DO SUL": "MS",
    "MINAS GERAIS": "MG", "MINAS": "MG", "PARA": "PA", "PARAIBA": "PB", "PARANA": "PR", "PERNAMBUCO": "PE", "PIAUI": "PI",
    "RIO DE JANEIRO": "RJ", "RIO GRANDE DO NORTE": "RN", "RIO GRANDE DO SUL": "RS", "RONDONIA": "RO", "RORAIMA": "RR",
    "SANTA CATARINA": "SC", "SAO PAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO",
    # erros comuns
    "S P": "SP", "R J": "RJ", "M G": "MG", "SPO": "SP", "RJO": "RJ", "PRN": "PR", "STA CATARINA": "SC",
})

# --- Faixas oficiais de CEP por UF (inclusive) ----------------------------------------------------------------
FAIXAS_CEP_UF = [
    ("SP", 1000000, 19999999), ("RJ", 20000000, 28999999), ("ES", 29000000, 29999999), ("MG", 30000000, 39999999),
    ("BA", 40000000, 48999999), ("SE", 49000000, 49999999), ("PE", 50000000, 56999999), ("AL", 57000000, 57999999),
    ("PB", 58000000, 58999999), ("RN", 59000000, 59999999), ("CE", 60000000, 63999999), ("PI", 64000000, 64999999),
    ("MA", 65000000, 65999999), ("PA", 66000000, 68899999), ("AP", 68900000, 68999999), ("AM", 69000000, 69299999),
    ("RR", 69300000, 69399999), ("AM", 69400000, 69899999), ("AC", 69900000, 69999999), ("DF", 70000000, 72799999),
    ("GO", 72800000, 72999999), ("DF", 73000000, 73699999), ("GO", 73700000, 76799999), ("RO", 76800000, 76999999),
    ("TO", 77000000, 77999999), ("MT", 78000000, 78899999), ("MS", 79000000, 79999999), ("PR", 80000000, 87999999),
    ("SC", 88000000, 89999999), ("RS", 90000000, 99999999),
]

# --- Apelidos de cidades (aplicado ao campo cidade normalizado) -----------------------------------  # >>> AJUSTAR
ALIASES_CIDADE = {
    "SP": "SAO PAULO", "SAMPA": "SAO PAULO", "S PAULO": "SAO PAULO", "SPAULO": "SAO PAULO", "SAO PAULO SP": "SAO PAULO", "SAO PAULO CAPITAL": "SAO PAULO",
    "RIO": "RIO DE JANEIRO", "RJ": "RIO DE JANEIRO", "BH": "BELO HORIZONTE", "B HORIZONTE": "BELO HORIZONTE", "POA": "PORTO ALEGRE",
    "P ALEGRE": "PORTO ALEGRE", "BSB": "BRASILIA", "SSA": "SALVADOR", "FLORIPA": "FLORIANOPOLIS", "SJC": "SAO JOSE DOS CAMPOS",
    "S J DOS CAMPOS": "SAO JOSE DOS CAMPOS", "S JOSE DOS CAMPOS": "SAO JOSE DOS CAMPOS", "SBC": "SAO BERNARDO DO CAMPO",
    "S B DO CAMPO": "SAO BERNARDO DO CAMPO", "S BERNARDO DO CAMPO": "SAO BERNARDO DO CAMPO", "S ANDRE": "SANTO ANDRE",
    "STO ANDRE": "SANTO ANDRE", "S CAETANO DO SUL": "SAO CAETANO DO SUL", "SCS": "SAO CAETANO DO SUL", "GRU": "GUARULHOS",
    "MOGI": "MOGI DAS CRUZES", "S GONCALO": "SAO GONCALO", "N IGUACU": "NOVA IGUACU", "D DE CAXIAS": "DUQUE DE CAXIAS",
    "S LUIS": "SAO LUIS", "S LUIZ": "SAO LUIS", "SAO LUIZ": "SAO LUIS", "F DE SANTANA": "FEIRA DE SANTANA", "J PESSOA": "JOAO PESSOA",
    "S JOSE DO RIO PRETO": "SAO JOSE DO RIO PRETO", "S J DO RIO PRETO": "SAO JOSE DO RIO PRETO", "RIBEIRAO": "RIBEIRAO PRETO",
    "S CARLOS": "SAO CARLOS", "S VICENTE": "SAO VICENTE", "P GROSSA": "PONTA GROSSA", "F IGUACU": "FOZ DO IGUACU",
    "V VELHA": "VILA VELHA", "C GRANDE": "CAMPO GRANDE", "CG": "CAMPO GRANDE", "J DE FORA": "JUIZ DE FORA",
}

# --- Colunas Spark prontas (mapas) ------------------------------------------------------------------------------
from pyspark.sql import functions as F, types as T, Window

def _mapa_col(d: dict):
    itens = []
    for k, v in d.items():
        itens += [F.lit(k), F.lit(v)]
    return F.create_map(*itens)

MAPA_TIPOS_CLIENTE_COL = _mapa_col(MAPA_TIPOS_CLIENTE)
MAPA_TIPOS_CORREIOS_COL = _mapa_col(MAPA_TIPOS_CORREIOS)
MAPA_TITULOS_COL = _mapa_col(MAPA_TITULOS)
MAPA_COMPL_COL = _mapa_col(MAPA_COMPL)
MAPA_UF_COL = _mapa_col(MAPA_UF)
MAPA_ALIASES_CIDADE_COL = _mapa_col(ALIASES_CIDADE)
ORDEM_COMPL_COL = _mapa_col({k: i for i, k in enumerate(ORDEM_COMPL)})

info(f"referências: {len(MAPA_TIPOS_CLIENTE)} variantes de tipo (cliente), {len(MAPA_TIPOS_CORREIOS)} abreviações DNE, "
     f"{len(MAPA_TITULOS)} títulos, {len(MAPA_COMPL)} tokens de complemento, {len(MAPA_UF)} formas de UF, {len(ALIASES_CIDADE)} aliases de cidade")

## Funções auxiliares (expressões Spark nativas — sem UDF)

Todas as funções abaixo recebem e devolvem **`Column`** e são compostas dentro de `select`/`withColumns`, ou seja, rodam no executor em Scala, sem serialização Python. Isso é o que torna viável aplicar o parsing nos bilhões de variantes.

| Função | O que faz |
|---|---|
| `norm_basica` | maiúsculas, remove acentos (`translate`), mantém `, - /` para o parsing, resto vira espaço |
| `norm_chave` | idem, mas remove toda pontuação (forma de comparação) |
| `norm_local` | `norm_chave` + remove prefixos `BAIRRO`/`CIDADE` + expande `S`→`SAO`, `STO`→`SANTO`, `JD`→`JARDIM`... |
| `fonetica_br` | chave fonética pt-BR (Metaphone-BR simplificado) via cadeia de `regexp_replace` |
| `tokens`, `sem_stopwords`, `chave_tokens` | tokenização, remoção de `DE/DA/DO/DAS/DOS/E`, chave ordenada |
| `forma_reduzida` | regra do DNE: 1º e último token inteiros, do meio só a inicial |
| `expandir_titulos` | `DR`→`DOUTOR`, `N S`→`NOSSA SENHORA`, `S JOAO`→`SAO JOAO` |
| `sim_jaccard`, `sim_contencao`, `sim_lev`, `sim_logradouro` | similaridades nativas (0–1); `sim_logradouro` combina tokens, Levenshtein, fonética e forma reduzida |
| `faixa_uf_expr` | UF esperada pela faixa oficial do CEP |
| `flags_array` | monta array de flags a partir de pares (nome, condição) |

In [ ]:
# ============================================================================
# HELPERS — expressões Spark nativas reutilizadas em todas as etapas
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.column import Column

# --- acentos -> ASCII (translate é 1:1, por isso as listas são construídas em pares) ---------------------------
_PARES_ACENTO = [("ÁÀÂÃÄÅ", "A"), ("ÉÈÊË", "E"), ("ÍÌÎÏ", "I"), ("ÓÒÔÕÖ", "O"), ("ÚÙÛÜ", "U"), ("Ç", "C"), ("Ñ", "N"), ("Ý", "Y"),
                 ("áàâãäå", "a"), ("éèêë", "e"), ("íìîï", "i"), ("óòôõö", "o"), ("úùûü", "u"), ("ç", "c"), ("ñ", "n"), ("ý", "y"),
                 ("º°", "O"), ("ª", "A")]
ACENTOS_DE = "".join(k for k, _ in _PARES_ACENTO)
ACENTOS_PARA = "".join(v * len(k) for k, v in _PARES_ACENTO)
assert len(ACENTOS_DE) == len(ACENTOS_PARA)


def limpar_espacos(c: Column) -> Column:
    return F.trim(F.regexp_replace(c, r"\s+", " "))


def _upper_sem_acento(c: Column) -> Column:
    return F.upper(F.translate(F.coalesce(c.cast("string"), F.lit("")), ACENTOS_DE, ACENTOS_PARA))


def norm_basica(c: Column) -> Column:
    """Maiúsculas, sem acento; mantém ',', '-', '/' (separadores usados no parsing); o resto vira espaço."""
    s = _upper_sem_acento(c)
    s = F.regexp_replace(s, r"[^A-Z0-9,\-/ ]", " ")
    return limpar_espacos(s)


def norm_chave(c: Column) -> Column:
    """Maiúsculas, sem acento, sem pontuação, espaços únicos (forma de comparação)."""
    s = _upper_sem_acento(c)
    s = F.regexp_replace(s, r"[^A-Z0-9 ]", " ")
    return limpar_espacos(s)


def tokens(c: Column) -> Column:
    return F.filter(F.split(F.coalesce(c, F.lit("")), " "), lambda t: t != "")


def sem_stopwords(arr: Column) -> Column:
    return F.filter(arr, lambda t: ~t.isin(STOPWORDS))


def chave_tokens(c: Column) -> Column:
    """Tokens sem stopwords, distintos, ordenados, unidos por espaço — chave de igualdade robusta à ordem."""
    return F.array_join(F.array_sort(F.array_distinct(sem_stopwords(tokens(c)))), " ")


def forma_reduzida(c: Column) -> Column:
    """Regra do DNE: primeiro e último token inteiros; tokens do meio (não stopwords, não numéricos) só a inicial."""
    arr = tokens(c)
    n = F.size(arr)
    red = F.transform(
        arr,
        lambda t, i: F.when(
            (i > 0) & (i < n - 1) & (~t.isin(STOPWORDS)) & (F.length(t) > 1) & (~t.rlike(r"^\d+$")),
            F.substring(t, 1, 1),
        ).otherwise(t),
    )
    return F.array_join(red, " ")


_SANTOS = ("JOAO|JOSE|PAULO|PEDRO|FRANCISCO|LUIS|LUIZ|MIGUEL|SEBASTIAO|BENTO|VICENTE|CARLOS|MATEUS|MATHEUS|MARCOS|LUCAS|JORGE|"
           "ROQUE|GONCALO|CRISTOVAO|DOMINGOS|GABRIEL|JUDAS|LOURENCO|RAFAEL|SALVADOR|TOME|TIAGO|GERALDO|ANTONIO|BERNARDO|"
           "CAETANO|JERONIMO|LEOPOLDO|MANUEL|MANOEL|MATIAS|RAIMUNDO|ROMAO|SIMAO|VALENTIM|LAZARO|BARTOLOMEU|BRAS|CRISTOVAO|"
           "FELIX|FIDELIS|GgONCALO|JOAQUIM|JUDAS|LUCAS|MARTINHO|NICOLAU|PATRICIO|SILVESTRE|TOMAS|VITO")


def expandir_titulos(c: Column) -> Column:
    """DR->DOUTOR etc. (token inteiro); N S/NSA/N SRA -> NOSSA SENHORA; S <santo> -> SAO <santo>."""
    arr = tokens(c)
    exp = F.transform(arr, lambda t: F.coalesce(F.try_element_at(MAPA_TITULOS_COL, t), t))
    s = F.array_join(exp, " ")
    s = F.regexp_replace(s, r"\bN\s?S(?:RA)?\b", "NOSSA SENHORA")
    s = F.regexp_replace(s, r"\bS (?=(?:" + _SANTOS + r")\b)", "SAO ")
    return limpar_espacos(s)


def norm_local(c: Column) -> Column:
    """Normalização de bairro/cidade: remove rótulos, expande abreviações típicas (S->SAO, STO->SANTO, JD->JARDIM...)."""
    s = norm_chave(c)
    s = F.regexp_replace(s, r"^(?:BAIRRO|BR|B|CIDADE|CID|MUNICIPIO|MUN|DISTRITO|DIST)\s+", "")
    s = expandir_titulos(s)
    s = F.regexp_replace(s, r"\bS (?=[A-Z]{3,})", "SAO ")  # em bairro/cidade, "S " isolado é sempre SAO
    return limpar_espacos(s)


def fonetica_br(c: Column) -> Column:
    """Chave fonética pt-BR (Metaphone-BR simplificado). Entrada: texto já em maiúsculas sem acento."""
    s = F.regexp_replace(F.coalesce(c, F.lit("")), r"[^A-Z ]", "")
    regras = [
        (r"\bH", ""),                       # H inicial mudo
        (r"PH", "F"), (r"TH", "T"),
        (r"SCH", "X"), (r"SH", "X"), (r"CH", "X"),
        (r"LH", "L"), (r"NH", "N"),
        (r"SC(?=[EI])", "S"), (r"SS", "S"), (r"Z", "S"), (r"XC", "S"),
        (r"C(?=[EI])", "S"), (r"QU", "K"), (r"Q", "K"), (r"CK", "K"), (r"C", "K"),
        (r"GU(?=[EI])", "1"), (r"G(?=[EI])", "J"), (r"1", "G"),
        (r"W", "V"), (r"Y", "I"), (r"H", ""),
        (r"N\b", "M"),                      # N final soa como M (JOAN ~ JOAM)
        (r"(?<=[A-Z])[AEIOU]", ""),         # remove vogais internas (mantém a inicial)
        (r"([A-Z])\1+", "$1"),              # letras duplicadas
    ]
    for p, r in regras:
        s = F.regexp_replace(s, p, r)
    return limpar_espacos(s)


def fonetica_tokens(arr: Column) -> Column:
    """Aplica a fonética token a token (array) e devolve a chave unida por espaço."""
    return F.array_join(F.filter(F.transform(arr, lambda t: fonetica_br(t)), lambda t: t != ""), " ")


# --- similaridades nativas (0..1) -----------------------------------------------------------------------------
def _div(a: Column, b: Column) -> Column:
    return F.coalesce(F.try_divide(a.cast("double"), b.cast("double")), F.lit(0.0))


def sim_jaccard(a_arr: Column, b_arr: Column) -> Column:
    inter = F.size(F.array_intersect(a_arr, b_arr))
    uni = F.size(F.array_union(a_arr, b_arr))
    return F.when(uni > 0, _div(inter, uni)).otherwise(F.lit(0.0))


def sim_contencao(a_arr: Column, b_arr: Column) -> Column:
    inter = F.size(F.array_intersect(a_arr, b_arr))
    m = F.least(F.size(a_arr), F.size(b_arr))
    base = F.when(m > 0, _div(inter, m)).otherwise(F.lit(0.0))
    # contenção com 1 token só é fraca (BRASIL x BRASIL NOVO): rebaixa
    return F.when(m <= 1, base * 0.7).otherwise(base * 0.95)


def sim_lev(a: Column, b: Column) -> Column:
    L = F.greatest(F.length(a), F.length(b), F.lit(1))
    return F.when(a.isNull() | b.isNull() | (a == "") | (b == ""), F.lit(0.0)).otherwise(
        F.lit(1.0) - _div(F.levenshtein(a, b), L)
    )


def sim_logradouro(a_nome, a_tok, a_fon, a_red_chave, b_nome, b_tok, b_fon, b_red_chave) -> Column:
    """Similaridade combinada entre dois nomes de logradouro (sem tipo). 0 se algum lado vazio."""
    vazio = a_nome.isNull() | b_nome.isNull() | (a_nome == "") | (b_nome == "")
    j = sim_jaccard(a_tok, b_tok)
    ct = sim_contencao(a_tok, b_tok)
    lv = sim_lev(a_nome, b_nome)
    red_eq = F.when((a_red_chave == b_red_chave) & (F.length(a_red_chave) > 2), F.lit(0.95)).otherwise(F.lit(0.0))
    fon_eq = F.when((a_fon == b_fon) & (F.length(a_fon) > 2), F.lit(0.90)).otherwise(F.lit(0.0))
    # nome do cliente reduzido == nome dos correios (ou vice-versa) em forma de chave
    red_cross = F.when(
        ((chave_tokens(forma_reduzida(a_nome)) == b_red_chave) | (chave_tokens(forma_reduzida(b_nome)) == a_red_chave))
        & (F.length(a_red_chave) > 2), F.lit(0.93)).otherwise(F.lit(0.0))
    return F.when(vazio, F.lit(0.0)).otherwise(F.round(F.greatest(j, ct, lv, red_eq, fon_eq, red_cross), 4))


# --- faixa de CEP -> UF ---------------------------------------------------------------------------------------
def faixa_uf_expr(cep_int: Column) -> Column:
    expr = None
    for uf, ini, fim in FAIXAS_CEP_UF:
        cond = (cep_int >= ini) & (cep_int <= fim)
        expr = F.when(cond, F.lit(uf)) if expr is None else expr.when(cond, F.lit(uf))
    return expr.otherwise(F.lit(None).cast("string"))


# --- flags -----------------------------------------------------------------------------------------------------
def flags_array(*pares) -> Column:
    """flags_array(("NOME", cond), ...) -> array<string> só com as flags cuja condição é verdadeira."""
    return F.array_compact(F.array(*[F.when(F.coalesce(cond, F.lit(False)), F.lit(nome)) for nome, cond in pares]))


def nulo_se_vazio(c: Column) -> Column:
    return F.nullif(limpar_espacos(F.coalesce(c, F.lit(""))), F.lit(""))


info("helpers carregados")

## (Opcional) Dados sintéticos para smoke test — `MODO_TESTE_SINTETICO=True`

Gera, **no driver**, uma mini `tb_correios` (≈ 60 logradouros em 5 municípios, com CEPs por lado par/ímpar e um município com CEP único de localidade) e uma `tb_endereco` com `N_CPFS_SINTETICO` CPFs, cada um com 1–2 endereços verdadeiros e 3–14 variantes sujas por endereço, reproduzindo **todas** as patologias descritas na documentação (endereço inteiro no logradouro, CEP sem zero à esquerda, CEP inteiro na parte 1, CEP aleatório, tipo omitido/abreviado, nome reduzido, complemento nulo/abreviado, acentos, bairro/cidade/CEP embutidos, número com marcador, S/N, sufixo de letra, erro de digitação).

As duas ficam como **temp views** e o notebook passa a apontar para elas. Um gabarito (`tb_gabarito_sintetico`) guarda o endereço verdadeiro de cada linha para a etapa E9 medir pureza de cluster e acerto por campo. **Não roda** quando `MODO_TESTE_SINTETICO=False`.

In [ ]:
# ============================================================================
# DADOS SINTÉTICOS (só com MODO_TESTE_SINTETICO=True)
# ============================================================================
if MODO_TESTE_SINTETICO:
    import random
    rnd = random.Random(42)
    N_CPFS_SINTETICO = param("N_CPFS_SINTETICO", 400)

    # ---- mini base dos Correios --------------------------------------------------------------------------
    municipios = [  # (uf, municipio, prefixo cep, bairros)
        ("SP", "SAO PAULO", "013", ["BELA VISTA", "CONSOLACAO", "JARDIM PAULISTA"]),
        ("SP", "CAMPINAS", "130", ["CENTRO", "CAMBUI", "TAQUARAL"]),
        ("RJ", "RIO DE JANEIRO", "220", ["COPACABANA", "IPANEMA", "BOTAFOGO"]),
        ("MG", "BELO HORIZONTE", "301", ["CENTRO", "SAVASSI", "FUNCIONARIOS"]),
        ("PR", "CURITIBA", "800", ["CENTRO", "BATEL", "AGUA VERDE"]),
    ]
    nomes = ["ADINEI EMIDIO DE ALMEIDA", "JOSE MARIA DA SILVA", "QUINZE DE NOVEMBRO", "SANTO ANTONIO", "SAO JOAO BATISTA",
             "DOUTOR ARNALDO", "PROFESSOR JOAQUIM NABUCO", "CORONEL PEDRO FERREIRA", "BRIGADEIRO LUIS ANTONIO",
             "PAULISTA", "NOSSA SENHORA DA CONCEICAO", "SETE DE SETEMBRO", "MARECHAL DEODORO DA FONSECA",
             "GENERAL CARNEIRO", "VISCONDE DE PIRAJA", "BARAO DE ITAPETININGA", "ANTONIO CARLOS MAGALHAES",
             "FRANCISCO XAVIER DE OLIVEIRA", "JOAO PESSOA", "PADRE ANCHIETA", "SENADOR FEIJO", "TIRADENTES",
             "GETULIO VARGAS", "PRESIDENTE KENNEDY", "PEDRO ALVARES CABRAL", "AUGUSTA", "HADDOCK LOBO",
             "MARIA ANTONIA", "ENGENHEIRO LUIS CARLOS BERRINI", "AFONSO PENA", "VINTE E CINCO DE MARCO",
             "SANTA LUZIA", "CAPITAO MANOEL DE SOUZA", "TENENTE JOSE DIAS", "DONA ANA NERI", "BENJAMIN CONSTANT"]
    tipos_dne = ["R", "AV", "TV", "AL", "PC"]
    correios_rows, cep_seq = [], 100
    ruas = []  # (uf, municipio, tipo_abrev, nome, bairro, [ceps: (cep8, lado)])
    for uf, mun, pref, bairros in municipios:
        for i in range(12):
            nome = nomes[(i * 3 + len(mun)) % len(nomes)] if i else "PAULISTA"
            nome = f"{nome} {i}" if i >= len(nomes) else nome
            tipo = tipos_dne[i % len(tipos_dne)]
            bairro = bairros[i % len(bairros)]
            ceps = []
            if i % 4 == 0:  # CEP por lado par/ímpar
                for lado in ("P", "I"):
                    cep8 = f"{pref}{cep_seq:05d}"
                    cep_seq += 1
                    correios_rows.append((cep8[:5], cep8[5:], nome, tipo, lado, uf, mun, bairro))
                    ceps.append((cep8, lado))
            else:
                cep8 = f"{pref}{cep_seq:05d}"
                cep_seq += 1
                correios_rows.append((cep8[:5], cep8[5:], nome, tipo, None, uf, mun, bairro))
                ceps.append((cep8, "A"))
            ruas.append((uf, mun, tipo, nome, bairro, ceps))
    # município pequeno com CEP único de localidade (sem logradouro)
    correios_rows.append(("13820", "000", None, None, None, "SP", "JAGUARIUNA", None))
    df_correios_sint = spark.createDataFrame(
        correios_rows, "cep_parte1 string, cep_parte2 string, logradouro_correios string, tipo string, lado string, uf string, municipio string, bairro string")
    df_correios_sint.createOrReplaceTempView("tb_correios_sintetica")

    # ---- variantes sujas ------------------------------------------------------------------------------------
    TIPO_EXT = {"R": "RUA", "AV": "AVENIDA", "TV": "TRAVESSA", "AL": "ALAMEDA", "PC": "PRACA"}
    TIPO_VAR = {"R": ["RUA", "R", "R.", "Rua", ""], "AV": ["AVENIDA", "AV", "AV.", "Av", "AVN", ""],
                "TV": ["TRAVESSA", "TV", "TRAV", ""], "AL": ["ALAMEDA", "AL", "AL.", ""], "PC": ["PRACA", "PÇA", "PC", "Praça", ""]}
    ACENTUAR = {"SAO": "São", "JOAO": "João", "ANTONIO": "Antônio", "CONCEICAO": "Conceição", "LUIS": "Luís",
                "MARCO": "Março", "PIRAJA": "Pirajá", "GETULIO": "Getúlio", "AUGUSTA": "Augusta", "JOSE": "José"}

    def reduzir(nome):
        t = nome.split()
        if len(t) <= 2:
            return nome
        return " ".join([t[0]] + [w if (w in STOPWORDS or len(w) < 2) else w[0] for w in t[1:-1]] + [t[-1]])

    def acentuar_lower(s):
        return " ".join(ACENTUAR.get(w, w.capitalize()) for w in s.split())

    def typo(s):
        if len(s) < 6:
            return s
        i = rnd.randrange(1, len(s) - 2)
        if s[i] == " " or s[i + 1] == " ":
            return s
        return s[:i] + s[i + 1] + s[i] + s[i + 2:]

    def gerar_variantes(rua, numero, compl, n):
        uf, mun, tipo, nome, bairro, ceps = rua
        # escolhe o CEP correto pela paridade
        cep_ok = next((c for c, l in ceps if l == "A" or (l == "P" and numero % 2 == 0) or (l == "I" and numero % 2 == 1)), ceps[0][0])
        out = []
        for _ in range(n):
            t = rnd.choice(TIPO_VAR[tipo])
            nm = nome if rnd.random() < 0.55 else reduzir(nome)
            if rnd.random() < 0.15:
                nm = typo(nm)
            if rnd.random() < 0.3:
                nm = acentuar_lower(nm)
            log = f"{t} {nm}".strip()
            num = str(numero)
            cp = compl
            forma = rnd.random()
            if forma < 0.25:      # tudo no logradouro
                marc = rnd.choice([", ", " N ", " Nº ", " ", " NUM "])
                log = f"{log}{marc}{numero}" + (f" {compl}" if compl and rnd.random() < 0.8 else "")
                num, cp = (None, None) if rnd.random() < 0.8 else (num, None)
                if rnd.random() < 0.3:
                    log += f" - {bairro} - {mun} {uf}"
                if rnd.random() < 0.2:
                    log += f" CEP {cep_ok[:5]}-{cep_ok[5:]}"
            elif forma < 0.35:   # número com sufixo/letra ou S/N
                num = rnd.choice([f"{numero}A", f"{numero} A", "S/N", "SN", "0"])
            if cp and rnd.random() < 0.4:
                cp = cp.replace("APTO", rnd.choice(["AP", "APT", "APARTAMENTO", "apto."])).replace("BLOCO", rnd.choice(["BL", "BLC", "Bloco"]))
            if rnd.random() < 0.3:
                cp = None
            # CEP
            r = rnd.random()
            if r < 0.45:
                p1, p2 = cep_ok[:5], cep_ok[5:]
            elif r < 0.55:
                p1, p2 = cep_ok[:5].lstrip("0"), cep_ok[5:]          # zero à esquerda perdido
            elif r < 0.65:
                p1, p2 = cep_ok, None                                  # cep inteiro na parte 1
            elif r < 0.72:
                p1, p2 = f"{int(cep_ok):d}"[:-3].lstrip("0"), cep_ok[5:].lstrip("0")  # ambos sem zeros
            elif r < 0.85:
                p1, p2 = f"{rnd.randrange(1000, 99999):05d}", f"{rnd.randrange(0, 999):03d}"  # aleatório
            elif r < 0.92:
                p1, p2 = None, None
            else:
                p1, p2 = cep_ok[:5], cep_ok[5:][::-1]                  # parte 2 embaralhada
            bairro_v = rnd.choice([bairro, bairro.title(), None, bairro, "JD " + bairro if rnd.random() < 0.2 else bairro])
            cidade_v = rnd.choice([mun, mun.title(), f"{mun} - {uf}", None, mun])
            if mun == "SAO PAULO" and rnd.random() < 0.3:
                cidade_v = rnd.choice(["S PAULO", "SP", "São Paulo"])
            uf_v = rnd.choice([uf, uf.lower(), None, uf, {"SP": "SAO PAULO", "RJ": "RIO DE JANEIRO"}.get(uf, uf)])
            dt = datetime.datetime(2018, 1, 1) + datetime.timedelta(days=rnd.randrange(0, 3000), hours=rnd.randrange(0, 24))
            out.append((log, num, cp, bairro_v, cidade_v, uf_v, p1, p2, dt))
        return out, cep_ok

    linhas, gabarito = [], []
    n_mesmo_predio = 0
    for k in range(N_CPFS_SINTETICO):
        cpf = f"{10000000000 + k * 7919:011d}"
        n_end = 1 if rnd.random() < 0.7 else 2
        # cenário crítico: 2 unidades no MESMO prédio (mesma rua, número e CEP; só o complemento muda)
        mesmo_predio = (n_end == 2) and (rnd.random() < 0.6)
        rua_prev = numero_prev = ap_prev = None
        for e in range(n_end):
            if mesmo_predio and e == 1:
                rua, numero = rua_prev, numero_prev
                compl = f"APTO {ap_prev + 1}" + (f" BLOCO {rnd.choice('ABCD')}" if rnd.random() < 0.5 else "")
                n_mesmo_predio += 1
            else:
                rua = rnd.choice(ruas)
                numero = rnd.randrange(1, 2500)
                if mesmo_predio:
                    ap_prev = rnd.randrange(1, 200)
                    compl = f"APTO {ap_prev}" + (f" BLOCO {rnd.choice('ABCD')}" if rnd.random() < 0.5 else "")
                else:
                    compl = rnd.choice([None, None, f"APTO {rnd.randrange(1, 200)}", f"APTO {rnd.randrange(1, 200)} BLOCO {rnd.choice('ABCD')}",
                                        "CASA 2", "FUNDOS", f"SALA {rnd.randrange(1, 50)}"])
                rua_prev, numero_prev = rua, numero
            variantes, cep_ok = gerar_variantes(rua, numero, compl, rnd.randrange(3, 15))
            id_end = f"{cpf}_{e}"
            for v in variantes:
                rep = 1 if rnd.random() < 0.7 else rnd.randrange(2, 6)  # linhas duplicadas idênticas
                for _ in range(rep):
                    linhas.append((cpf,) + v)
                    gabarito.append((cpf,) + v + (id_end, f"{TIPO_EXT[rua[2]]} {rua[3]}", str(numero), cep_ok, rua[4], rua[1], rua[0], compl))
    # dt_atualizacao: coluna de data opcional — para testar a recência, rode com COL_DATA="dt_atualizacao"
    esquema_end = ("idCPF string, logradouro string, numero string, complemento string, bairro string, cidade string, uf string, "
                   "cep_parte1 string, cep_parte2 string, dt_atualizacao timestamp")
    spark.createDataFrame(linhas, esquema_end).createOrReplaceTempView("tb_endereco_sintetica")
    spark.createDataFrame(gabarito, esquema_end + ", id_end_verdade string, logradouro_verdade string, numero_verdade string, cep_verdade string, bairro_verdade string, cidade_verdade string, uf_verdade string, complemento_verdade string") \
        .dropDuplicates().createOrReplaceTempView("tb_gabarito_sintetico")
    info(f"sintético: {n_mesmo_predio} CPFs com 2 unidades no mesmo prédio (teste de não-fusão de apartamentos)")

    NOME_TB_ENDERECO = "tb_endereco_sintetica"
    NOME_TB_CORREIOS = "tb_correios_sintetica"
    COLS_ENDERECO = {k: k for k in COLS_ENDERECO}
    COLS_CORREIOS.update({"cep_parte1": "cep_parte1", "cep_parte2": "cep_parte2", "logradouro": "logradouro_correios", "tipo": "tipo",
                          "lado": "lado", "uf": "uf", "municipio": "municipio", "bairro": "bairro", "logradouro_reduzido": None,
                          "numero_inicial": None, "numero_final": None})
    info(f"sintético: {len(linhas)} linhas de endereço, {len(correios_rows)} linhas correios, {N_CPFS_SINTETICO} CPFs -> views {NOME_TB_ENDERECO}/{NOME_TB_CORREIOS}")
    display(spark.table(NOME_TB_ENDERECO).limit(12))
else:
    info("MODO_TESTE_SINTETICO=False — usando as tabelas reais do ambiente ativo")
    verificar_ambiente(lancar_erro=True)   # falha cedo se tabela/coluna mapeada não existir

## E1 — Leitura, escopo (amostra/lotes) e dedup em variantes

1. Lê `tb_endereco` renomeando as colunas reais para os nomes internos (tudo como `string`).
2. Aplica o escopo: lista explícita de CPFs **ou** fração por `pmod(xxhash64(idCPF), 1e6)` (mantém todas as linhas de um CPF juntas) **e** lote por `pmod(xxhash64(idCPF), N_LOTES)`.
3. Calcula `hash_linha = xxhash64(idCPF, campos brutos)` — é a chave para devolver a sugestão a cada linha original em E9 (linhas idênticas compartilham o hash e a sugestão, o que é o comportamento desejado).
4. Agrupa por (idCPF, hash_linha, campos brutos) → **variantes** com `qtd_ocorrencias` (e `data_max` se houver `COL_DATA`). Nas bases reais isso costuma reduzir o volume em 3–10×.

In [ ]:
# ============================================================================
# E1 — leitura, escopo e dedup em variantes
# ============================================================================
def ler_endereco_bruto():
    """Lê tb_endereco com colunas renomeadas para os nomes internos, aplica amostra/lote e calcula hash_linha."""
    df = spark.table(NOME_TB_ENDERECO)
    sel = []
    for interno, real in COLS_ENDERECO.items():
        if real is None:
            sel.append(F.lit(None).cast("string").alias(interno))
        else:
            sel.append(F.col(f"`{real}`").cast("string").alias(interno))
    if COL_DATA:
        sel.append(F.col(f"`{COL_DATA}`").cast("timestamp").alias("data_ref"))
    df = df.select(*sel)
    h = F.xxhash64(F.col("idCPF"))
    if LISTA_CPFS_AMOSTRA:
        df = df.filter(F.col("idCPF").isin([c.strip() for c in LISTA_CPFS_AMOSTRA.split(",") if c.strip()]))
    elif MODO_AMOSTRA:
        df = df.filter(F.pmod(h, F.lit(1_000_000)) < F.lit(int(FRACAO_AMOSTRA * 1_000_000)))
    if N_LOTES > 1:
        df = df.filter(F.pmod(h, F.lit(N_LOTES)) == F.lit(LOTE_ATUAL))
    df = df.withColumn("hash_linha", F.xxhash64(*[F.coalesce(F.col(c), F.lit("<NULO>")) for c in ["idCPF"] + CAMPOS_ENDERECO]))
    return df


if etapa_ativa("E1"):
    t0 = time.time()
    df_bruto = ler_endereco_bruto()
    aggs = [F.count(F.lit(1)).alias("qtd_ocorrencias")]
    if COL_DATA:
        aggs.append(F.max("data_ref").alias("data_max"))
    df_var = (
        df_bruto.groupBy("idCPF", "hash_linha", *CAMPOS_ENDERECO)
        .agg(*aggs)
        .select("idCPF", "hash_linha", "qtd_ocorrencias", *(["data_max"] if COL_DATA else []),
                *[F.col(c).alias(f"{c}_raw") for c in CAMPOS_ENDERECO])
    )
    df_var = persistir(df_var, "01_variantes", cluster_by="idCPF")
    contar(df_bruto, "E1 linhas brutas no escopo")
    contar(df_var, "E1 variantes distintas")
    info(f"E1 concluída em {time.time() - t0:.0f}s")
    display(df_var.orderBy("idCPF").limit(12))
else:
    info("E1 pulada (lendo _01_variantes persistida quando necessário)")

## E2 — Normalização e parsing (Spark nativo)

Transforma cada variante em campos estruturados e chaves de comparação. Sub-etapas, na ordem em que o código as aplica:

- **A. Normalização básica** de todos os campos (`norm_basica`/`norm_chave`/`norm_local`), UF via `MAPA_UF`, cidade via `ALIASES_CIDADE` e remoção do sufixo `- UF`.
- **B. CEP**: remonta `cep8` a partir das partes cobrindo os casos (5+3, 8+vazio, 8+3 conflitante, vazio+8, partes curtas → `lpad` de zeros, concatenação com 7 dígitos, 5+vazio → incerto); anula padrões inválidos (`00000000`, `12345678`); extrai `cep_embutido` do logradouro; checa a faixa oficial da UF.
- **C. Logradouro**: remove CEP e rótulos `CEP`; converte `S/N` → `SN`; separa em segmentos por `,` / ` - ` e descarta segmentos que são o bairro, a cidade ou a UF da própria linha; protege datas (`15 DE NOVEMBRO`); corta o **complemento** na primeira palavra-chave com valor (`APTO 45`, `BL B`) ou solitária (`FUNDOS`); extrai o **número** (com marcador `N/Nº/NUM` ou no final); reverte a extração se o nome ficaria vazio (`RUA 7`); identifica o **tipo** pelo 1º token; expande títulos; gera `nome_chave`, `nome_fon`, `nome_red`, `nome_red_chave`.
- **D. Número (campo)**: dígitos sem zeros à esquerda, `SN`, sufixo de letra, `KM` vira complemento; coalesce entre campo / extraído do logradouro / marcador dentro do complemento; flag de conflito.
- **E. Complemento**: junta campo + o que foi extraído do logradouro; canoniza palavras-chave (`AP`→`APTO`); descola `APTO45`; `3 ANDAR`→`ANDAR 3`; extrai `compl_pares` (`APTO 45`, `BLOCO B`, `FUNDOS`) e `compl_livre` (sobra).
- **F. Flags** de tudo que foi corrigido/detectado.

No fim imprime os tokens de complemento mais frequentes que **não** estão no dicionário (para você estender `MAPA_COMPL`).

In [ ]:
# ============================================================================
# E2 — normalização e parsing das variantes
# ============================================================================
# --- regexes compartilhadas (Java regex; strings raw do Python vão direto ao Spark) -----------------------------
_RE_MESES = "|".join(MESES)
RE_DATA_PROTEGE = r"\b(\d{1,2}) DE (" + _RE_MESES + r")\b"
RE_MARCADOR_NUM = r"\b(?:" + "|".join(MARCADORES_NUMERO) + r")\s*[.:]?\s*(\d{1,6})\s*([A-Z])?\b"
RE_MARCADOR_NUM_FULL = r"\b(?:" + "|".join(MARCADORES_NUMERO) + r")\s*[.:]?\s*\d{1,6}\s*(?:[A-Z]\b)?"
RE_NUM_FIM = r"(?:^|[\s,])(\d{1,6})\s*([A-Z])?\s*$"
RE_NUM_FIM_FULL = r"(?:^|[\s,])\d{1,6}\s*(?:[A-Z])?\s*$"

_kw_val = sorted({v for v, k in MAPA_COMPL.items() if k in COMPL_COM_VALOR}, key=len, reverse=True)
_kw_solo = sorted({v for v, k in MAPA_COMPL.items() if k in COMPL_SEM_VALOR} | {"CASA", "APTO"}, key=len, reverse=True)
RE_KW_VAL = r"\b(?:" + "|".join(_kw_val) + r")\s*[.:]?\s*(?:\d{1,5}[A-Z]?|[A-Z]|[IVX]{1,4})\b"
RE_KW_SOLO = r"\b(?:" + "|".join(_kw_solo) + r")\b"
RE_COMPL_SPLIT = r"^(.*?)\s*((?:" + RE_KW_VAL + r"|" + RE_KW_SOLO + r").*)$"
RE_PARES_VAL = r"\b(" + "|".join(COMPL_COM_VALOR) + r")\s+([A-Z]?\d{1,5}[A-Z]?|[A-Z]|[IVX]{1,4})\b"
RE_PARES_SOLO = r"\b(" + "|".join(COMPL_SEM_VALOR) + r")\b"
RE_DESCOLA_KW = r"\b(" + "|".join(sorted({v for v in MAPA_COMPL if v.isalpha()}, key=len, reverse=True)) + r")(\d)"
_UFS_ALT = "|".join(UFS)
RE_KW_QUALQUER = r"\b(?:" + "|".join(sorted(MAPA_COMPL.keys(), key=len, reverse=True)) + r")\b"

# municípios conhecidos (da tb_correios) -> reconhecer "CIDADE UF" embutidos no logradouro e inferir UF pela cidade
try:
    _mun_col, _uf_col = COLS_CORREIOS.get("municipio"), COLS_CORREIOS.get("uf")
    _mun_uf = (spark.table(NOME_TB_CORREIOS)
               .select(nulo_se_vazio(norm_local(F.col(f"`{_mun_col}`"))).alias("m"), F.try_element_at(MAPA_UF_COL, norm_chave(F.col(f"`{_uf_col}`"))).alias("u"))
               .filter(F.col("m").isNotNull() & F.col("u").isNotNull()).distinct()
               .groupBy("m").agg(F.collect_set("u").alias("ufs")).collect()) if _mun_col and _uf_col else []
    _municipios = [r["m"] for r in _mun_uf]
    _mun_uf_unico = {r["m"]: r["ufs"][0] for r in _mun_uf if len(r["ufs"]) == 1}   # município com UF única no país
except Exception as _e:
    info(f"E2 — não foi possível ler municípios da tb_correios ({type(_e).__name__}); seguindo sem a lista")
    _municipios, _mun_uf_unico = [], {}
MUNICIPIOS_MAPA_COL = F.map_from_arrays(F.lit(_municipios), F.lit([1] * len(_municipios))) if _municipios else F.create_map(F.lit("-"), F.lit(1))
MUN_UF_MAPA_COL = (F.map_from_arrays(F.lit(list(_mun_uf_unico.keys())), F.lit(list(_mun_uf_unico.values())))
                   if _mun_uf_unico else F.create_map(F.lit("-"), F.lit("-")))
info(f"E2 — {len(_municipios)} municípios conhecidos ({len(_mun_uf_unico)} com UF única) carregados")


def eh_municipio_conhecido(s: Column) -> Column:
    return F.try_element_at(MUNICIPIOS_MAPA_COL, s).isNotNull()


def strip_sufixo(col: Column, suf: Column) -> Column:
    """Remove ' <suf>' do fim de col quando suf não é nulo (>= 4 chars) e sobra algo antes."""
    cond = suf.isNotNull() & (F.length(suf) >= 4) & col.endswith(F.concat(F.lit(" "), suf)) & (F.length(col) > F.length(suf) + 1)
    return F.when(cond, F.substr(col, F.lit(1), F.length(col) - F.length(suf) - 1)).otherwise(col)


def eh_tipo_ou_stop(t: Column) -> Column:
    return F.try_element_at(MAPA_TIPOS_CLIENTE_COL, t).isNotNull() | t.isin(STOPWORDS)


def qtd_tokens_nome(c: Column) -> Column:
    """Quantos tokens do texto NÃO são tipo de logradouro nem stopword (para decidir se sobrou 'nome')."""
    return F.size(F.filter(tokens(c), lambda t: ~eh_tipo_ou_stop(t)))


if etapa_ativa("E2"):
    t0 = time.time()
    df = obter("01_variantes")

    # ---------------- A. normalização básica ---------------------------------------------------------------
    df = df.withColumns({
        "log_b": norm_basica(F.col("logradouro_raw")),
        "num_b": norm_chave(F.col("numero_raw")),
        "compl_b": norm_chave(F.col("complemento_raw")),
        "bairro_n": nulo_se_vazio(norm_local(F.col("bairro_raw"))),
        "cidade_n0": nulo_se_vazio(norm_local(F.col("cidade_raw"))),
        "uf_n0": nulo_se_vazio(norm_chave(F.col("uf_raw"))),
    })
    uf_n0 = F.try_element_at(MAPA_UF_COL, F.col("uf_n0"))
    cidade_sem_uf = limpar_espacos(F.regexp_replace(F.col("cidade_n0"), r"\s(?:" + _UFS_ALT + r")$", ""))
    cidade_n = F.coalesce(F.try_element_at(MAPA_ALIASES_CIDADE_COL, cidade_sem_uf), cidade_sem_uf)
    # UF ausente: infere pelo campo cidade (nome/sigla de UF no lugar da cidade) ou por município de UF única
    uf_n = F.coalesce(uf_n0, F.try_element_at(MAPA_UF_COL, F.col("cidade_n0")), F.try_element_at(MUN_UF_MAPA_COL, nulo_se_vazio(cidade_n)))
    df = df.withColumns({
        "uf_n": uf_n,
        "cidade_n": nulo_se_vazio(cidade_n),
        "flag_cidade_alias": F.try_element_at(MAPA_ALIASES_CIDADE_COL, cidade_sem_uf).isNotNull(),
        "flag_uf_invalida": F.col("uf_n0").isNotNull() & uf_n0.isNull(),
        "flag_uf_inferida": uf_n0.isNull() & uf_n.isNotNull(),
    })

    # ---------------- B. CEP ---------------------------------------------------------------------------------
    p1 = F.regexp_replace(F.coalesce(F.col("cep_parte1_raw"), F.lit("")), r"[^0-9]", "")
    p2 = F.regexp_replace(F.coalesce(F.col("cep_parte2_raw"), F.lit("")), r"[^0-9]", "")
    l1, l2 = F.length(p1), F.length(p2)
    cc = F.concat(p1, p2)

    def _s(cep, origem):
        return F.struct(cep.alias("cep"), F.lit(origem).alias("origem"))

    cep_calc = (
        F.when((l1 == 5) & (l2 == 3), _s(cc, "CAMPOS"))
        .when((l1 == 8) & ((l2 == 0) | (F.substring(p1, 6, 3) == p2)), _s(p1, "P1_COMPLETO"))
        .when((l1 == 8) & (l2 == 3), _s(p1, "P1_COMPLETO_P2_CONFLITANTE"))
        .when(l2 == 8, _s(p2, "P2_COMPLETO"))
        .when((l1 >= 1) & (l1 <= 5) & (l2 >= 1) & (l2 <= 3), _s(F.concat(F.lpad(p1, 5, "0"), F.lpad(p2, 3, "0")), "ZEROS_CORRIGIDOS"))
        .when(l1.isin(6, 7) & (l2 == 0), _s(F.lpad(p1, 8, "0"), "ZEROS_CORRIGIDOS"))
        .when(F.length(cc) == 8, _s(cc, "CONCAT"))
        .when(F.length(cc) == 7, _s(F.lpad(cc, 8, "0"), "ZEROS_CORRIGIDOS"))
        .when((l1 == 5) & (l2 == 0), _s(F.concat(p1, F.lit("000")), "SUFIXO_000_INCERTO"))
        .otherwise(_s(F.lit(None).cast("string"), "AUSENTE"))
    )
    cep_embutido = F.regexp_replace(
        F.regexp_extract(F.col("log_b"), r"(?<!\d)(\d{5}\s?[-.]?\s?\d{3})(?!\d)", 1), r"[^0-9]", "")
    padrao_invalido = lambda c: c.rlike(r"^(\d)\1{7}$") | c.isin("12345678", "87654321", "01234567")
    df = df.withColumn("cep_calc", cep_calc).withColumns({
        "cep8_0": F.col("cep_calc.cep"),
        "cep_origem_0": F.col("cep_calc.origem"),
        "cep_embutido": F.when((F.length(cep_embutido) == 8) & ~padrao_invalido(cep_embutido), cep_embutido),
    })
    df = df.withColumns({
        "cep8": F.when(padrao_invalido(F.col("cep8_0")), F.lit(None).cast("string")).otherwise(F.col("cep8_0")),
        "cep_origem": F.when(padrao_invalido(F.col("cep8_0")), F.lit("PADRAO_INVALIDO")).otherwise(F.col("cep_origem_0")),
    })
    df = df.withColumns({
        "cep_int": F.col("cep8").try_cast("bigint"),
        "cep_embutido_int": F.col("cep_embutido").try_cast("bigint"),
    })
    df = df.withColumns({
        "cep_uf_faixa": faixa_uf_expr(F.col("cep_int")),
        "cep_embutido_uf_faixa": faixa_uf_expr(F.col("cep_embutido_int")),
    })
    df = df.withColumn("cep_faixa_uf_ok", F.when(F.col("cep8").isNull() | F.col("uf_n").isNull(), F.lit(None).cast("boolean"))
                       .otherwise(F.col("cep_uf_faixa") == F.col("uf_n")))

    # ---------------- C. parsing do logradouro ---------------------------------------------------------------
    log1 = F.regexp_replace(F.col("log_b"), r"\bCEP\b\s*[:.\-]?\s*", " ")
    log1 = F.regexp_replace(log1, r"(?<!\d)\d{5}\s?[-.]?\s?\d{3}(?!\d)", " ")            # CEP embutido
    log1 = F.regexp_replace(log1, r"\bS\s*/\s*N[O]?\b|\bS/N\b", " SN ")                    # S/N
    log1 = F.regexp_replace(log1, r"\b(?:" + "|".join(MARCADORES_NUMERO) + r")\s*[.:]\s*(?=\d)", "N ")  # "N.: 123" -> "N 123"
    segs = F.split(limpar_espacos(log1), r"\s*,\s*|\s+-\s+|\s*;\s*")
    segs = F.transform(segs, lambda s: limpar_espacos(F.regexp_replace(s, r"[^A-Z0-9 ]", " ")))
    head = F.get(segs, 0)
    tail = F.slice(segs, 2, 100)
    cabeca_tem_nome = qtd_tokens_nome(head) > 0
    tail_filtrado = F.filter(
        tail,
        lambda s: ~(
            (s == "")
            # segmento só de letras depois do nome (sem dígito, sem palavra-chave de complemento, não é SN):
            # bairro / cidade / ponto de referência -> descarta (o nome da rua está no 1º segmento)
            | (cabeca_tem_nome & ~s.rlike(r"\d") & ~s.rlike(RE_KW_QUALQUER) & (s != "SN"))
            | (s == F.coalesce(F.col("bairro_n"), F.lit("")))
            | (s == F.coalesce(F.col("cidade_n"), F.lit("")))
            | (s == F.coalesce(F.col("uf_n"), F.lit("")))
            | (s == F.concat_ws(" ", F.col("cidade_n"), F.col("uf_n")))
            | s.isin(UFS)
            | s.rlike(r"^(?:BAIRRO|CIDADE|CID|MUNICIPIO|UF)\b")
            | (F.try_element_at(MAPA_UF_COL, s).isNotNull() & (F.length(s) > 2))
            # "SAO PAULO SP" / "CAMPINAS" sem cidade informada: município conhecido dos Correios (com ou sem UF no fim)
            | (~s.rlike(r"\d") & ~s.rlike(RE_KW_QUALQUER)
               & (eh_municipio_conhecido(s) | eh_municipio_conhecido(F.regexp_replace(s, r"\s(?:" + _UFS_ALT + r")$", ""))))
        ),
    )
    df = df.withColumns({
        "log_head": head,
        "log_tail_n": F.size(tail),
        "log_tail_f": tail_filtrado,
    })
    log2 = limpar_espacos(F.array_join(F.array_compact(F.concat(F.array(F.col("log_head")), F.col("log_tail_f"))), " "))
    log2 = F.regexp_replace(log2, RE_DATA_PROTEGE, "$1_DE_$2")
    df = df.withColumn("log2", log2)

    # C4: separa complemento embutido
    core_pre = F.regexp_extract(F.col("log2"), RE_COMPL_SPLIT, 1)
    compl_ext = F.regexp_extract(F.col("log2"), RE_COMPL_SPLIT, 2)
    df = df.withColumns({"core_pre": core_pre, "compl_ext_0": compl_ext})
    reverte_compl = (F.col("compl_ext_0") != "") & (qtd_tokens_nome(F.col("core_pre")) == 0)
    df = df.withColumns({
        "log3": F.when((F.col("compl_ext_0") != "") & ~reverte_compl, F.col("core_pre")).otherwise(F.col("log2")),
        "compl_ext": F.when(reverte_compl, F.lit("")).otherwise(F.col("compl_ext_0")),
    })

    # C5: número embutido (marcador > final)
    num_marc = F.regexp_extract(F.col("log3"), RE_MARCADOR_NUM, 1)
    num_marc_suf = F.regexp_extract(F.col("log3"), RE_MARCADOR_NUM, 2)
    num_fim = F.regexp_extract(F.col("log3"), RE_NUM_FIM, 1)
    num_fim_suf = F.regexp_extract(F.col("log3"), RE_NUM_FIM, 2)
    df = df.withColumns({"num_marc": num_marc, "num_marc_suf": num_marc_suf, "num_fim": num_fim, "num_fim_suf": num_fim_suf})
    log4 = (F.when(F.col("num_marc") != "", F.regexp_replace(F.col("log3"), RE_MARCADOR_NUM_FULL, " "))
            .when(F.col("num_fim") != "", F.regexp_replace(F.col("log3"), RE_NUM_FIM_FULL, " "))
            .otherwise(F.col("log3")))
    df = df.withColumn("log4_0", limpar_espacos(log4))
    reverte_num = ((F.col("num_marc") != "") | (F.col("num_fim") != "")) & (qtd_tokens_nome(F.col("log4_0")) == 0) & (F.col("num_marc") == "")
    df = df.withColumns({
        "numero_ext": F.when(reverte_num, F.lit(None).cast("string"))
                       .otherwise(F.nullif(F.coalesce(F.nullif(F.col("num_marc"), F.lit("")), F.nullif(F.col("num_fim"), F.lit(""))), F.lit(""))),
        "numero_ext_suf": F.when(reverte_num, F.lit(None).cast("string"))
                           .when(F.col("num_marc") != "", F.nullif(F.col("num_marc_suf"), F.lit("")))
                           .otherwise(F.nullif(F.col("num_fim_suf"), F.lit(""))),
        "log4": F.when(reverte_num, F.col("log3")).otherwise(F.col("log4_0")),
    })
    # C6/C7: SN no logradouro, desprotege datas, limpa, tira "BAIRRO"/"CIDADE UF" colados no fim (sem separador)
    df = df.withColumn("tem_sn_log", F.col("log4").rlike(r"\bSN\b|\bSEM NUMERO\b"))
    log_core = F.regexp_replace(F.col("log4"), r"\bSN\b|\bSEM NUMERO\b", " ")
    log_core = F.regexp_replace(log_core, "_DE_", " DE ")
    log_core = limpar_espacos(F.regexp_replace(log_core, r"[^A-Z0-9 ]", " "))
    df = df.withColumn("log_core_0", log_core).withColumn("log_core_1", F.col("log_core_0"))
    # cada passo materializa uma coluna (withColumn) em vez de aninhar expressões: strip_sufixo referencia a coluna 5x,
    # e 6 chamadas aninhadas dariam 5^6 cópias da expressão no plano
    for _ in range(2):  # até 2 sufixos (ex.: "... CENTRO SAO PAULO SP")
        df = df.withColumn("log_core_1", strip_sufixo(F.col("log_core_1"), F.concat_ws(" ", F.col("cidade_n"), F.col("uf_n"))))
        df = df.withColumn("log_core_1", strip_sufixo(F.col("log_core_1"), F.col("cidade_n")))
        df = df.withColumn("log_core_1", strip_sufixo(F.col("log_core_1"), F.col("bairro_n")))
    df = df.withColumn("log_core_1", limpar_espacos(F.col("log_core_1")))
    df = df.withColumn("log_core", F.when(qtd_tokens_nome(F.col("log_core_1")) > 0, F.col("log_core_1")).otherwise(F.col("log_core_0")))

    # C8: tipo + nome + chaves
    tok = tokens(F.col("log_core"))
    t0c = F.get(tok, 0)
    tipo_canon = F.try_element_at(MAPA_TIPOS_CLIENTE_COL, t0c)
    nome0 = F.when(tipo_canon.isNotNull(), F.array_join(F.slice(tok, 2, 200), " ")).otherwise(F.col("log_core"))
    df = df.withColumns({"tipo_canon": tipo_canon, "nome_log0": nome0})
    df = df.withColumn("nome_log", nulo_se_vazio(expandir_titulos(F.col("nome_log0"))))
    df = df.withColumn("nome_tok", sem_stopwords(tokens(F.col("nome_log"))))
    df = df.withColumns({
        "nome_chave": F.array_join(F.array_sort(F.array_distinct(F.col("nome_tok"))), " "),
        "nome_fon": fonetica_tokens(F.col("nome_tok")),
        "nome_red": forma_reduzida(F.col("nome_log")),
    })
    df = df.withColumns({
        "nome_red_chave": chave_tokens(F.col("nome_red")),
        "logradouro_limpo": nulo_se_vazio(F.concat_ws(" ", F.col("tipo_canon"), F.col("nome_log"))),
    })

    # ---------------- D. número (campo) -----------------------------------------------------------------------
    num_sn = F.col("num_b").rlike(RE_SEM_NUMERO)
    num_dig = F.regexp_extract(F.col("num_b"), r"(\d{1,6})", 1)
    num_suf = F.regexp_extract(F.col("num_b"), r"^\d{1,6}\s*([A-Z])$", 1)
    num_km = F.col("num_b").rlike(r"\bKM\b")
    numero_campo = (F.when(num_sn, F.lit("SN"))
                    .when((num_dig != "") & ~num_km, num_dig.try_cast("bigint").cast("string"))
                    .otherwise(F.lit(None).cast("string")))
    numero_compl = F.nullif(F.regexp_extract(F.col("compl_b"), RE_MARCADOR_NUM, 1), F.lit(""))
    df = df.withColumns({
        "numero_campo": numero_campo,
        "numero_sufixo_campo": F.nullif(num_suf, F.lit("")),
        "num_km": num_km,
        "numero_compl": numero_compl.try_cast("bigint").cast("string"),
        "numero_ext_norm": F.col("numero_ext").try_cast("bigint").cast("string"),
    })
    numero_num = F.coalesce(
        F.when(F.col("numero_campo") != "SN", F.col("numero_campo")),
        F.col("numero_ext_norm"),
        F.col("numero_compl"),
    )
    df = df.withColumns({
        "numero_final": F.coalesce(numero_num, F.when((F.col("numero_campo") == "SN") | F.col("tem_sn_log"), F.lit("SN"))),
        "flag_numero_conflito": (F.col("numero_campo") != "SN") & F.col("numero_campo").isNotNull()
                                & F.col("numero_ext_norm").isNotNull() & (F.col("numero_campo") != F.col("numero_ext_norm")),
    })
    df = df.withColumns({
        "numero_int": F.col("numero_final").try_cast("bigint"),
        "numero_sufixo": F.coalesce(F.col("numero_sufixo_campo"), F.col("numero_ext_suf")),
    })

    # ---------------- E. complemento --------------------------------------------------------------------------
    df = df.withColumn("compl_ext_n", F.regexp_replace(F.col("compl_ext"), "_DE_", " DE "))
    for _ in range(2):  # bairro/cidade colados depois do complemento embutido ("APTO 4 CENTRO SAO PAULO SP") — um withColumn por passo
        df = df.withColumn("compl_ext_n", strip_sufixo(F.col("compl_ext_n"), F.concat_ws(" ", F.col("cidade_n"), F.col("uf_n"))))
        df = df.withColumn("compl_ext_n", strip_sufixo(F.col("compl_ext_n"), F.col("cidade_n")))
        df = df.withColumn("compl_ext_n", strip_sufixo(F.col("compl_ext_n"), F.col("bairro_n")))
    compl_ext_n = F.col("compl_ext_n")
    compl_all = F.concat_ws(" ", F.col("compl_b"), compl_ext_n,
                            F.when(F.col("num_km") & (num_dig != ""), F.concat(F.lit("KM "), num_dig)))
    compl_all = F.regexp_replace(compl_all, RE_MARCADOR_NUM_FULL, " ")        # "N 123" dentro do complemento
    compl_all = F.regexp_replace(compl_all, r"\bSN\b|\bSEM NUMERO\b", " ")
    compl_all = F.regexp_replace(compl_all, RE_DESCOLA_KW, "$1 $2")           # APTO45 -> APTO 45
    compl_all = limpar_espacos(F.regexp_replace(compl_all, r"[^A-Z0-9 ]", " "))
    compl_canon = F.array_join(F.transform(tokens(compl_all), lambda t: F.coalesce(F.try_element_at(MAPA_COMPL_COL, t), t)), " ")
    compl_canon = F.regexp_replace(compl_canon, r"\b(\d{1,3})\s*O?\s+ANDAR\b", "ANDAR $1")
    compl_canon = F.regexp_replace(compl_canon, r"\b(" + "|".join(COMPL_COM_VALOR) + r")\s+(?:N|NO|NR|NUM|NUMERO)\s+(?=\d)", "$1 ")
    compl_canon = F.regexp_replace(compl_canon, r"\b(" + "|".join(COMPL_COM_VALOR) + r")(?:\s+\1\b)+", "$1")  # APTO APTO 45
    df = df.withColumn("compl_canon", limpar_espacos(compl_canon))
    pares_val = F.regexp_extract_all(F.col("compl_canon"), F.lit(RE_PARES_VAL), F.lit(0))
    pares_solo = F.regexp_extract_all(F.col("compl_canon"), F.lit(RE_PARES_SOLO), F.lit(0))
    df = df.withColumns({"pares_val": pares_val, "pares_solo": pares_solo})
    pares_solo_ok = F.filter(F.col("pares_solo"), lambda s: ~F.exists(F.col("pares_val"), lambda p: p.startswith(F.concat(s, F.lit(" ")))))
    df = df.withColumn("compl_pares", F.array_distinct(F.concat(F.col("pares_val"), pares_solo_ok)))
    compl_livre = F.regexp_replace(F.col("compl_canon"), RE_PARES_VAL, " ")
    compl_livre = F.regexp_replace(compl_livre, RE_PARES_SOLO, " ")
    df = df.withColumns({
        "compl_livre": nulo_se_vazio(compl_livre),
        "compl_chave": F.array_join(F.array_sort(F.col("compl_pares")), " "),
    })

    # ---------------- F. flags --------------------------------------------------------------------------------
    df = df.withColumn("flags_norm", flags_array(
        ("CEP_ZEROS_CORRIGIDOS", F.col("cep_origem") == "ZEROS_CORRIGIDOS"),
        ("CEP_P1_COMPLETO", F.col("cep_origem").startswith("P1_COMPLETO")),
        ("CEP_P2_COMPLETO", F.col("cep_origem") == "P2_COMPLETO"),
        ("CEP_SUFIXO_000_INCERTO", F.col("cep_origem") == "SUFIXO_000_INCERTO"),
        ("CEP_PADRAO_INVALIDO", F.col("cep_origem") == "PADRAO_INVALIDO"),
        ("CEP_AUSENTE", F.col("cep8").isNull()),
        ("CEP_FORA_FAIXA_UF", F.col("cep_faixa_uf_ok") == False),  # noqa: E712
        ("CEP_EMBUTIDO_NO_LOGRADOURO", F.col("cep_embutido").isNotNull()),
        ("NUMERO_EXTRAIDO_DO_LOGRADOURO", F.col("numero_ext").isNotNull()),
        ("NUMERO_NO_COMPLEMENTO", F.col("numero_compl").isNotNull()),
        ("NUMERO_CONFLITO", F.col("flag_numero_conflito")),
        ("SEM_NUMERO", F.col("numero_final") == "SN"),
        ("NUMERO_AUSENTE", F.col("numero_final").isNull()),
        ("COMPLEMENTO_EXTRAIDO_DO_LOGRADOURO", F.col("compl_ext") != ""),
        ("BAIRRO_CIDADE_NO_LOGRADOURO", F.col("log_tail_n") > F.size(F.col("log_tail_f"))),
        ("TIPO_LOGRADOURO_AUSENTE", F.col("tipo_canon").isNull() & F.col("nome_log").isNotNull()),
        ("LOGRADOURO_VAZIO", F.col("nome_log").isNull()),
        ("UF_INVALIDA", F.col("flag_uf_invalida")),
        ("UF_INFERIDA_PELA_CIDADE", F.col("flag_uf_inferida")),
        ("CIDADE_ALIAS", F.col("flag_cidade_alias")),
        ("KM_NO_NUMERO", F.col("num_km")),
    ))

    colunas_saida = (
        ["idCPF", "hash_linha", "qtd_ocorrencias"] + (["data_max"] if COL_DATA else [])
        + [f"{c}_raw" for c in CAMPOS_ENDERECO]
        + ["uf_n", "cidade_n", "bairro_n", "cep8", "cep_origem", "cep_int", "cep_embutido", "cep_uf_faixa", "cep_faixa_uf_ok",
           "log_core", "tipo_canon", "nome_log", "nome_tok", "nome_chave", "nome_fon", "nome_red", "nome_red_chave", "logradouro_limpo",
           "numero_campo", "numero_ext", "numero_compl", "numero_final", "numero_int", "numero_sufixo", "flag_numero_conflito",
           "compl_canon", "compl_pares", "compl_livre", "compl_chave", "flags_norm"]
    )
    df_norm = df.select(*colunas_saida)
    df_norm = persistir(df_norm, "02_normalizado", cluster_by="idCPF")
    contar(df_norm, "E2 variantes normalizadas")
    info(f"E2 concluída em {time.time() - t0:.0f}s")

    if CALCULAR_CONTAGENS:
        info("E2 — tokens de complemento livre mais frequentes fora do dicionário (candidatos a MAPA_COMPL):")
        display(df_norm.select(F.explode(tokens(F.col("compl_livre"))).alias("token"))
                .filter(~F.col("token").rlike(r"^\d+$")).groupBy("token").count().orderBy(F.desc("count")).limit(30))
        info("E2 — distribuição da origem do CEP:")
        display(df_norm.groupBy("cep_origem").count().orderBy(F.desc("count")))
    display(df_norm.select("logradouro_raw", "numero_raw", "complemento_raw", "cep_parte1_raw", "cep_parte2_raw", "cep8", "cep_origem",
                           "tipo_canon", "nome_log", "nome_red", "nome_fon", "numero_final", "numero_sufixo", "compl_pares", "compl_livre", "flags_norm")
            .limit(20))
else:
    info("E2 pulada")

## E3 — Base dos Correios: normalização e dimensões

Aplica à `tb_correios` **as mesmas funções** de normalização usadas no cliente (é isso que garante que as chaves sejam comparáveis) e produz duas dimensões:

- `_dim_correios_cep` — 1 linha por CEP (8 dígitos): UF, município, bairro, tipo canônico, nome do logradouro (+ chaves `c_nome_chave`, `c_nome_fon`, `c_nome_red_chave`, `c_nome_red_oficial_chave`), lados existentes (`c_lados`), faixa numérica (se houver), `c_cep_tipo` = `LOGRADOURO` ou `LOCALIDADE` (CEP único de município, sem logradouro) e `c_id_log` (id do logradouro, mesma fórmula da outra dim).
- `_dim_correios_logradouro` — 1 linha por (UF, município, tipo, chave do nome) com a lista de CEPs (`ceps`: cep, lado, bairro, faixa) e chaves de blocking (`primeiro_token`, `ultimo_token`, fonética, reduzido).

Se `logradouro_correios` vier com o tipo na frente (`"R ADINEI..."`), o 1º token é removido quando coincide com `tipo`. Se existir a coluna de logradouro reduzido oficial, ela vira uma chave adicional (`c_nome_red_oficial_chave`).

Imprime os **tipos não mapeados** (para estender `MAPA_TIPOS_CORREIOS`). As dims não têm sufixo de lote (são compartilhadas entre lotes).

In [ ]:
# ============================================================================
# E3 — normalização da tb_correios e dimensões (CEP / logradouro)
# ============================================================================
def id_log_expr(uf: Column, municipio: Column, tipo: Column, nome_chave: Column) -> Column:
    """Id determinístico de um logradouro dos Correios — MESMA fórmula nas duas dims e no fuzzy."""
    return F.xxhash64(F.coalesce(uf, F.lit("")), F.coalesce(municipio, F.lit("")), F.coalesce(tipo, F.lit("")), F.coalesce(nome_chave, F.lit("")))


def remover_tipo_na_frente(log_norm: Column, tipo_abrev: Column, tipo_canon: Column) -> Column:
    tk = tokens(log_norm)
    t0 = F.get(tk, 0)
    tem_tipo = t0.isNotNull() & ((t0 == tipo_abrev) | (F.try_element_at(MAPA_TIPOS_CORREIOS_COL, t0) == tipo_canon))
    return F.when(tem_tipo, F.array_join(F.slice(tk, 2, 200), " ")).otherwise(log_norm)


if etapa_ativa("E3"):
    t0 = time.time()
    dfc = spark.table(NOME_TB_CORREIOS)
    sel = []
    for interno, real in COLS_CORREIOS.items():
        sel.append((F.col(f"`{real}`").cast("string") if real else F.lit(None).cast("string")).alias(f"c_{interno}"))
    dfc = dfc.select(*sel)

    p1 = F.regexp_replace(F.coalesce(F.col("c_cep_parte1"), F.lit("")), r"[^0-9]", "")
    p2 = F.regexp_replace(F.coalesce(F.col("c_cep_parte2"), F.lit("")), r"[^0-9]", "")
    cc = F.concat(p1, p2)
    cep8 = (F.when(F.length(p1) == 8, p1)
            .when(F.length(cc) == 8, cc)
            .when((F.length(p1).between(1, 5)) & (F.length(p2).between(1, 3)), F.concat(F.lpad(p1, 5, "0"), F.lpad(p2, 3, "0")))
            .when(F.length(cc) == 7, F.lpad(cc, 8, "0"))
            .otherwise(F.lit(None).cast("string")))
    tipo_abrev = nulo_se_vazio(norm_chave(F.col("c_tipo")))
    tipo_canon_c = F.coalesce(F.try_element_at(MAPA_TIPOS_CORREIOS_COL, tipo_abrev), tipo_abrev)
    dfc = dfc.withColumns({"cep8": cep8, "tipo_abrev": tipo_abrev, "tipo_canon_c": tipo_canon_c})
    log_c1 = remover_tipo_na_frente(norm_chave(F.col("c_logradouro")), F.col("tipo_abrev"), F.col("tipo_canon_c"))
    red_c1 = remover_tipo_na_frente(norm_chave(F.col("c_logradouro_reduzido")), F.col("tipo_abrev"), F.col("tipo_canon_c"))
    dfc = dfc.withColumns({
        "nome_log_c": nulo_se_vazio(expandir_titulos(log_c1)),
        "nome_red_oficial_c": nulo_se_vazio(expandir_titulos(red_c1)),
        "bairro_c": nulo_se_vazio(norm_local(F.col("c_bairro"))),
        "municipio_c": nulo_se_vazio(norm_local(F.col("c_municipio"))),
        "uf_c": F.try_element_at(MAPA_UF_COL, norm_chave(F.col("c_uf"))),
        "lado_c": F.when(F.upper(F.trim(F.col("c_lado"))).isin("P", "I"), F.upper(F.trim(F.col("c_lado")))).otherwise(F.lit("A")),
        "num_ini_c": F.regexp_replace(F.coalesce(F.col("c_numero_inicial"), F.lit("")), r"[^0-9]", "").try_cast("bigint"),
        "num_fim_c": F.regexp_replace(F.coalesce(F.col("c_numero_final"), F.lit("")), r"[^0-9]", "").try_cast("bigint"),
    })
    dfc = dfc.withColumn("nome_tok_c", sem_stopwords(tokens(F.col("nome_log_c"))))
    dfc = dfc.withColumns({
        "nome_chave_c": F.array_join(F.array_sort(F.array_distinct(F.col("nome_tok_c"))), " "),
        "nome_fon_c": fonetica_tokens(F.col("nome_tok_c")),
        "nome_red_chave_c": chave_tokens(forma_reduzida(F.col("nome_log_c"))),
    })
    dfc = dfc.withColumn("nome_red_oficial_chave_c", F.coalesce(nulo_se_vazio(chave_tokens(F.col("nome_red_oficial_c"))), F.col("nome_red_chave_c")))
    dfc_n = dfc.filter(F.col("cep8").isNotNull() & F.col("uf_c").isNotNull())

    if CALCULAR_CONTAGENS:
        info("E3 — abreviações de tipo da tb_correios NÃO mapeadas (candidatas a MAPA_TIPOS_CORREIOS):")
        display(dfc_n.filter(F.col("tipo_abrev").isNotNull() & F.try_element_at(MAPA_TIPOS_CORREIOS_COL, F.col("tipo_abrev")).isNull())
                .groupBy("tipo_abrev").count().orderBy(F.desc("count")).limit(60))

    # ---- dim por CEP -------------------------------------------------------------------------------------------
    dim_cep = (
        dfc_n.groupBy("cep8").agg(
            F.min("uf_c").alias("c_uf"), F.min("municipio_c").alias("c_municipio"), F.min("bairro_c").alias("c_bairro"),
            F.min("tipo_canon_c").alias("c_tipo_canon"), F.min("nome_log_c").alias("c_nome_log"),
            F.min("nome_chave_c").alias("c_nome_chave"), F.min("nome_fon_c").alias("c_nome_fon"),
            F.min("nome_red_chave_c").alias("c_nome_red_chave"), F.min("nome_red_oficial_chave_c").alias("c_nome_red_oficial_chave"),
            F.collect_set("lado_c").alias("c_lados"), F.min("num_ini_c").alias("c_num_ini"), F.max("num_fim_c").alias("c_num_fim"),
            F.count(F.lit(1)).alias("c_qtd_linhas"), F.size(F.collect_set("nome_chave_c")).alias("c_qtd_logradouros"),
        )
        .withColumn("c_cep_tipo", F.when(F.col("c_nome_log").isNull(), F.lit("LOCALIDADE")).otherwise(F.lit("LOGRADOURO")))
        .withColumn("c_id_log", F.when(F.col("c_nome_log").isNotNull(),
                                       id_log_expr(F.col("c_uf"), F.col("c_municipio"), F.col("c_tipo_canon"), F.col("c_nome_chave"))))
    )
    dim_cep = persistir(dim_cep, "dim_correios_cep", cluster_by="cep8", com_lote=False)
    contar(dim_cep, "E3 CEPs distintos na dim")

    # ---- dim por logradouro ------------------------------------------------------------------------------------
    dim_log = (
        dfc_n.filter(F.col("nome_chave_c").isNotNull() & (F.col("nome_chave_c") != ""))
        .groupBy("uf_c", "municipio_c", "tipo_canon_c", "nome_chave_c")
        .agg(
            F.min("nome_log_c").alias("nome_log"), F.min("nome_fon_c").alias("nome_fon"),
            F.min("nome_red_chave_c").alias("nome_red_chave"), F.min("nome_red_oficial_chave_c").alias("nome_red_oficial_chave"),
            F.collect_list(F.struct("cep8", "lado_c", "bairro_c", "num_ini_c", "num_fim_c")).alias("ceps"),
            F.count_distinct("cep8").alias("qtd_ceps"), F.collect_set("bairro_c").alias("bairros"),
        )
        .withColumnRenamed("uf_c", "uf").withColumnRenamed("municipio_c", "municipio")
        .withColumnRenamed("tipo_canon_c", "tipo_canon").withColumnRenamed("nome_chave_c", "nome_chave")
    )
    dim_log = dim_log.withColumn("id_log", id_log_expr(F.col("uf"), F.col("municipio"), F.col("tipo_canon"), F.col("nome_chave")))
    dim_log = dim_log.withColumn("nome_tok", sem_stopwords(tokens(F.col("nome_log"))))
    dim_log = dim_log.withColumns({
        "primeiro_token": F.get(F.col("nome_tok"), 0),
        "ultimo_token": F.get(F.col("nome_tok"), F.size(F.col("nome_tok")) - 1),
    })
    dim_log = persistir(dim_log, "dim_correios_logradouro", cluster_by=["uf", "municipio"], com_lote=False)
    contar(dim_log, "E3 logradouros distintos na dim")
    info(f"E3 concluída em {time.time() - t0:.0f}s")
    display(dim_cep.limit(8))
    display(dim_log.select("uf", "municipio", "tipo_canon", "nome_log", "nome_chave", "nome_fon", "nome_red_chave", "qtd_ceps", "ceps").limit(8))
else:
    info("E3 pulada (dims lidas das tabelas persistidas quando necessário)")

## E4 — Validação via CEP (join com a dim dos Correios) e score de qualidade da variante

1. Join `left` das variantes com `_dim_correios_cep` pelo `cep8` (broadcast). Segundo join pelo `cep_embutido` (CEP achado dentro do logradouro) para as variantes cujo `cep8` não existe nos Correios — se o embutido existe, ele passa a ser o CEP de referência (`cep_origem = LOGRADOURO`).
2. `sim_log_cep`: similaridade combinada entre o nome do logradouro da variante e o do CEP (tokens, Levenshtein, fonética, forma reduzida, reduzido oficial). `logradouro_bate = sim ≥ LIMIAR_SIM_LOGRADOURO`.
3. Coerências: `uf_ok`, `municipio_ok`, `bairro_ok`, `lado_ok` (paridade do número × lado do CEP), `faixa_ok` (faixa numérica, se existir). `cep_confiavel` = CEP existe **e** (logradouro bate, ou é CEP de localidade com município coerente, ou a variante não tem logradouro mas UF/município batem). `cep_contradiz` = CEP existe, logradouro não bate e UF diverge → CEP quase certamente errado.
4. `score_qualidade` (0–100) da variante = soma ponderada dos indicadores acima + completude − penalidades (conflito de número, CEP reconstruído, CEP contraditório). Esse score entra no **peso do voto** de cada variante em E7.

In [ ]:
# ============================================================================
# E4 — validação via CEP + score de qualidade da variante
# ============================================================================
COLS_DIM_CEP = ["c_uf", "c_municipio", "c_bairro", "c_tipo_canon", "c_nome_log", "c_nome_chave", "c_nome_fon", "c_nome_red_chave",
                "c_nome_red_oficial_chave", "c_lados", "c_num_ini", "c_num_fim", "c_cep_tipo", "c_id_log", "c_qtd_logradouros"]

if etapa_ativa("E4"):
    t0 = time.time()
    dfn = obter("02_normalizado")
    dim = obter("dim_correios_cep", com_lote=False).select("cep8", *COLS_DIM_CEP)
    dim_a = dim.withColumnRenamed("cep8", "j_cep8")
    dim_b = dim.select(F.col("cep8").alias("e_cep8"), *[F.col(c).alias("e_" + c[2:]) for c in COLS_DIM_CEP])
    if BROADCAST_CORREIOS:
        dim_a, dim_b = F.broadcast(dim_a), F.broadcast(dim_b)

    dfj = dfn.join(dim_a, dfn["cep8"] == dim_a["j_cep8"], "left").drop("j_cep8")
    dfj = dfj.join(dim_b, dfj["cep_embutido"] == dim_b["e_cep8"], "left")
    usa_embutido = F.col("c_uf").isNull() & F.col("e_uf").isNotNull()
    dfj = dfj.withColumn("usa_embutido", usa_embutido)
    dfj = dfj.withColumns({c: F.when(F.col("usa_embutido"), F.col("e_" + c[2:])).otherwise(F.col(c)) for c in COLS_DIM_CEP})
    dfj = dfj.withColumns({
        "cep8": F.when(F.col("usa_embutido"), F.col("cep_embutido")).otherwise(F.col("cep8")),
        "cep_origem": F.when(F.col("usa_embutido"), F.lit("LOGRADOURO")).otherwise(F.col("cep_origem")),
    }).drop("e_cep8", *["e_" + c[2:] for c in COLS_DIM_CEP])
    dfj = dfj.withColumn("cep_int", F.col("cep8").try_cast("bigint"))
    dfj = dfj.withColumn("cep_uf_faixa", faixa_uf_expr(F.col("cep_int")))
    dfj = dfj.withColumn("cep_faixa_uf_ok", F.when(F.col("cep8").isNull() | F.col("uf_n").isNull(), F.lit(None).cast("boolean"))
                         .otherwise(F.col("cep_uf_faixa") == F.col("uf_n")))

    cep_existe = F.col("c_uf").isNotNull()
    c_nome_tok = sem_stopwords(tokens(F.col("c_nome_log")))
    sim_base = F.when(cep_existe & (F.col("c_cep_tipo") == "LOGRADOURO"),
                      sim_logradouro(F.col("nome_log"), F.col("nome_tok"), F.col("nome_fon"), F.col("nome_red_chave"),
                                     F.col("c_nome_log"), c_nome_tok, F.col("c_nome_fon"), F.col("c_nome_red_chave"))).otherwise(F.lit(0.0))
    sim_red_of = F.when(F.col("c_nome_red_oficial_chave").isNotNull() & (F.length(F.col("c_nome_red_oficial_chave")) > 2)
                        & ((F.col("nome_red_chave") == F.col("c_nome_red_oficial_chave")) | (F.col("nome_chave") == F.col("c_nome_red_oficial_chave"))),
                        F.lit(0.95)).otherwise(F.lit(0.0))
    dfj = dfj.withColumn("sim_log_cep", F.greatest(sim_base, sim_red_of))
    dfj = dfj.withColumns({
        "cep_existe": cep_existe,
        "logradouro_bate": cep_existe & (F.col("sim_log_cep") >= F.lit(LIMIAR_SIM_LOGRADOURO)),
        "uf_ok": F.when(F.col("uf_n").isNull() | ~cep_existe, F.lit(None).cast("boolean")).otherwise(F.col("uf_n") == F.col("c_uf")),
        "municipio_ok": F.when(F.col("cidade_n").isNull() | ~cep_existe | F.col("c_municipio").isNull(), F.lit(None).cast("boolean"))
                         .otherwise((F.col("cidade_n") == F.col("c_municipio")) | (sim_lev(F.col("cidade_n"), F.col("c_municipio")) >= 0.85)),
        "bairro_ok": F.when(F.col("bairro_n").isNull() | F.col("c_bairro").isNull(), F.lit(None).cast("boolean"))
                      .otherwise((F.col("bairro_n") == F.col("c_bairro")) | (sim_jaccard(tokens(F.col("bairro_n")), tokens(F.col("c_bairro"))) >= 0.5)
                                 | (sim_lev(F.col("bairro_n"), F.col("c_bairro")) >= 0.85)),
        "lado_ok": (F.when(F.col("numero_int").isNull() | F.col("c_lados").isNull() | (F.size(F.col("c_lados")) == 0) | F.array_contains(F.col("c_lados"), "A"), F.lit(None).cast("boolean"))
                    .when(F.array_contains(F.col("c_lados"), "P") & F.array_contains(F.col("c_lados"), "I"), F.lit(True))
                    .when(F.array_contains(F.col("c_lados"), "P"), F.col("numero_int") % 2 == 0)
                    .when(F.array_contains(F.col("c_lados"), "I"), F.col("numero_int") % 2 == 1)),
        "faixa_ok": F.when(F.col("numero_int").isNull() | F.col("c_num_ini").isNull() | F.col("c_num_fim").isNull(), F.lit(None).cast("boolean"))
                     .otherwise(F.col("numero_int").between(F.col("c_num_ini"), F.col("c_num_fim"))),
    })
    loc_ok = F.coalesce(F.col("municipio_ok"), F.col("uf_ok"), F.lit(False))
    dfj = dfj.withColumns({
        "cep_confiavel": F.col("cep_existe") & (
            F.col("logradouro_bate")
            | ((F.col("c_cep_tipo") == "LOCALIDADE") & loc_ok)
            | (F.col("nome_log").isNull() & loc_ok)
        ),
        "cep_contradiz": F.col("cep_existe") & ~F.col("logradouro_bate") & (F.col("uf_ok") == False),  # noqa: E712
    })
    b = lambda c: F.when(F.coalesce(c, F.lit(False)), F.lit(1)).otherwise(F.lit(0))  # bool -> 0/1
    score = (
        25 * b(F.col("cep_existe")) + 20 * b(F.col("logradouro_bate"))
        + 10 * b((F.col("sim_log_cep") >= 0.6) & ~F.col("logradouro_bate"))
        + 10 * b(F.col("uf_ok")) + 10 * b(F.col("municipio_ok")) + 5 * b(F.col("bairro_ok"))
        + 10 * b(F.col("numero_int").isNotNull()) + 5 * b(F.size(F.col("compl_pares")) > 0)
        + 5 * b(F.col("tipo_canon").isNotNull()) + 5 * b(F.col("nome_log").isNotNull()) + 5 * b(F.col("lado_ok"))
        - 10 * b(F.col("flag_numero_conflito"))
        - 5 * b(F.col("cep_origem").isin("ZEROS_CORRIGIDOS", "SUFIXO_000_INCERTO", "CONCAT", "P1_COMPLETO_P2_CONFLITANTE"))
        - 15 * b(F.col("cep_contradiz")) - 5 * b(F.col("cep_faixa_uf_ok") == False)  # noqa: E712
        - 5 * b(F.col("lado_ok") == False) - 5 * b(F.col("faixa_ok") == False)  # noqa: E712
    )
    dfj = dfj.withColumn("score_qualidade", F.greatest(F.lit(0), F.least(F.lit(100), score)).cast("int"))
    dfj = dfj.withColumn("flags_val", flags_array(
        ("CEP_EXISTE_CORREIOS", F.col("cep_existe")),
        ("CEP_INEXISTENTE_CORREIOS", F.col("cep8").isNotNull() & ~F.col("cep_existe")),
        ("CEP_VIA_LOGRADOURO_EMBUTIDO", F.col("usa_embutido")),
        ("LOGRADOURO_BATE_CEP", F.col("logradouro_bate")),
        ("LOGRADOURO_NAO_BATE_CEP", F.col("cep_existe") & (F.col("c_cep_tipo") == "LOGRADOURO") & ~F.col("logradouro_bate") & F.col("nome_log").isNotNull()),
        ("CEP_LOCALIDADE", F.col("c_cep_tipo") == "LOCALIDADE"),
        ("UF_DIVERGE_CEP", F.col("uf_ok") == False),  # noqa: E712
        ("MUNICIPIO_DIVERGE_CEP", F.col("municipio_ok") == False),  # noqa: E712
        ("BAIRRO_DIVERGE_CEP", F.col("bairro_ok") == False),  # noqa: E712
        ("LADO_DIVERGE_CEP", F.col("lado_ok") == False),  # noqa: E712
        ("FAIXA_DIVERGE_CEP", F.col("faixa_ok") == False),  # noqa: E712
        ("CEP_CONTRADIZ_ENDERECO", F.col("cep_contradiz")),
    ))
    df_val = dfj.drop("usa_embutido")
    df_val = persistir(df_val, "03_validado", cluster_by="idCPF")
    contar(df_val, "E4 variantes validadas")
    if CALCULAR_CONTAGENS:
        info("E4 — resumo da validação:")
        display(df_val.agg(
            F.avg(b(F.col("cep_existe"))).alias("pct_cep_existe"), F.avg(b(F.col("logradouro_bate"))).alias("pct_logradouro_bate"),
            F.avg(b(F.col("cep_confiavel"))).alias("pct_cep_confiavel"), F.avg(b(F.col("cep_contradiz"))).alias("pct_cep_contradiz"),
            F.avg("score_qualidade").alias("score_medio")))
    info(f"E4 concluída em {time.time() - t0:.0f}s")
    display(df_val.select("logradouro_raw", "cep8", "cep_origem", "c_nome_log", "sim_log_cep", "logradouro_bate", "uf_ok", "municipio_ok",
                          "lado_ok", "cep_confiavel", "score_qualidade", "flags_val").limit(15))
else:
    info("E4 pulada")

## E5 — Fallback fuzzy contra os Correios (logradouro → CEP)

Para variantes cujo CEP **não existe**, **contradiz** o endereço ou cujo logradouro **não bate** com o do CEP, procura o logradouro nos Correios pelo nome:

1. **Consultas distintas** = (UF, cidade, tipo, chave do nome) — bilhões de variantes viram alguns milhões de consultas.
2. **Blocking** dentro de UF + município por 4 chaves: 1º token do nome, chave fonética, último token, forma reduzida (própria ou oficial). Consultas sem candidato no município (cidade nula, distrito, grafia diferente) ganham uma **2ª passada** só por UF (fonética + 1º token, ou forma reduzida), com penalidade no score.
3. Similaridade **nativa** (`sim_logradouro`) para podar (≥ 0.35, no máximo `MAX_CANDIDATOS_FUZZY` por consulta) e, se `USAR_RAPIDFUZZ`, refinamento com `rapidfuzz` (`token_set_ratio`, `WRatio`, `partial_token_sort_ratio`) numa pandas UDF **só sobre os pares candidatos**.
4. Score final 0–100 (+3 se o tipo coincide, −6 se conflita, −4 no bloco só-UF). Aceita o melhor candidato se `score ≥ LIMIAR_FUZZY_CORREIOS` **e** a margem sobre o 2º é `≥ MARGEM_FUZZY` (senão `FUZZY_AMBIGUO`).
5. De volta às variantes: escolhe o CEP entre os CEPs do logradouro casado usando a **paridade do número × lado** e a **faixa numérica** (quando existir). Se sobrar mais de um CEP, mantém o da própria variante se estiver entre eles; senão fica sem CEP (mas com logradouro/bairro/município sugeridos).

Saídas: `_04_fuzzy_correios` (por consulta) e `_04b_variantes_enriquecidas` (variantes + colunas `fz_*`). Com `USAR_FUZZY_CORREIOS=False`, a etapa só copia as variantes com as colunas `fz_*` nulas, para manter o contrato com E6/E7.

In [ ]:
# ============================================================================
# E5 — fuzzy contra os Correios (consultas distintas -> melhor logradouro -> CEP por paridade/faixa)
# ============================================================================
import pandas as pd

COLS_FZ = ["fz_id_log", "fz_tipo_canon", "fz_nome_log", "fz_nome_chave", "fz_nome_fon", "fz_nome_red_chave", "fz_municipio", "fz_uf",
           "fz_ceps", "fz_qtd_ceps", "fz_bairros", "fz_score", "fz_score_2", "fz_status", "fz_bloco_uf_apenas"]
ESQUEMA_CEPS = "array<struct<cep8:string,lado_c:string,bairro_c:string,num_ini_c:bigint,num_fim_c:bigint>>"


def id_consulta_expr(uf, cidade, tipo, nome_chave):
    return F.xxhash64(F.coalesce(uf, F.lit("")), F.coalesce(cidade, F.lit("")), F.coalesce(tipo, F.lit("")), F.coalesce(nome_chave, F.lit("")))


def rapidfuzz_disponivel() -> bool:
    try:
        import rapidfuzz  # noqa: F401
        return True
    except ImportError:
        return False


if etapa_ativa("E5"):
    t0 = time.time()
    dfv = obter("03_validado")
    dfv = dfv.withColumn("id_consulta", id_consulta_expr(F.col("uf_n"), F.col("cidade_n"), F.col("tipo_canon"), F.col("nome_chave")))

    if USAR_FUZZY_CORREIOS:
        diml = obter("dim_correios_logradouro", com_lote=False)
        precisa = (
            (~F.col("cep_existe"))
            | (F.col("cep_existe") & (F.col("c_cep_tipo") == "LOGRADOURO") & ~F.col("logradouro_bate") & (F.col("sim_log_cep") < 0.6))
            | F.col("cep_contradiz")
        )
        cons = (
            dfv.filter(precisa & F.col("nome_log").isNotNull() & (F.col("nome_chave") != "") & (F.col("uf_n").isNotNull() | F.col("cidade_n").isNotNull()))
            .groupBy("id_consulta", "uf_n", "cidade_n", "tipo_canon", "nome_chave")
            .agg(F.min("nome_log").alias("nome_log"), F.min("nome_fon").alias("nome_fon"), F.min("nome_red_chave").alias("nome_red_chave"))
        )
        cons = cons.withColumn("nome_tok", sem_stopwords(tokens(F.col("nome_log"))))
        cons = cons.withColumns({"q_primeiro": F.get(F.col("nome_tok"), 0), "q_ultimo": F.get(F.col("nome_tok"), F.size(F.col("nome_tok")) - 1)})
        contar(cons, "E5 consultas fuzzy distintas")

        dl = diml.select(
            F.col("uf").alias("b_uf"), F.col("municipio").alias("b_municipio"), F.col("tipo_canon").alias("b_tipo"),
            F.col("nome_log").alias("b_nome_log"), F.col("nome_tok").alias("b_nome_tok"), F.col("nome_chave").alias("b_nome_chave"),
            F.col("nome_fon").alias("b_nome_fon"), F.col("nome_red_chave").alias("b_nome_red_chave"),
            F.col("nome_red_oficial_chave").alias("b_nome_red_oficial_chave"), F.col("id_log").alias("b_id_log"),
            F.col("ceps").alias("b_ceps"), F.col("qtd_ceps").alias("b_qtd_ceps"), F.col("bairros").alias("b_bairros"),
            F.col("primeiro_token").alias("b_primeiro_token"), F.col("ultimo_token").alias("b_ultimo_token"),
        )

        # --- blocking 1: município (+ UF quando informada; UF nula = qualquer UF) --------------------------------
        cc = cons.filter(F.col("cidade_n").isNotNull())
        base = [cc["cidade_n"] == dl["b_municipio"], F.coalesce(cc["uf_n"] == dl["b_uf"], F.lit(True))]
        blocos = [
            cc.join(dl, base + [cc["q_primeiro"] == dl["b_primeiro_token"]], "inner"),
            cc.join(dl, base + [cc["nome_fon"] == dl["b_nome_fon"]], "inner"),
            cc.join(dl, base + [cc["q_ultimo"] == dl["b_ultimo_token"]], "inner"),
            cc.join(dl, base + [(cc["nome_red_chave"] == dl["b_nome_red_chave"]) | (cc["nome_red_chave"] == dl["b_nome_red_oficial_chave"])
                                | (cc["nome_chave"] == dl["b_nome_red_oficial_chave"])], "inner"),
        ]
        cand1 = blocos[0]
        for b_ in blocos[1:]:
            cand1 = cand1.unionByName(b_)
        cand1 = cand1.dropDuplicates(["id_consulta", "b_id_log"]).withColumn("bloco_uf_apenas", F.lit(False))

        # --- blocking 2: só UF, para consultas sem candidato no município -----------------------------------------
        sem = cons.join(cand1.select("id_consulta").distinct(), "id_consulta", "left_anti").filter(F.col("uf_n").isNotNull())
        c2a = sem.join(dl, [sem["uf_n"] == dl["b_uf"], sem["nome_fon"] == dl["b_nome_fon"], sem["q_primeiro"] == dl["b_primeiro_token"]], "inner")
        c2b = sem.join(dl, [sem["uf_n"] == dl["b_uf"], (sem["nome_red_chave"] == dl["b_nome_red_chave"]) | (sem["nome_chave"] == dl["b_nome_red_oficial_chave"])], "inner")
        cand2 = c2a.unionByName(c2b).dropDuplicates(["id_consulta", "b_id_log"]).withColumn("bloco_uf_apenas", F.lit(True))
        cand = cand1.unionByName(cand2)

        # --- similaridade nativa + poda --------------------------------------------------------------------------
        sim_nat = sim_logradouro(F.col("nome_log"), F.col("nome_tok"), F.col("nome_fon"), F.col("nome_red_chave"),
                                 F.col("b_nome_log"), F.col("b_nome_tok"), F.col("b_nome_fon"), F.col("b_nome_red_chave"))
        sim_of = F.when((F.col("nome_red_chave") == F.col("b_nome_red_oficial_chave")) | (F.col("nome_chave") == F.col("b_nome_red_oficial_chave")), F.lit(0.95)).otherwise(F.lit(0.0))
        cand = cand.withColumn("sim_nat", F.greatest(sim_nat, sim_of))
        cand = cand.withColumn("tipo_bonus", F.when(F.col("tipo_canon").isNull() | F.col("b_tipo").isNull(), F.lit(0.0))
                               .when(F.col("tipo_canon") == F.col("b_tipo"), F.lit(3.0)).otherwise(F.lit(-6.0)))
        w_nat = Window.partitionBy("id_consulta").orderBy(F.desc("sim_nat"))
        cand = cand.filter(F.col("sim_nat") >= 0.35).withColumn("rk_nat", F.row_number().over(w_nat)).filter(F.col("rk_nat") <= MAX_CANDIDATOS_FUZZY)

        # --- rapidfuzz (pandas UDF só nos pares candidatos) -------------------------------------------------------
        if USAR_RAPIDFUZZ and rapidfuzz_disponivel():
            @F.pandas_udf("double")
            def rf_score(a: pd.Series, b: pd.Series) -> pd.Series:
                from rapidfuzz import fuzz
                out = []
                for x, y in zip(a, b):
                    if not x or not y:
                        out.append(0.0)
                        continue
                    out.append(float(max(fuzz.token_set_ratio(x, y), fuzz.WRatio(x, y) * 0.97, fuzz.partial_token_sort_ratio(x, y) * 0.95)))
                return pd.Series(out, dtype="float64")
            cand = cand.withColumn("score_rf", rf_score(F.col("nome_log"), F.col("b_nome_log")))
        else:
            info("E5 — rapidfuzz desativado/indisponível: usando só similaridade nativa")
            cand = cand.withColumn("score_rf", F.lit(None).cast("double"))

        score = F.greatest(F.col("sim_nat") * 100, F.coalesce(F.col("score_rf"), F.lit(0.0))) + F.col("tipo_bonus") - F.when(F.col("bloco_uf_apenas"), F.lit(4.0)).otherwise(F.lit(0.0))
        cand = cand.withColumn("score_fuzzy", F.round(score, 2))
        w2 = Window.partitionBy("id_consulta").orderBy(F.desc("score_fuzzy"), F.asc("b_qtd_ceps"), F.asc("b_id_log"))
        cand = cand.withColumn("rk", F.row_number().over(w2)).withColumn("score_2", F.lead("score_fuzzy", 1).over(w2))
        melhor = cand.filter(F.col("rk") == 1)
        melhor = melhor.withColumn(
            "fz_status",
            F.when((F.col("score_fuzzy") >= LIMIAR_FUZZY_CORREIOS)
                   & (F.col("score_2").isNull() | (F.col("score_fuzzy") - F.col("score_2") >= MARGEM_FUZZY) | (F.col("score_2") < LIMIAR_FUZZY_CORREIOS)), F.lit("FUZZY_OK"))
            .when(F.col("score_fuzzy") >= LIMIAR_FUZZY_CORREIOS, F.lit("FUZZY_AMBIGUO"))
            .otherwise(F.lit("FUZZY_BAIXO")),
        )
        fuzzy = melhor.select(
            "id_consulta", F.col("b_id_log").alias("fz_id_log"), F.col("b_tipo").alias("fz_tipo_canon"), F.col("b_nome_log").alias("fz_nome_log"),
            F.col("b_nome_chave").alias("fz_nome_chave"), F.col("b_nome_fon").alias("fz_nome_fon"), F.col("b_nome_red_chave").alias("fz_nome_red_chave"),
            F.col("b_municipio").alias("fz_municipio"), F.col("b_uf").alias("fz_uf"), F.col("b_ceps").alias("fz_ceps"), F.col("b_qtd_ceps").alias("fz_qtd_ceps"),
            F.col("b_bairros").alias("fz_bairros"), F.col("score_fuzzy").alias("fz_score"), F.col("score_2").alias("fz_score_2"), "fz_status",
            F.col("bloco_uf_apenas").alias("fz_bloco_uf_apenas"),
        )
        fuzzy = persistir(fuzzy, "04_fuzzy_correios")
        if CALCULAR_CONTAGENS:
            display(fuzzy.groupBy("fz_status").count())
        dfe = dfv.join(fuzzy, "id_consulta", "left")
    else:
        info("E5 — USAR_FUZZY_CORREIOS=False: colunas fz_* ficam nulas")
        dfe = dfv
        tipos = {"fz_id_log": "bigint", "fz_tipo_canon": "string", "fz_nome_log": "string", "fz_nome_chave": "string", "fz_nome_fon": "string",
                 "fz_nome_red_chave": "string", "fz_municipio": "string", "fz_uf": "string", "fz_ceps": ESQUEMA_CEPS, "fz_qtd_ceps": "bigint",
                 "fz_bairros": "array<string>", "fz_score": "double", "fz_score_2": "double", "fz_status": "string", "fz_bloco_uf_apenas": "boolean"}
        dfe = dfe.withColumns({c: F.lit(None).cast(t) for c, t in tipos.items()})

    # --- escolha do CEP dentro do logradouro casado (paridade x lado, faixa numérica) ------------------------------
    n = F.col("numero_int")
    ceps_ok = F.filter(
        F.coalesce(F.col("fz_ceps"), F.array().cast(ESQUEMA_CEPS)),
        lambda c: (
            (n.isNull() | (c.getField("lado_c") == "A")
             | ((c.getField("lado_c") == "P") & (n % 2 == 0)) | ((c.getField("lado_c") == "I") & (n % 2 == 1)))
            & (n.isNull() | c.getField("num_ini_c").isNull() | c.getField("num_fim_c").isNull() | n.between(c.getField("num_ini_c"), c.getField("num_fim_c")))
        ),
    )
    dfe = dfe.withColumn("fz_ceps_ok", ceps_ok)
    ceps_dist = F.array_distinct(F.transform(F.col("fz_ceps_ok"), lambda c: c.getField("cep8")))
    bairros_dist = F.array_distinct(F.filter(F.transform(F.col("fz_ceps_ok"), lambda c: c.getField("bairro_c")), lambda x: x.isNotNull()))
    fz_ok = F.col("fz_status") == "FUZZY_OK"
    dfe = dfe.withColumns({
        "fz_cep": F.when(fz_ok & (F.size(ceps_dist) == 1), F.get(ceps_dist, 0))
                   .when(fz_ok & F.array_contains(ceps_dist, F.coalesce(F.col("cep8"), F.lit("-"))), F.col("cep8"))
                   .otherwise(F.lit(None).cast("string")),
        "fz_bairro": F.when(fz_ok & (F.size(bairros_dist) == 1), F.get(bairros_dist, 0))
                      .when(fz_ok & (F.size(F.coalesce(F.col("fz_bairros"), F.array().cast("array<string>"))) == 1), F.get(F.col("fz_bairros"), 0))
                      .otherwise(F.lit(None).cast("string")),
        "fz_cep_ambiguo": fz_ok & (F.size(ceps_dist) > 1),
    })
    dfe = dfe.withColumn("flags_fz", flags_array(
        ("FUZZY_CORREIOS_OK", fz_ok), ("FUZZY_CORREIOS_AMBIGUO", F.col("fz_status") == "FUZZY_AMBIGUO"),
        ("FUZZY_CORREIOS_BAIXO", F.col("fz_status") == "FUZZY_BAIXO"), ("FUZZY_CEP_SUGERIDO", F.col("fz_cep").isNotNull()),
        ("FUZZY_CEP_AMBIGUO", F.col("fz_cep_ambiguo")), ("FUZZY_SO_UF", F.coalesce(F.col("fz_bloco_uf_apenas"), F.lit(False))),
    )).drop("fz_ceps_ok")
    df_enr = persistir(dfe, "04b_variantes_enriquecidas", cluster_by="idCPF")
    contar(df_enr, "E5 variantes enriquecidas")
    info(f"E5 concluída em {time.time() - t0:.0f}s")
    display(df_enr.filter(F.col("fz_status").isNotNull())
            .select("logradouro_raw", "cidade_n", "cep8", "cep_existe", "logradouro_bate", "fz_nome_log", "fz_score", "fz_score_2", "fz_status", "fz_cep", "fz_bairro")
            .limit(15))
else:
    info("E5 pulada")

## E6 — Clustering intra-CPF (união por múltiplas chaves)

Objetivo: agrupar, **dentro de cada CPF**, as variantes que descrevem o mesmo endereço, sem comparar todas contra todas.

1. Cada variante recebe até 7 **chaves de ligação**, todas contendo o número (ou `SN`):
   `CEP|cep|num`, `LOG|chave do nome|num`, `FON|fonética|num`, `RED|forma reduzida|num`, `COR|id do logradouro nos Correios (via CEP ou fuzzy)|num`, `P5|5 primeiros dígitos do CEP|num|1º token do nome`, `ANA|UF|anagrama (letras ordenadas) do nome|num` (casa erros de transposição como `FOSNECA`/`FONSECA` sem depender do CEP).
2. **Propagação de rótulo mínimo**: rótulo inicial = posição da variante no CPF; a cada iteração o rótulo vira o menor rótulo entre todas as variantes que compartilham qualquer chave (janelas `min(rotulo) over (idCPF, chave)`); repete até nenhum rótulo mudar (`MAX_ITER_CLUSTER` no máximo). É equivalente a componentes conexas do grafo "compartilha chave", com custo de 1 shuffle por iteração (os dados são reparticionados por `idCPF` antes das janelas).
3. **Anexação das variantes sem número** (número nulo **ou** `SN`/`0`): a variante entra num cluster numerado existente do mesmo CPF só se houver **exatamente um** cluster com a mesma chave de nome / fonética / id Correios / CEP; senão fica no próprio cluster. (Uma linha `S/N` de quem mora no nº 1127 da mesma rua é quase sempre o mesmo endereço mal preenchido.)
4. `cluster_id = idCPF || '_' || rotulo`.

Por que o número está em todas as chaves: é o que impede fundir "RUA X, 100" com "RUA X, 200" (dois endereços reais na mesma rua). O custo é que erros de digitação no número separam clusters — a eleição de E7 ainda escolhe o número majoritário dentro de cada cluster.

In [ ]:
# ============================================================================
# E6 — clustering intra-CPF por união de chaves (propagação de rótulo mínimo)
# ============================================================================
CHAVES_CLUSTER = ["k_cep", "k_log", "k_fon", "k_red", "k_cor", "k_p5", "k_ana"]


def anagrama(c: Column) -> Column:
    """Letras do texto ordenadas (sem espaços): iguala erros de transposição (FONSECA ~ FOSNECA)."""
    return F.array_join(F.array_sort(F.split(F.regexp_replace(F.coalesce(c, F.lit("")), " ", ""), "")), "")

if etapa_ativa("E6"):
    t0 = time.time()
    dfe = obter("04b_variantes_enriquecidas")
    fz_ok = F.col("fz_status") == "FUZZY_OK"
    id_log_ef = F.coalesce(F.when(F.col("logradouro_bate"), F.col("c_id_log")), F.when(fz_ok, F.col("fz_id_log")))
    num_k = F.coalesce(F.col("numero_int").cast("string"), F.when(F.col("numero_final") == "SN", F.lit("SN")))
    tem_num = num_k.isNotNull()
    df = dfe.withColumns({
        "id_log_ef": id_log_ef,
        "num_k": num_k,
        "k_cep": F.when(F.col("cep8").isNotNull() & tem_num, F.concat_ws("|", F.lit("CEP"), F.col("cep8"), num_k)),
        "k_log": F.when((F.col("nome_chave") != "") & tem_num, F.concat_ws("|", F.lit("LOG"), F.col("nome_chave"), num_k)),
        "k_fon": F.when((F.length(F.col("nome_fon")) >= 3) & tem_num, F.concat_ws("|", F.lit("FON"), F.col("nome_fon"), num_k)),
        "k_red": F.when((F.length(F.col("nome_red_chave")) >= 3) & tem_num, F.concat_ws("|", F.lit("RED"), F.col("nome_red_chave"), num_k)),
        "k_cor": F.when(id_log_ef.isNotNull() & tem_num, F.concat_ws("|", F.lit("COR"), id_log_ef.cast("string"), num_k)),
        "k_p5": F.when(F.col("cep8").isNotNull() & tem_num & (F.size(F.col("nome_tok")) > 0),
                       F.concat_ws("|", F.lit("P5"), F.substring(F.col("cep8"), 1, 5), num_k, F.get(F.col("nome_tok"), 0))),
        # anagrama do nome + UF + número: casa erros de transposição de letras sem depender do CEP
        "k_ana": F.when((F.length(F.col("nome_chave")) >= 6) & tem_num,
                        F.concat_ws("|", F.lit("ANA"), F.coalesce(F.col("uf_n"), F.lit("")), anagrama(F.col("nome_chave")), num_k)),
    })
    df = persistir(df, "05_base", cluster_by="idCPF")  # corta linhagem antes do loop
    if CHAVES_CLUSTER_DESATIVADAS:  # ablação: anula chaves escolhidas (só para testar E7b / medir contribuição de cada chave)
        _off = [k.strip() for k in CHAVES_CLUSTER_DESATIVADAS.split(",") if k.strip() in CHAVES_CLUSTER]
        df = df.withColumns({k: F.lit(None).cast("string") for k in _off})
        info(f"E6 — chaves desativadas para teste: {_off}")
    w_cpf = Window.partitionBy("idCPF").orderBy("hash_linha")
    # o loop roda só sobre as colunas estreitas (idCPF, hash_linha, chaves, rótulo): muito menos IO por iteração
    estreito = (df.select("idCPF", "hash_linha", *CHAVES_CLUSTER)
                .withColumn("id_var", F.row_number().over(w_cpf)).withColumn("rotulo", F.col("id_var")))

    # --- propagação iterativa ------------------------------------------------------------------------------------
    for it in range(1, MAX_ITER_CLUSTER + 1):
        estreito = estreito.repartition("idCPF")  # as janelas por (idCPF, chave) reaproveitam esta partição -> sem shuffle extra
        novo = F.col("rotulo")
        for k in CHAVES_CLUSTER:
            wk = Window.partitionBy("idCPF", k)
            novo = F.least(novo, F.when(F.col(k).isNotNull(), F.min("rotulo").over(wk)).otherwise(F.col("rotulo")))
        it_df = persistir(estreito.withColumn("rotulo_novo", novo), "05_iter", cluster_by="idCPF")
        mudou = not it_df.filter(F.col("rotulo_novo") != F.col("rotulo")).isEmpty()
        estreito = it_df.withColumn("rotulo", F.col("rotulo_novo")).drop("rotulo_novo")
        info(f"E6 iteração {it}: {'houve mudanças' if mudou else 'convergiu'}")
        if not mudou:
            break
    df = df.join(estreito.select("idCPF", "hash_linha", "id_var", "rotulo"), ["idCPF", "hash_linha"], "inner")

    # --- anexação de variantes sem número (nulo ou SN) a um cluster numerado único do mesmo logradouro -------------
    com_num = df.filter(F.col("num_k").isNotNull() & (F.col("num_k") != "SN"))
    sem_num = df.filter(F.col("num_k").isNull() | (F.col("num_k") == "SN"))
    for chave, cond in [("nome_chave", F.col("nome_chave") != ""), ("id_log_ef", F.col("id_log_ef").isNotNull()),
                        ("nome_fon", F.length(F.col("nome_fon")) >= 3), ("cep8", F.col("cep8").isNotNull())]:
        mapa = (com_num.filter(cond).groupBy("idCPF", chave).agg(F.collect_set("rotulo").alias("rots"))
                .filter(F.size("rots") == 1).select("idCPF", chave, F.get(F.col("rots"), 0).alias(f"rot_{chave}")))
        sem_num = sem_num.join(mapa, ["idCPF", chave], "left")
    sem_num = sem_num.withColumn("rotulo", F.coalesce(F.col("rot_nome_chave"), F.col("rot_id_log_ef"), F.col("rot_nome_fon"), F.col("rot_cep8"), F.col("rotulo")))
    sem_num = sem_num.withColumn("flag_anexada_sem_numero", F.coalesce(F.col("rot_nome_chave"), F.col("rot_id_log_ef"), F.col("rot_nome_fon"), F.col("rot_cep8")).isNotNull())
    # variantes sem número que NÃO foram anexadas: agrupa-as entre si pelo logradouro / fonética / CEP (sem número)
    for _ in range(2):
        for chave, cond in [("nome_chave", F.col("nome_chave") != ""), ("nome_fon", F.length(F.col("nome_fon")) >= 3), ("cep8", F.col("cep8").isNotNull())]:
            w_sn = Window.partitionBy("idCPF", chave)
            sem_num = sem_num.withColumn("rotulo", F.when(~F.col("flag_anexada_sem_numero") & cond, F.min("rotulo").over(w_sn)).otherwise(F.col("rotulo")))
    com_num = com_num.withColumn("flag_anexada_sem_numero", F.lit(False))
    df = com_num.unionByName(sem_num.select(*com_num.columns))
    # grupo_local = "mesma rua + mesmo número" (nível 1). O cluster final (nível 2, por unidade/complemento) é feito em E6b.
    df = df.withColumn("grupo_local", F.concat_ws("_", F.col("idCPF"), F.col("rotulo").cast("string")))
    df_gl = persistir(df.drop(*CHAVES_CLUSTER, "id_var"), "05_grupos_locais", cluster_by="idCPF")
    contar(df_gl, "E6 variantes com grupo_local (nível 1)")
    if CALCULAR_CONTAGENS:
        stats = df_gl.groupBy("grupo_local").agg(F.count(F.lit(1)).alias("n_var"), F.first("idCPF").alias("idCPF"))
        info("E6 — distribuição de variantes por grupo_local / grupos por CPF:")
        display(stats.groupBy(F.when(F.col("n_var") >= 10, F.lit("10+")).otherwise(F.col("n_var").cast("string")).alias("variantes_no_grupo")).count().orderBy("variantes_no_grupo"))
        display(stats.groupBy("idCPF").count().groupBy(F.when(F.col("count") >= 5, F.lit("5+")).otherwise(F.col("count").cast("string")).alias("grupos_por_cpf")).count().orderBy("grupos_por_cpf"))
    info(f"E6 concluída em {time.time() - t0:.0f}s")
    display(df_gl.orderBy("idCPF", "rotulo").select("idCPF", "grupo_local", "logradouro_raw", "numero_raw", "complemento_raw", "cep8", "nome_chave", "numero_final", "flag_anexada_sem_numero").limit(25))
else:
    info("E6 pulada")

## E6b — Unidades dentro do mesmo local (apartamentos, blocos, salas, lotes...)

E6 agrupa por **rua + número** e produz o `grupo_local` ("o prédio / o lote"). Mas um cliente pode ter **dois ou mais
endereços no mesmo prédio** (APTO 45 e APTO 46; BLOCO A e BLOCO B; SALA 101 e SALA 205; KM 12 e KM 15; casa da
FRENTE e dos FUNDOS). Fundir isso numa sugestão única seria um erro grave. E6b resolve dividindo cada `grupo_local`
por **compatibilidade de complemento**:

1. Cada variante recebe uma **assinatura**: os pares distintivos de `compl_pares` (`APTO 45`, `BLOCO B`, `KM 12`...;
   palavras de posição viram `POSICAO FUNDOS`/`POSICAO FRENTE`). Zeros à esquerda são removidos (`APTO 045` = `APTO 45`).
2. Duas assinaturas são **compatíveis** se, para todo tipo presente nas duas, o valor é igual. `{APTO 45}` × `{BLOCO B}`
   são compatíveis (nada em comum contradiz); `{APTO 45}` × `{APTO 46}` **conflitam**.
3. Por grupo: assinaturas **maximais** (não contidas em outra) definem as unidades. Se nenhuma maximal conflita com
   outra, tudo é **uma** unidade (união dos pares: `APTO 45 | BLOCO B`). Se há conflito, as maximais em conflito são
   **âncoras** (unidades distintas por evidência); assinaturas parciais (`{BLOCO B}`, subconjuntos) só entram numa
   unidade se forem compatíveis com **exatamente uma**; senão são **ambíguas** e formam cluster próprio.
4. Variantes **sem complemento**: se o grupo tem uma unidade só, entram nela (é o caso comum: o complemento faltou
   em algumas linhas). Se tem duas ou mais, ficam num cluster próprio **sem complemento sugerido**, com a flag
   `COMPLEMENTO_AMBIGUO` — o processo não inventa em qual apartamento a linha estava.
5. `cluster_id = grupo_local # hash(unidade)`; o campo `unidade` guarda os pares que definem o cluster.

A resolução roda em Python (pandas UDF) **só para os grupos com 2+ assinaturas distintas** — uma pequena fração dos
grupos, cada um com um punhado de strings. Os demais são resolvidos nativamente. A célula contém auto-testes da
função de resolução com os cenários acima; se um falhar, a célula para.

In [ ]:
# ============================================================================
# E6b — divisão do grupo_local (rua+número) em UNIDADES por compatibilidade de complemento
# ============================================================================
import pandas as pd

TIPOS_DISTINTIVOS = list(COMPL_COM_VALOR) + ["POSICAO"]   # tipos de par que distinguem unidades
_TIPOS_DISTINTIVOS_SET = set(TIPOS_DISTINTIVOS)


def assinatura_expr(pares: Column) -> Column:
    """compl_pares -> assinatura ordenada de pares distintivos. FUNDOS/FRENTE/... viram 'POSICAO X'; APTO 045 -> APTO 45."""
    conv = F.transform(
        pares,
        lambda p: F.when(p.isin(COMPL_SEM_VALOR), F.concat(F.lit("POSICAO "), p)).otherwise(
            F.concat(F.split(p, " ").getItem(0), F.lit(" "),
                     F.regexp_replace(F.coalesce(F.split(p, " ").getItem(1), F.lit("")), r"^0+(?=\d)", ""))),
    )
    conv = F.filter(conv, lambda p: F.split(p, " ").getItem(0).isin(TIPOS_DISTINTIVOS))
    return F.array_sort(F.array_distinct(conv))


def resolver_unidades(sigs):
    """Recebe a lista de assinaturas distintas de um grupo_local (strings 'APTO 45;BLOCO B', '' = sem complemento).
    Devolve {assinatura: (unidade, ambiguo)}. Regras na célula de documentação acima."""
    S = [tuple(s.split(";")) if s else () for s in sigs]

    def tipos(sig):
        return {p.split(" ", 1)[0]: p for p in sig}

    def compat(a, b):
        ta, tb = tipos(a), tipos(b)
        return all(ta[t] == tb[t] for t in ta.keys() & tb.keys())

    nao_vazias = [s for s in S if s]
    maximais = [a for a in nao_vazias if not any(set(a) < set(b) for b in nao_vazias)]
    conflita = {a: any(not compat(a, b) for b in maximais if b != a) for a in maximais}
    unit_of = {}
    if maximais and not any(conflita.values()):
        uniao = " | ".join(sorted(set().union(*[set(m) for m in maximais])))
        unit_of = {a: uniao for a in maximais}
    else:
        ancoras = [a for a in maximais if conflita[a]]
        for a in ancoras:
            unit_of[a] = " | ".join(sorted(a))
        for f in maximais:
            if not conflita[f]:
                comp = [a for a in ancoras if compat(f, a)]
                unit_of[f] = unit_of[comp[0]] if len(comp) == 1 else None
    unidades = sorted({u for u in unit_of.values() if u})
    res = {}
    for s in S:
        chave = ";".join(s)
        if not s:
            res[chave] = (unidades[0], False) if len(unidades) == 1 else ("", len(unidades) > 1)
        elif s in unit_of:
            u = unit_of[s]
            res[chave] = (u, False) if u else ("AMBIGUO: " + " | ".join(s), True)
        else:  # assinatura parcial: subconjunto de maximais
            conts = {unit_of.get(m) for m in maximais if set(s) <= set(m)}
            if len(conts) == 1 and None not in conts:
                res[chave] = (conts.pop(), False)
            else:
                res[chave] = ("AMBIGUO: " + " | ".join(s), True)
    return res


# ---- auto-testes (param a célula se a lógica regredir) ------------------------------------------------------------
def _t(sigs, esperado):
    r = resolver_unidades(sigs)
    obtido = {k: v for k, v in r.items()}
    for k, (u, amb) in esperado.items():
        assert obtido[k][1] == amb and (u is None or obtido[k][0] == u), f"resolver_unidades falhou para {sigs}: {k} -> {obtido[k]} (esperado {u}, {amb})"

_t(["APTO 45;BLOCO B", "APTO 45", "BLOCO B", ""], {"APTO 45;BLOCO B": ("APTO 45 | BLOCO B", False), "APTO 45": ("APTO 45 | BLOCO B", False), "BLOCO B": ("APTO 45 | BLOCO B", False), "": ("APTO 45 | BLOCO B", False)})
_t(["APTO 45", "APTO 46", ""], {"APTO 45": ("APTO 45", False), "APTO 46": ("APTO 46", False), "": ("", True)})
_t(["APTO 45", "APTO 46", "BLOCO B"], {"BLOCO B": (None, True)})
_t(["APTO 45;BLOCO A", "APTO 46;BLOCO B", "BLOCO A", ""], {"BLOCO A": ("APTO 45 | BLOCO A", False), "": ("", True)})
_t(["APTO 45", "TORRE 1"], {"APTO 45": ("APTO 45 | TORRE 1", False), "TORRE 1": ("APTO 45 | TORRE 1", False)})
_t(["POSICAO FUNDOS", "POSICAO FRENTE", ""], {"POSICAO FUNDOS": ("POSICAO FUNDOS", False), "POSICAO FRENTE": ("POSICAO FRENTE", False), "": ("", True)})
_t(["KM 12", "KM 15"], {"KM 12": ("KM 12", False), "KM 15": ("KM 15", False)})
_t([""], {"": ("", False)})
_t(["CASA 2", ""], {"CASA 2": ("CASA 2", False), "": ("CASA 2", False)})
info("E6b — auto-testes de resolver_unidades OK")


@F.pandas_udf("string")
def resolver_unidades_udf(sigs_json: pd.Series) -> pd.Series:
    import json as _json
    out = []
    for sj in sigs_json:
        sigs = _json.loads(sj) if sj else []
        res = resolver_unidades(sigs)
        # lista de OBJETOS (não de listas): é o que from_json(array<struct<...>>) consegue ler
        out.append(_json.dumps([{"sig": k, "unidade": v[0], "ambiguo": bool(v[1])} for k, v in res.items()], ensure_ascii=False))
    return pd.Series(out)


def dividir_por_unidade(df):
    """Entrada: variantes com grupo_local e compl_pares. Saída: + unidade, flag_compl_ambiguo, cluster_id (nível 2)."""
    df = df.withColumn("sig_str", F.array_join(assinatura_expr(F.coalesce(F.col("compl_pares"), F.array().cast("array<string>"))), ";"))
    grp = df.groupBy("grupo_local").agg(F.collect_set("sig_str").alias("sigs"))
    grp = grp.withColumn("n_ne", F.size(F.filter(F.col("sigs"), lambda s: s != "")))
    # grupos simples (0 ou 1 assinatura não vazia): resolvidos nativamente — todos entram na única unidade
    unica = F.coalesce(F.get(F.filter(F.col("sigs"), lambda s: s != ""), 0), F.lit(""))
    simples = (grp.filter(F.col("n_ne") <= 1)
               .select("grupo_local", F.explode("sigs").alias("sig_str"), F.regexp_replace(unica, ";", " | ").alias("unidade"), F.lit(False).alias("flag_compl_ambiguo")))
    # grupos com 2+ assinaturas: pandas UDF (fração pequena dos grupos)
    complexos = (grp.filter(F.col("n_ne") >= 2)
                 .withColumn("res", resolver_unidades_udf(F.to_json(F.col("sigs"))))
                 .select("grupo_local", F.explode(F.from_json(F.col("res"), "array<struct<sig:string,unidade:string,ambiguo:boolean>>")).alias("r"))
                 .select("grupo_local", F.col("r.sig").alias("sig_str"), F.col("r.unidade").alias("unidade"), F.col("r.ambiguo").alias("flag_compl_ambiguo")))
    mapa = simples.unionByName(complexos)
    df = df.join(mapa, ["grupo_local", "sig_str"], "left")
    df = df.withColumns({
        "unidade": F.coalesce(F.col("unidade"), F.lit("")),
        "flag_compl_ambiguo": F.coalesce(F.col("flag_compl_ambiguo"), F.lit(False)),
    })
    df = df.withColumn("cluster_id", F.concat(F.col("grupo_local"), F.lit("#"),
                                              F.when(F.col("unidade") == "", F.lit("0")).otherwise(F.substring(F.sha1(F.col("unidade")), 1, 8))))
    return df.drop("sig_str")


if etapa_ativa("E6"):
    t0 = time.time()
    df_gl = obter("05_grupos_locais")
    df_cl = persistir(dividir_por_unidade(df_gl), "05_clusters", cluster_by="idCPF")
    contar(df_cl, "E6b variantes com cluster (nível 2)")
    if CALCULAR_CONTAGENS:
        n_amb = df_cl.filter(F.col("flag_compl_ambiguo")).count()
        n_multi = df_cl.groupBy("grupo_local").agg(F.count_distinct("cluster_id").alias("n")).filter(F.col("n") > 1).count()
        n_grp_2sig = df_cl.groupBy("grupo_local").agg(F.count_distinct(F.when(F.col("unidade") != "", F.col("unidade"))).alias("n")).filter(F.col("n") > 1).count()
        info(f"E6b — grupos locais com 2+ unidades: {n_multi} (com 2+ assinaturas distintas: {n_grp_2sig}) | variantes com complemento ambíguo: {n_amb}")
        display(df_cl.filter(F.col("flag_compl_ambiguo") | (F.col("unidade") != ""))
                .groupBy("grupo_local").agg(F.count_distinct("cluster_id").alias("unidades"), F.sort_array(F.collect_set("unidade")).alias("lista"))
                .filter(F.col("unidades") > 1).limit(10))
    info(f"E6b concluída em {time.time() - t0:.0f}s")
else:
    info("E6b pulada")

## E7 — Eleição do endereço canônico por cluster (voto ponderado por campo)

1. **Valores efetivos** por variante (o que entra na urna): CEP confiável (Correios via CEP ou via fuzzy) → senão CEP que existe e não contradiz → senão bruto; logradouro dos Correios quando bate (mantendo o nome completo do cliente se ele for a versão expandida do reduzido dos Correios) → senão o do cliente limpo; bairro/município/UF dos Correios quando o CEP é confiável → senão os do cliente.
2. **Peso do voto** = `qtd_ocorrencias × (1 + score_qualidade/100) × recência`, multiplicado por um fator por campo (CEP dos Correios pesa 2×, fuzzy 1.6×, bruto 0.5×; logradouro/bairro/cidade/UF respaldados pelos Correios 1.8×).
3. **Urna em formato longo** `(cluster_id, campo, valor, peso, bônus)` → soma por valor → vencedor por `peso/peso_total + bônus` (bônus de completude para logradouro: tipo presente e nome mais longo; bônus 1.0 para valores respaldados pelos Correios) → `pivot` para o formato largo. Um shuffle para todos os campos.
4. **Complemento**: os pares `TIPO VALOR` (`APTO 45`, `BLOCO B`) são votados **por tipo** (`APTO` majoritário, `BLOCO` majoritário...) e remontados na ordem de `ORDEM_COMPL`; a sobra textual (`compl_livre`) é votada como campo. Assim `"APTO 45"` de uma linha e `"BLOCO B"` de outra viram `"BLOCO B APTO 45"`; `"APTO 45"` × `"APTO 54"` decide pela maioria.
5. **Número**: vota primeiro só entre valores numéricos; `SN` só vence se ninguém tiver número. Sufixo de letra votado à parte e concatenado.
6. `metodo_sugestao`, `score_confianca_sugestao` (0–100), `flags_sugestao` (união das flags das variantes), `candidatos_correios` e `variantes_amostra` (usados pelo LLM e para auditoria). Todos os desempates são determinísticos (rank → peso → ordem alfabética) e os arrays são ordenados, então duas execuções produzem a mesma sugestão.

7. **Grau de certeza** (`grau_certeza_cluster`): parte do score (A ≥ 85, B ≥ 65, C abaixo) e aplica **restrições** — cada regra impõe um nível mínimo e registra um motivo em `motivos_revisao`; o pior nível vence. Condições críticas (complemento ambíguo, fuzzy ambíguo, CEP contraditório, concordância muito baixa, sem sugestão) levam direto a `D_REVISAO_HUMANA`. E9 ainda rebaixa por linha (número/logradouro/CEP alterados sem respaldo). Limiares em `LIMIAR_GRAU_ALTA` / `LIMIAR_GRAU_MEDIA`; regras na função, fáceis de estender.

### E7b — consolidação (garantia de uniformidade)

Se o mesmo local tiver escapado em **dois grupos locais** do mesmo CPF (E6 não achou chave em comum, por exemplo erro de digitação sem CEP), as eleições podem divergir. Por isso, depois da 1ª eleição, grupos locais do mesmo CPF cuja **sugestão eleita** descreve o mesmo local (`CEP+número`, `logradouro+número+cidade`, `fonética+número+cidade`, ou as versões sem número) são **fundidos** (mesma propagação de rótulo mínimo, agora sobre grupos; a chave `GL` move juntas todas as unidades de um grupo). Nos grupos fundidos a divisão por unidade (E6b) é **refeita** — apartamentos distintos continuam separados — e os clusters resultantes são **reeleitos** (flag `CLUSTER_CONSOLIDADO`). O mapa linha→cluster consolidado vai para `_05b_clusters_consolidados`, que é o que E9 usa. Resultado: **um endereço (local + unidade) ⇒ um `cluster_id` ⇒ uma única sugestão para todas as suas linhas**.

In [ ]:
# ============================================================================
# E7 — eleição por campo dentro de cada cluster  +  E7b — consolidação de clusters com a mesma sugestão
# ============================================================================
CAMPOS_VOTO = ["cep", "cep_bruto", "logradouro", "numero", "numero_any", "numero_sufixo", "bairro", "cidade", "uf", "compl_livre"]
COLS_SUGESTAO = ["logradouro_sugestao", "numero_sugestao", "complemento_sugestao", "bairro_sugestao", "cidade_sugestao", "uf_sugestao",
                 "cep_parte1_sugestao", "cep_parte2_sugestao"]
COLS_SUG_SAIDA = (["cluster_id", "idCPF", "qtd_variantes_cluster", "qtd_ocorrencias_cluster"] + COLS_SUGESTAO
                  + ["cep_sugestao", "endereco_sugestao_completo", "metodo_sugestao", "score_confianca_sugestao", "concordancias", "flags_sugestao",
                     "candidatos_correios", "variantes_amostra", "tem_cep_correios", "tem_fuzzy", "tem_logradouro_correios", "tem_cep_localidade",
                     "grupo_local", "unidade", "compl_ambiguo", "n_unidades_local",
                     "grau_certeza", "nivel_certeza", "motivos_revisao", "requer_revisao_humana"])
_RE_TIPOS_INICIO = r"^(?:" + "|".join(sorted(set(MAPA_TIPOS_CLIENTE.values()))) + r") "
NOMES_GRAU = {1: "A_ALTA", 2: "B_MEDIA", 3: "C_BAIXA", 4: "D_REVISAO_HUMANA"}


def nome_grau_expr(nivel: Column) -> Column:
    return F.when(nivel <= 1, F.lit("A_ALTA")).when(nivel == 2, F.lit("B_MEDIA")).when(nivel == 3, F.lit("C_BAIXA")).otherwise(F.lit("D_REVISAO_HUMANA"))


def grau_certeza_cluster(df):
    """Escala de certeza no nível do cluster. Parte do score (A/B/C) e aplica RESTRIÇÕES: cada regra impõe um nível
    mínimo (2 = no máximo B, 3 = no máximo C, 4 = revisão humana) e registra o motivo. O pior nível vence.
    Requer: score_confianca_sugestao, metodo_sugestao, flags_sugestao, concordancias (json), qtd_variantes_cluster,
    numero_sugestao, n_unidades_local."""
    flags = F.col("flags_sugestao")
    tem = lambda f: F.array_contains(flags, f)
    metodo = F.col("metodo_sugestao")
    conc_log = F.get_json_object(F.col("concordancias"), "$.logradouro").cast("double")
    conc_num = F.get_json_object(F.col("concordancias"), "$.numero").cast("double")
    qv = F.col("qtd_variantes_cluster")
    num = F.col("numero_sugestao")
    base = (F.when(F.col("score_confianca_sugestao") >= LIMIAR_GRAU_ALTA, F.lit(1))
            .when(F.col("score_confianca_sugestao") >= LIMIAR_GRAU_MEDIA, F.lit(2)).otherwise(F.lit(3)))
    regras = [  # (motivo, condição, nível mínimo imposto)
        ("SEM_SUGESTAO", metodo == "SEM_SUGESTAO", 4),
        ("COMPLEMENTO_AMBIGUO", tem("COMPLEMENTO_AMBIGUO"), 4),
        ("FUZZY_AMBIGUO", (tem("FUZZY_CORREIOS_AMBIGUO") | tem("FUZZY_CEP_AMBIGUO")) & (metodo != "CORREIOS_CEP"), 4),
        ("CEP_CONTRADIZ_ENDERECO", tem("CEP_CONTRADIZ_ENDERECO") & tem("CEP_SUGERIDO_NAO_VALIDADO"), 4),
        ("BAIXA_CONCORDANCIA_LOGRADOURO", (conc_log < 0.4) & (qv >= 3), 4),
        ("BAIXA_CONCORDANCIA_NUMERO", (conc_num < 0.6) & (qv >= 3) & num.isNotNull() & (num != "SN"), 4),
        ("MULTIPLAS_UNIDADES_NO_LOCAL", F.col("n_unidades_local") > 1, 2),
        ("CEP_NAO_VALIDADO_CORREIOS", tem("CEP_SUGERIDO_NAO_VALIDADO"), 3),
        ("CEP_AUSENTE", F.col("cep_sugestao").isNull(), 3),
        ("NUMERO_CONFLITO_NAS_VARIANTES", tem("NUMERO_CONFLITO"), 3),
        ("SEM_NUMERO", num.isNull() | (num == "SN"), 2),
        ("VARIANTE_UNICA_SEM_CORREIOS", metodo == "VARIANTE_UNICA", 3),
        ("CONSENSO_INTERNO_FORTE", (metodo == "CONSENSO_INTERNO") & (conc_log >= 0.8) & (qv >= 3), 2),
        ("CONSENSO_INTERNO_FRACO", (metodo == "CONSENSO_INTERNO") & ~((conc_log >= 0.8) & (qv >= 3)), 3),
        ("LLM_APLICADO", tem("LLM_APLICADO"), 2),
        ("LLM_BAIXA_CONFIANCA", tem("LLM_BAIXA_CONFIANCA"), 3),
    ]
    # ATENÇÃO (performance): NÃO encadear `nivel = when(cond, greatest(nivel, x)).otherwise(nivel)` — cada regra duplicaria a
    # expressão anterior (2^n nós) e o driver trava construindo o plano. greatest(base, regra1, regra2, ...) é linear.
    nivel = F.greatest(base, *[F.when(F.coalesce(cond, F.lit(False)), F.lit(minimo)).otherwise(F.lit(0)) for _, cond, minimo in regras])
    return df.withColumns({
        "nivel_certeza": nivel,
        "motivos_revisao": flags_array(*[(m, cond) for m, cond, _ in regras]),
    }).withColumns({
        "grau_certeza": nome_grau_expr(F.col("nivel_certeza")),
        "requer_revisao_humana": F.col("nivel_certeza") >= 4,
    })


def endereco_completo_expr(cep_col: Column) -> Column:
    return nulo_se_vazio(F.concat_ws(", ",
        F.concat_ws(" ", F.col("logradouro_sugestao"), F.col("numero_sugestao")), F.col("complemento_sugestao"), F.col("bairro_sugestao"),
        F.concat_ws(" - ", F.col("cidade_sugestao"), F.col("uf_sugestao")),
        F.when(cep_col.isNotNull(), F.concat(F.lit("CEP "), F.substring(cep_col, 1, 5), F.lit("-"), F.substring(cep_col, 6, 3)))))


def eleger_por_cluster(dfc):
    """Entrada: variantes com cluster_id (E6). Saída: 1 linha por cluster com a sugestão eleita (determinística)."""
    fz_ok = F.col("fz_status") == "FUZZY_OK"

    # ---- valores efetivos ----------------------------------------------------------------------------------------
    cep_trust = F.coalesce(F.when(F.col("cep_confiavel"), F.col("cep8")), F.col("fz_cep"))
    fonte_cep = (F.when(F.col("cep_confiavel"), F.lit("CORREIOS_CEP")).when(F.col("fz_cep").isNotNull(), F.lit("CORREIOS_FUZZY"))
                 .when(F.col("cep8").isNotNull() & F.col("cep_existe") & ~F.col("cep_contradiz"), F.lit("CORREIOS_EXISTE"))
                 .when(F.col("cep8").isNotNull(), F.lit("BRUTO")))
    cliente_mais_completo_cep = ((F.length(F.col("nome_log")) > F.length(F.col("c_nome_log")))
                                 & ((F.col("nome_red_chave") == F.col("c_nome_red_chave")) | (F.col("nome_chave") == F.col("c_nome_red_oficial_chave"))))
    log_correios_cep = F.concat_ws(" ", F.coalesce(F.col("c_tipo_canon"), F.col("tipo_canon")),
                                   F.when(cliente_mais_completo_cep, F.col("nome_log")).otherwise(F.col("c_nome_log")))
    cliente_mais_completo_fz = (F.length(F.col("nome_log")) > F.length(F.col("fz_nome_log"))) & (F.col("nome_red_chave") == F.col("fz_nome_red_chave"))
    log_fuzzy = F.concat_ws(" ", F.coalesce(F.col("fz_tipo_canon"), F.col("tipo_canon")),
                            F.when(cliente_mais_completo_fz, F.col("nome_log")).otherwise(F.col("fz_nome_log")))
    logradouro_ef = F.when(F.col("logradouro_bate"), log_correios_cep).when(fz_ok, log_fuzzy).otherwise(F.col("logradouro_limpo"))
    fonte_log = F.when(F.col("logradouro_bate"), F.lit("CORREIOS_CEP")).when(fz_ok, F.lit("CORREIOS_FUZZY")).when(F.col("logradouro_limpo").isNotNull(), F.lit("CLIENTE"))
    local_correios = F.col("cep_confiavel") | fz_ok
    df = dfc.withColumns({
        "cep_voto": F.coalesce(cep_trust, F.when(F.col("cep_existe") & ~F.col("cep_contradiz"), F.col("cep8"))),
        "fonte_cep": fonte_cep,
        "logradouro_ef": nulo_se_vazio(logradouro_ef),
        "fonte_log": fonte_log,
        "bairro_ef": F.when(F.col("cep_confiavel") & F.col("c_bairro").isNotNull(), F.col("c_bairro")).when(F.col("fz_bairro").isNotNull(), F.col("fz_bairro")).otherwise(F.col("bairro_n")),
        "cidade_ef": F.when(F.col("cep_confiavel"), F.col("c_municipio")).when(fz_ok, F.col("fz_municipio")).otherwise(F.col("cidade_n")),
        "uf_ef": F.when(F.col("cep_confiavel"), F.col("c_uf")).when(fz_ok, F.col("fz_uf"))
                  .otherwise(F.coalesce(F.col("uf_n"), F.when(F.col("cep_existe") & ~F.col("cep_contradiz"), F.col("c_uf")))),
        "local_correios": local_correios,
        # CEP bruto só concorre se não for sabidamente errado (fora da faixa da UF / contradiz o endereço)
        "cep_bruto_voto": F.when((F.col("cep_faixa_uf_ok") != False) & ~F.col("cep_contradiz"), F.col("cep8")),  # noqa: E712
    })

    # ---- pesos ---------------------------------------------------------------------------------------------------
    if COL_DATA:
        w_cl = Window.partitionBy("cluster_id")
        fator_rec = F.when(F.col("data_max") >= F.date_sub(F.max("data_max").over(w_cl).cast("date"), 365), F.lit(1.25)).otherwise(F.lit(1.0))
    else:
        fator_rec = F.lit(1.0)
    df = df.withColumn("peso", F.col("qtd_ocorrencias") * (1 + F.col("score_qualidade") / 100.0) * fator_rec)
    df = df.withColumns({
        "peso_cep": F.col("peso") * F.when(F.col("fonte_cep") == "CORREIOS_CEP", 2.0).when(F.col("fonte_cep") == "CORREIOS_FUZZY", 1.6)
                                        .when(F.col("fonte_cep") == "CORREIOS_EXISTE", 1.0).otherwise(0.5),
        "peso_log": F.col("peso") * F.when(F.col("fonte_log").isin("CORREIOS_CEP", "CORREIOS_FUZZY"), 1.8).otherwise(1.0),
        "peso_local": F.col("peso") * F.when(F.col("local_correios"), 1.8).otherwise(1.0),
        "bonus_log": F.when(F.col("fonte_log").isin("CORREIOS_CEP", "CORREIOS_FUZZY"), 1.0).otherwise(0.0)
                     + 0.10 * F.when(F.col("logradouro_ef").rlike(_RE_TIPOS_INICIO), 1).otherwise(0)
                     + 0.15 * F.least(F.length(F.coalesce(F.col("logradouro_ef"), F.lit(""))) / 60.0, F.lit(1.0)),
        "bonus_local": F.when(F.col("local_correios"), 0.5).otherwise(0.0),
    })

    # ---- urna em formato longo (1 shuffle para todos os campos) -------------------------------------------------
    def voto(campo, valor, peso_c, bonus=F.lit(0.0)):
        return F.struct(F.lit(campo).alias("campo"), valor.cast("string").alias("valor"), peso_c.cast("double").alias("peso"), bonus.cast("double").alias("bonus"))
    votos_struct = F.array(
        voto("cep", F.col("cep_voto"), F.col("peso_cep")),
        voto("cep_bruto", F.col("cep_bruto_voto"), F.col("peso")),
        voto("logradouro", F.col("logradouro_ef"), F.col("peso_log"), F.col("bonus_log")),
        voto("numero", F.col("numero_int"), F.col("peso")),
        voto("numero_any", F.col("numero_final"), F.col("peso")),
        # sufixo: a AUSÊNCIA também vota (senão "795A" de 1 linha vence "795" de 10)
        voto("numero_sufixo", F.when(F.col("numero_int").isNotNull(), F.coalesce(F.col("numero_sufixo"), F.lit(""))), F.col("peso")),
        voto("bairro", F.col("bairro_ef"), F.col("peso_local"), F.col("bonus_local")),
        voto("cidade", F.col("cidade_ef"), F.col("peso_local"), F.col("bonus_local")),
        voto("uf", F.col("uf_ef"), F.col("peso_local"), F.col("bonus_local")),
        voto("compl_livre", F.col("compl_livre"), F.col("peso")),
    )
    longo = (df.select("cluster_id", F.explode(votos_struct).alias("v")).select("cluster_id", "v.*").filter(F.col("valor").isNotNull()))
    votos = longo.groupBy("cluster_id", "campo", "valor").agg(F.sum("peso").alias("p"), F.max("bonus").alias("bonus"))
    w_tot = Window.partitionBy("cluster_id", "campo")
    votos = votos.withColumn("p_tot", F.sum("p").over(w_tot)).withColumn("conc", F.col("p") / F.col("p_tot"))
    votos = votos.withColumn("rank_val", F.col("conc") + F.col("bonus"))
    # desempate determinístico: rank, peso, e por fim o próprio valor (ordem alfabética)
    w_rk = Window.partitionBy("cluster_id", "campo").orderBy(F.desc("rank_val"), F.desc("p"), F.asc("valor"))
    venc = votos.withColumn("rn", F.row_number().over(w_rk)).filter(F.col("rn") == 1)
    largo = venc.groupBy("cluster_id").pivot("campo", CAMPOS_VOTO).agg(F.first("valor").alias("valor"), F.first("conc").alias("conc"))

    # ---- complemento: voto por tipo de par ---------------------------------------------------------------------------
    pares = df.select("cluster_id", "peso", F.explode(F.col("compl_pares")).alias("par"))
    pares = pares.withColumn("tipo_par", F.split(F.col("par"), " ").getItem(0))
    pv = pares.groupBy("cluster_id", "tipo_par", "par").agg(F.sum("peso").alias("p"))
    pv = pv.withColumn("p_tot", F.sum("p").over(Window.partitionBy("cluster_id", "tipo_par"))).withColumn("conc", F.col("p") / F.col("p_tot"))
    w_par = Window.partitionBy("cluster_id", "tipo_par").orderBy(F.desc("p"), F.asc("par"))
    pv = pv.withColumn("rn", F.row_number().over(w_par)).filter(F.col("rn") == 1)
    pv = pv.withColumn("ordem", F.coalesce(F.try_element_at(ORDEM_COMPL_COL, F.col("tipo_par")), F.lit(999)))
    compl = (pv.groupBy("cluster_id").agg(
        F.array_join(F.transform(F.array_sort(F.collect_list(F.struct("ordem", "par"))), lambda s: s.getField("par")), " ").alias("compl_pares_str"),
        F.min("conc").alias("compl_conc_min"), F.avg("conc").alias("compl_conc")))

    # ---- estatísticas do cluster (arrays ORDENADOS -> saída e prompt do LLM determinísticos) --------------------------
    cand_cep = F.when(F.col("cep_existe"), F.struct(F.col("cep8").alias("cep"), F.col("c_tipo_canon").alias("tipo"), F.col("c_nome_log").alias("logradouro"),
                                                    F.col("c_bairro").alias("bairro"), F.col("c_municipio").alias("municipio"), F.col("c_uf").alias("uf"),
                                                    F.round(F.col("sim_log_cep"), 2).alias("sim"), F.lit("CEP").alias("via")))
    cand_fz = F.when(fz_ok, F.struct(F.col("fz_cep").alias("cep"), F.col("fz_tipo_canon").alias("tipo"), F.col("fz_nome_log").alias("logradouro"),
                                     F.col("fz_bairro").alias("bairro"), F.col("fz_municipio").alias("municipio"), F.col("fz_uf").alias("uf"),
                                     F.round(F.col("fz_score") / 100.0, 2).alias("sim"), F.lit("FUZZY").alias("via")))
    variante_struct = F.struct(F.col("qtd_ocorrencias").alias("qtd"), F.col("logradouro_raw").alias("logradouro"), F.col("numero_raw").alias("numero"),
                               F.col("complemento_raw").alias("complemento"), F.col("bairro_raw").alias("bairro"), F.col("cidade_raw").alias("cidade"),
                               F.col("uf_raw").alias("uf"), F.col("cep_parte1_raw").alias("cep_parte1"), F.col("cep_parte2_raw").alias("cep_parte2"),
                               F.col("score_qualidade").alias("score"))
    stats = df.groupBy("cluster_id").agg(
        F.first("idCPF").alias("idCPF"), F.count(F.lit(1)).alias("qtd_variantes_cluster"), F.sum("qtd_ocorrencias").alias("qtd_ocorrencias_cluster"),
        F.max("score_qualidade").alias("score_qualidade_max"), F.max(F.col("cep_confiavel").cast("int")).alias("tem_cep_correios"),
        F.max(fz_ok.cast("int")).alias("tem_fuzzy"), F.max(F.col("logradouro_bate").cast("int")).alias("tem_logradouro_correios"),
        F.coalesce(F.max((F.col("c_cep_tipo") == "LOCALIDADE").cast("int")), F.lit(0)).alias("tem_cep_localidade"),
        F.max(F.col("numero_int").isNotNull().cast("int")).alias("tem_numero"),
        F.first("grupo_local").alias("grupo_local"), F.first("unidade").alias("unidade"),
        F.coalesce(F.max(F.col("flag_compl_ambiguo").cast("int")), F.lit(0)).alias("compl_ambiguo"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list(F.concat(F.col("flags_norm"), F.col("flags_val"), F.col("flags_fz")))))).alias("flags_sugestao"),
        F.slice(F.sort_array(F.array_distinct(F.array_compact(F.collect_list(cand_cep))), asc=False), 1, 6).alias("cand_cep"),
        F.slice(F.sort_array(F.array_distinct(F.array_compact(F.collect_list(cand_fz))), asc=False), 1, 4).alias("cand_fz"),
        F.slice(F.sort_array(F.collect_list(variante_struct), asc=False), 1, LLM_MAX_VARIANTES_PROMPT).alias("variantes_amostra"),
    ).withColumn("candidatos_correios", F.concat(F.col("cand_cep"), F.col("cand_fz"))).drop("cand_cep", "cand_fz")

    # ---- montagem da sugestão ------------------------------------------------------------------------------------
    sug = stats.join(largo, "cluster_id", "left").join(compl, "cluster_id", "left")
    cep_sug = F.coalesce(F.col("cep_valor"), F.col("cep_bruto_valor"))
    numero_base = F.coalesce(F.col("numero_valor"), F.col("numero_any_valor"))
    sufixo_sug = F.nullif(F.col("numero_sufixo_valor"), F.lit(""))
    numero_sug = F.when(numero_base.rlike(r"^\d+$") & sufixo_sug.isNotNull(), F.concat(numero_base, sufixo_sug)).otherwise(numero_base)
    compl_sug = nulo_se_vazio(F.concat_ws(" ", F.col("compl_pares_str"), F.col("compl_livre_valor")))
    sug = sug.withColumns({
        "logradouro_sugestao": F.col("logradouro_valor"),
        "numero_sugestao": numero_sug,
        "complemento_sugestao": compl_sug,
        "bairro_sugestao": F.col("bairro_valor"),
        "cidade_sugestao": F.col("cidade_valor"),
        "uf_sugestao": F.col("uf_valor"),
        "cep_sugestao": cep_sug,
        "cep_parte1_sugestao": F.substring(cep_sug, 1, 5),
        "cep_parte2_sugestao": F.substring(cep_sug, 6, 3),
        "cep_respaldado_correios": F.col("cep_valor").isNotNull() & ((F.col("tem_cep_correios") == 1) | (F.col("tem_fuzzy") == 1)),
    })
    sug = sug.withColumn("flags_sugestao", F.when(F.col("cep_valor").isNull() & F.col("cep_bruto_valor").isNotNull(),
                                                  F.array_union(F.col("flags_sugestao"), F.array(F.lit("CEP_SUGERIDO_NAO_VALIDADO"))))
                         .otherwise(F.col("flags_sugestao")))
    # cluster ambíguo (sem complemento num prédio com 2+ unidades): não sugere complemento algum
    sug = sug.withColumns({
        "complemento_sugestao": F.when(F.col("compl_ambiguo") == 1, F.lit(None).cast("string")).otherwise(F.col("complemento_sugestao")),
        "flags_sugestao": F.when(F.col("compl_ambiguo") == 1, F.array_union(F.col("flags_sugestao"), F.array(F.lit("COMPLEMENTO_AMBIGUO"))))
                           .otherwise(F.col("flags_sugestao")),
    })
    metodo = (F.when(F.col("logradouro_sugestao").isNull() & F.col("cep_sugestao").isNull(), F.lit("SEM_SUGESTAO"))
              .when(F.col("cep_respaldado_correios") & (F.col("tem_cep_correios") == 1), F.lit("CORREIOS_CEP"))
              .when(F.col("cep_respaldado_correios") & (F.col("tem_fuzzy") == 1), F.lit("CORREIOS_FUZZY"))
              .when(F.col("qtd_variantes_cluster") == 1, F.lit("VARIANTE_UNICA"))
              .otherwise(F.lit("CONSENSO_INTERNO")))
    c = lambda nome: F.coalesce(F.col(nome), F.lit(0.0))
    conf = (
        0.30 * F.when(F.col("tem_cep_correios") == 1, 1.0).when(F.col("tem_fuzzy") == 1, 0.85).otherwise(0.0)
        + 0.20 * F.when(F.col("tem_logradouro_correios") == 1, 1.0).when(F.col("tem_fuzzy") == 1, 0.8).otherwise(0.0)
        + 0.15 * c("logradouro_conc") + 0.10 * F.coalesce(F.col("numero_conc"), F.col("numero_any_conc"), F.lit(0.0)) + 0.10 * c("cep_conc")
        + 0.05 * F.when(F.col("numero_valor").isNotNull(), 1.0).otherwise(0.0)
        + 0.05 * F.when(F.col("uf_valor").isNotNull() & F.col("cidade_valor").isNotNull(), 1.0).otherwise(0.0)
        + 0.05 * F.when(F.col("bairro_valor").isNotNull(), 1.0).otherwise(0.0)
        - 0.10 * F.when(F.array_contains(F.col("flags_sugestao"), "FUZZY_CORREIOS_AMBIGUO") & (F.col("tem_cep_correios") == 0), 1.0).otherwise(0.0)
        - 0.10 * F.when(F.col("compl_ambiguo") == 1, 1.0).otherwise(0.0)
    )
    sug = sug.withColumns({
        "metodo_sugestao": metodo,
        "score_confianca_sugestao": F.round(F.greatest(F.lit(0.0), F.least(F.lit(100.0), conf * 100)), 1),
        "concordancias": F.to_json(F.struct(*[F.round(F.coalesce(F.col(f"{k}_conc"), F.lit(0.0)), 3).alias(k) for k in ["cep", "logradouro", "numero", "bairro", "cidade", "uf"]]
                                             + [F.round(F.coalesce(F.col("compl_conc"), F.lit(0.0)), 3).alias("complemento")])),
    })
    sug = sug.withColumn("endereco_sugestao_completo", endereco_completo_expr(cep_sug))
    # quantas unidades (clusters) existem no mesmo local: 2+ = prédio com mais de um apartamento deste cliente
    sug = sug.withColumn("n_unidades_local", F.count(F.lit(1)).over(Window.partitionBy("grupo_local")))
    sug = grau_certeza_cluster(sug)
    return sug.select(*COLS_SUG_SAIDA)


def consolidar_grupos(sug):
    """E7b (nível 1): grupos locais do MESMO CPF cuja sugestão eleita descreve o mesmo LOCAL (rua+número) -> grupo_final.
    Chaves: CEP+número (só CEP de logradouro), logradouro+número+cidade, fonética+número+cidade e versões sem número.
    A chave 'GL' amarra os clusters (unidades) do mesmo grupo para se moverem juntos. Devolve (grupo_local, grupo_final) só
    para os grupos que mudam. A divisão por unidade (E6b) é refeita nos grupos fundidos, então apartamentos distintos
    continuam separados."""
    num = F.col("numero_sugestao")
    tem = num.isNotNull()
    lchave = chave_tokens(F.col("logradouro_sugestao"))
    lfon = fonetica_tokens(sem_stopwords(tokens(F.col("logradouro_sugestao"))))
    cid = F.coalesce(F.col("cidade_sugestao"), F.lit(""))
    s = sug.select("idCPF", "cluster_id", "grupo_local", "cep_sugestao", "numero_sugestao", "logradouro_sugestao", "cidade_sugestao", "tem_cep_localidade")
    s = s.withColumns({
        "cc_gl": F.concat(F.lit("GL|"), F.col("grupo_local")),
        "cc_cep": F.when(F.col("cep_sugestao").isNotNull() & tem & (F.col("tem_cep_localidade") == 0), F.concat_ws("|", F.lit("CEP"), F.col("cep_sugestao"), num)),
        "cc_log": F.when((lchave != "") & tem, F.concat_ws("|", F.lit("LOG"), lchave, num, cid)),
        "cc_fon": F.when((F.length(lfon) >= 3) & tem, F.concat_ws("|", F.lit("FON"), lfon, num, cid)),
        "cc_log_sn": F.when((lchave != "") & ~tem, F.concat_ws("|", F.lit("LOGSN"), lchave, cid)),
        "cc_cep_sn": F.when(F.col("cep_sugestao").isNotNull() & ~tem & (F.col("tem_cep_localidade") == 0), F.concat_ws("|", F.lit("CEPSN"), F.col("cep_sugestao"), lchave)),
    }).withColumn("rotulo", F.col("grupo_local"))
    chaves = ["cc_gl", "cc_cep", "cc_log", "cc_fon", "cc_log_sn", "cc_cep_sn"]
    for it in range(1, 8):
        s = s.repartition("idCPF")
        novo = F.col("rotulo")
        for k in chaves:
            novo = F.least(novo, F.when(F.col(k).isNotNull(), F.min("rotulo").over(Window.partitionBy("idCPF", k))).otherwise(F.col("rotulo")))
        s2 = persistir(s.withColumn("rotulo_novo", novo), "06a_consol_iter", cluster_by="idCPF")
        mudou = not s2.filter(F.col("rotulo_novo") != F.col("rotulo")).isEmpty()
        s = s2.withColumn("rotulo", F.col("rotulo_novo")).drop("rotulo_novo")
        if not mudou:
            break
    return s.filter(F.col("rotulo") != F.col("grupo_local")).select("grupo_local", F.col("rotulo").alias("grupo_final")).distinct()


# colunas por linha que E9 usa para o grau de certeza no nível da linha (o que a sugestão muda em relação à própria linha)
COLS_CLUSTER_LINHA = ["idCPF", "hash_linha", "cluster_id", "nome_chave", "numero_int", "cep8"]

if etapa_ativa("E7"):
    t0 = time.time()
    dfc = obter("05_clusters")
    sug1 = persistir(eleger_por_cluster(dfc), "06a_sugestao_pass1", cluster_by="idCPF")
    contar(sug1, "E7 clusters eleitos (1ª passada)")

    # ---- E7b: funde grupos locais do mesmo CPF com a mesma sugestão de local, refaz as unidades e reelege só os afetados ----
    mapa = persistir(consolidar_grupos(sug1), "06a_consolidacao")
    if not mapa.isEmpty():
        contar(mapa, "E7b grupos locais fundidos em outro grupo do mesmo CPF (sugestão apontava para o mesmo local)")
        afetados = mapa.select("grupo_local").unionByName(mapa.select(F.col("grupo_final").alias("grupo_local"))).distinct()
        sub = (dfc.join(afetados, "grupo_local", "inner").join(mapa, "grupo_local", "left")
               .withColumn("grupo_local", F.coalesce(F.col("grupo_final"), F.col("grupo_local")))
               .drop("grupo_final", "cluster_id", "unidade", "flag_compl_ambiguo"))
        sub = dividir_por_unidade(sub)   # unidades (apartamentos etc.) recalculadas no grupo fundido
        sug_re = eleger_por_cluster(sub)
        sug_re = sug_re.withColumn("flags_sugestao", F.array_union(F.col("flags_sugestao"), F.array(F.lit("CLUSTER_CONSOLIDADO"))))
        sug_final = sug1.join(afetados, "grupo_local", "left_anti").unionByName(sug_re)
        clusters_final = (dfc.join(afetados, "grupo_local", "left_anti").select(*COLS_CLUSTER_LINHA)
                          .unionByName(sub.select(*COLS_CLUSTER_LINHA)))
    else:
        info("E7b — nenhum grupo a consolidar")
        sug_final, clusters_final = sug1, dfc.select(*COLS_CLUSTER_LINHA)
    df_cl_final = persistir(clusters_final, "05b_clusters_consolidados", cluster_by="idCPF")   # usado por E9
    df_sug = persistir(sug_final, "06_sugestao_cluster", cluster_by="idCPF")
    contar(df_sug, "E7 clusters com sugestão (final)")
    if CALCULAR_CONTAGENS:
        info("E7 — sugestões por método / score médio:")
        display(df_sug.groupBy("metodo_sugestao").agg(F.count(F.lit(1)).alias("clusters"), F.round(F.avg("score_confianca_sugestao"), 1).alias("score_medio"),
                                                       F.sum("qtd_ocorrencias_cluster").alias("linhas")).orderBy(F.desc("clusters")))
        info("E7 — grau de certeza por cluster (nível do cluster; E9 ainda pode rebaixar por linha):")
        display(df_sug.groupBy("grau_certeza").agg(F.count(F.lit(1)).alias("clusters"), F.sum("qtd_ocorrencias_cluster").alias("linhas")).orderBy("grau_certeza"))
        display(df_sug.select(F.explode("motivos_revisao").alias("motivo")).groupBy("motivo").count().orderBy(F.desc("count")))
    info(f"E7 concluída em {time.time() - t0:.0f}s")
    display(df_sug.select("cluster_id", "unidade", "qtd_variantes_cluster", "endereco_sugestao_completo", "metodo_sugestao", "score_confianca_sugestao", "concordancias").limit(15))
    if CALCULAR_CONTAGENS:
        info("E7 — exemplos de prédios/locais com 2+ unidades no mesmo CPF (devem ter sugestões distintas):")
        multi = df_sug.groupBy("grupo_local").agg(F.count(F.lit(1)).alias("n")).filter(F.col("n") > 1).select("grupo_local")
        display(df_sug.join(multi, "grupo_local").orderBy("grupo_local", "cluster_id")
                .select("grupo_local", "cluster_id", "unidade", "complemento_sugestao", "endereco_sugestao_completo", "flags_sugestao").limit(12))
else:
    info("E7 pulada")

## E8 — Fallback com LLM (amostra de clusters de baixa confiança)

Só roda com `USAR_LLM=True`. Seleciona até `LIMITE_LLM` clusters com `score_confianca_sugestao < LIMIAR_CONFIANCA_PARA_LLM` (os mais frequentes primeiro) e pede ao modelo o endereço canônico em **JSON estruturado**, dando: as variantes brutas do cluster (até `LLM_MAX_VARIANTES_PROMPT`, com a frequência de cada uma), os candidatos dos Correios (via CEP e via fuzzy) e a sugestão atual do pipeline.

- **Provedores** (`PROVEDOR_LLM`): `anthropic` (SDK `anthropic` ≥ 1.x, saída estruturada via `output_config.format`, `effort=medium`), `openai` / `azure_openai` / `databricks` (SDK `openai`; para Databricks Model Serving use `ENDPOINT_LLM=https://<workspace>/serving-endpoints` e o token do workspace) e `mock` (não chama API: ecoa a sugestão do pipeline com confiança 0.9 — serve para testar o encanamento, o cache e a sobrescrita sem custo).
- **Modelo**: default `claude-sonnet-5` (pedido do projeto). `claude-opus-5` é a opção mais forte; se usar Opus 5 / Fable 5.1, considere `client.beta.messages.create(..., betas=["server-side-fallback-2026-07-01"], fallbacks="default")` para tratar recusas de classificador automaticamente.
- **Credencial**: `dbutils.secrets.get(SEGREDO_SCOPE, SEGREDO_CHAVE_LLM)` → variáveis de ambiente `LLM_API_KEY` / `ANTHROPIC_API_KEY` / `OPENAI_API_KEY` / `DATABRICKS_TOKEN`. Nunca em texto no notebook.
- **Cache** em `_07_llm_cache` (chave = hash do prompt + modelo): reexecuções não pagam de novo.
- **Segurança**: o prompt **não** contém o CPF nem qualquer identificador do cliente — só as strings de endereço.
- **Aplicação**: a resposta só sobrescreve a sugestão do cluster se `confianca ≥ LIMIAR_CONFIANCA_LLM`; nesses casos `metodo_sugestao = 'LLM'` e o score vira `confianca × 100`. Resultado em `_06b_sugestao_cluster_llm`.
- **Escala**: para rodar em milhões de clusters, troque o loop no driver por `ai_query()` do Databricks SQL (pseudocódigo comentado no fim da célula) ou pelo Batch API do provedor.

In [ ]:
# ============================================================================
# E8 — fallback LLM em amostra (com cache, sem CPF no prompt)
# ============================================================================
import concurrent.futures, threading

ESQUEMA_JSON_LLM = {
    "type": "object",
    "properties": {
        "logradouro": {"type": ["string", "null"], "description": "Tipo + nome por extenso, maiúsculas sem acento. Ex.: AVENIDA MARECHAL DEODORO DA FONSECA"},
        "numero": {"type": ["string", "null"], "description": "Só dígitos, ou 'SN' se sem número"},
        "complemento": {"type": ["string", "null"], "description": "Ex.: BLOCO B APTO 45. null se não houver"},
        "bairro": {"type": ["string", "null"]},
        "cidade": {"type": ["string", "null"]},
        "uf": {"type": ["string", "null"], "description": "Sigla com 2 letras"},
        "cep": {"type": ["string", "null"], "description": "8 dígitos sem hífen, ou null se não for possível determinar"},
        "confianca": {"type": "number", "description": "0 a 1"},
        "justificativa": {"type": "string", "description": "Uma frase"},
    },
    "required": ["logradouro", "numero", "complemento", "bairro", "cidade", "uf", "cep", "confianca", "justificativa"],
    "additionalProperties": False,
}

PROMPT_SISTEMA_LLM = (
    "Você é um especialista em endereços postais do Brasil (padrão DNE dos Correios). "
    "Recebe várias variantes do MESMO endereço de um cliente, escritas de formas diferentes, incompletas ou com erros, "
    "mais candidatos da base oficial dos Correios e a sugestão atual de um pipeline automático. "
    "Devolva o endereço canônico mais provável seguindo as regras: "
    "(1) nunca invente número ou complemento que não apareça nas variantes; "
    "(2) prefira CEP, logradouro, bairro, município e UF de um candidato dos Correios quando ele for compatível com as variantes; "
    "(3) logradouro por extenso com o tipo (RUA, AVENIDA, TRAVESSA...), sem abreviações, em maiúsculas sem acento; "
    "(4) número só com dígitos, ou 'SN'; complemento com palavras-chave padronizadas (APTO, BLOCO, CASA, SALA, LOTE, QUADRA, FUNDOS); "
    "(5) se as variantes descrevem endereços claramente diferentes, escolha o mais frequente e reduza a confiança; "
    "(6) confiança entre 0 e 1 refletindo o quanto os dados sustentam a resposta. Responda apenas com o JSON pedido."
)


def montar_prompt_llm(linha) -> str:
    partes = ["VARIANTES DO ENDERECO (qtd = vezes que apareceu):"]
    for v in (linha["variantes_amostra"] or []):
        cep = "-".join([p for p in [v["cep_parte1"], v["cep_parte2"]] if p]) or "null"
        partes.append(f"- (x{v['qtd']}) logradouro={v['logradouro']!r} | numero={v['numero']!r} | complemento={v['complemento']!r} | "
                      f"bairro={v['bairro']!r} | cidade={v['cidade']!r} | uf={v['uf']!r} | cep={cep}")
    partes.append("\nCANDIDATOS DOS CORREIOS:")
    cands = linha["candidatos_correios"] or []
    if not cands:
        partes.append("- (nenhum)")
    for c in cands:
        partes.append(f"- CEP {c['cep']} | {c['tipo'] or ''} {c['logradouro'] or ''} | bairro {c['bairro'] or '?'} | {c['municipio'] or '?'}/{c['uf'] or '?'} | similaridade {c['sim']} | via {c['via']}")
    partes.append("\nSUGESTAO ATUAL DO PIPELINE: " + (linha["endereco_sugestao_completo"] or "(nenhuma)")
                  + f" | metodo={linha['metodo_sugestao']} | confianca={linha['score_confianca_sugestao']}")
    partes.append("\nProduza o JSON com o endereco canonico.")
    return "\n".join(partes)


def obter_segredo_llm():
    if dbutils is not None:
        try:
            return dbutils.secrets.get(SEGREDO_SCOPE, SEGREDO_CHAVE_LLM)
        except Exception as e:
            info(f"E8 — segredo {SEGREDO_SCOPE}/{SEGREDO_CHAVE_LLM} indisponível ({type(e).__name__}); tentando variáveis de ambiente")
    for var in ("LLM_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "DATABRICKS_TOKEN"):
        if os.environ.get(var):
            return os.environ[var]
    return None


def criar_cliente_llm(chave):
    if PROVEDOR_LLM == "mock":  # dry-run do encanamento sem chamar API: devolve a própria sugestão do pipeline com confiança 0.9
        return None
    if PROVEDOR_LLM == "anthropic":
        import anthropic
        return anthropic.Anthropic(api_key=chave) if chave else anthropic.Anthropic()
    from openai import OpenAI  # openai | azure_openai | databricks (endpoint compatível)
    kwargs = {}
    if chave:
        kwargs["api_key"] = chave
    if ENDPOINT_LLM:
        kwargs["base_url"] = ENDPOINT_LLM
    elif PROVEDOR_LLM == "databricks":
        host = spark.conf.get("spark.databricks.workspaceUrl", None)
        if host:
            kwargs["base_url"] = f"https://{host}/serving-endpoints"
    return OpenAI(**kwargs)


def _extrair_json(texto: str):
    try:
        return json.loads(texto)
    except Exception:
        m = re.search(r"\{.*\}", texto or "", re.S)
        return json.loads(m.group(0)) if m else None


def chamar_llm(cliente, prompt_usuario: str):
    """Uma chamada, com saída estruturada; devolve dict ou None (recusa/erro de formato)."""
    if PROVEDOR_LLM == "mock":
        m = re.search(r"SUGESTAO ATUAL DO PIPELINE: (.*?) \| metodo=", prompt_usuario)
        partes = [p.strip() for p in (m.group(1) if m else "").split(",")]
        log_num = partes[0].rsplit(" ", 1) if partes and partes[0] else ["", ""]
        cid_uf = (partes[-2] if len(partes) >= 2 else "").split(" - ")
        cep = next((p.replace("CEP ", "").replace("-", "") for p in partes if p.startswith("CEP ")), None)
        return {"logradouro": log_num[0] or None, "numero": log_num[1] if len(log_num) > 1 else None,
                "complemento": None, "bairro": None, "cidade": cid_uf[0] or None, "uf": cid_uf[1] if len(cid_uf) > 1 else None,
                "cep": cep, "confianca": 0.9, "justificativa": "mock: eco da sugestão do pipeline"}
    if PROVEDOR_LLM == "anthropic":
        resp = cliente.messages.create(
            model=MODELO_LLM, max_tokens=2048, system=PROMPT_SISTEMA_LLM,
            messages=[{"role": "user", "content": prompt_usuario}],
            output_config={"effort": "medium", "format": {"type": "json_schema", "schema": ESQUEMA_JSON_LLM}},
        )
        if getattr(resp, "stop_reason", None) == "refusal":
            return None
        texto = next((b.text for b in resp.content if getattr(b, "type", "") == "text"), None)
        return _extrair_json(texto) if texto else None
    msgs = [{"role": "system", "content": PROMPT_SISTEMA_LLM}, {"role": "user", "content": prompt_usuario}]
    try:
        resp = cliente.chat.completions.create(
            model=MODELO_LLM, messages=msgs, temperature=0,
            response_format={"type": "json_schema", "json_schema": {"name": "endereco_canonico", "schema": ESQUEMA_JSON_LLM, "strict": True}},
        )
    except Exception:  # endpoints sem suporte a json_schema (alguns modelos do Model Serving)
        resp = cliente.chat.completions.create(model=MODELO_LLM, messages=msgs, temperature=0, response_format={"type": "json_object"})
    return _extrair_json(resp.choices[0].message.content)


def chamar_llm_com_retry(cliente, prompt: str, tentativas: int = 4):
    for i in range(tentativas):
        try:
            return chamar_llm(cliente, prompt), None
        except Exception as e:
            erro = f"{type(e).__name__}: {str(e)[:200]}"
            time.sleep(min(30, 2 ** i + 0.5))
    return None, erro


if etapa_ativa("E8") and USAR_LLM:
    t0 = time.time()
    sug = obter("06_sugestao_cluster")
    alvo = (sug.filter(F.col("score_confianca_sugestao") < LIMIAR_CONFIANCA_PARA_LLM)
            .orderBy(F.desc("qtd_ocorrencias_cluster"), F.asc("cluster_id")).limit(LIMITE_LLM))
    linhas = [r.asDict(recursive=True) for r in alvo.select("cluster_id", "variantes_amostra", "candidatos_correios", "endereco_sugestao_completo",
                                                            "metodo_sugestao", "score_confianca_sugestao").collect()]
    info(f"E8 — {len(linhas)} clusters selecionados para o LLM ({PROVEDOR_LLM}/{MODELO_LLM})")

    # cache
    nome_cache = nome_intermediario("07_llm_cache", com_lote=False)
    cache = {}
    if PERSISTIR_INTERMEDIARIOS and tabela_existe(nome_cache):
        cache = {r["hash_prompt"]: r["resposta_json"] for r in spark.table(nome_cache).filter(F.col("modelo") == MODELO_LLM).select("hash_prompt", "resposta_json").collect()}
        info(f"E8 — cache com {len(cache)} respostas para {MODELO_LLM}")
    cache.update(ARTEFATOS.get("_llm_cache_mem", {}))

    pedidos = []
    for ln in linhas:
        prompt = montar_prompt_llm(ln)
        h = hashlib.sha256((MODELO_LLM + "\n" + PROMPT_SISTEMA_LLM + "\n" + prompt).encode("utf-8")).hexdigest()
        pedidos.append((ln["cluster_id"], h, prompt))
    faltam = [p for p in pedidos if p[1] not in cache]
    info(f"E8 — {len(pedidos) - len(faltam)} no cache, {len(faltam)} chamadas a fazer")

    novos = []
    if faltam:
        cliente = criar_cliente_llm(obter_segredo_llm())
        trava = threading.Lock()
        def _job(p):
            cid, h, prompt = p
            resp, erro = chamar_llm_com_retry(cliente, prompt)
            return cid, h, resp, erro
        with concurrent.futures.ThreadPoolExecutor(max_workers=LLM_CONCORRENCIA) as ex:
            for i, (cid, h, resp, erro) in enumerate(ex.map(_job, faltam), 1):
                if resp is not None:
                    with trava:
                        cache[h] = json.dumps(resp, ensure_ascii=False)
                        novos.append((h, PROVEDOR_LLM, MODELO_LLM, cid, cache[h], datetime.datetime.now()))
                elif erro:
                    info(f"E8 — falha no cluster {cid}: {erro}")
                if i % 50 == 0:
                    info(f"E8 — {i}/{len(faltam)} chamadas")
        if novos:
            df_novos = spark.createDataFrame(novos, "hash_prompt string, provedor string, modelo string, cluster_id string, resposta_json string, ts timestamp")
            if PERSISTIR_INTERMEDIARIOS:
                df_novos.write.format("delta").mode("append").saveAsTable(nome_cache)
                info(f"E8 — {len(novos)} respostas gravadas em {nome_cache}")
            ARTEFATOS.setdefault("_llm_cache_mem", {}).update({n[0]: n[4] for n in novos})

    # aplica
    respostas = []
    for cid, h, _ in pedidos:
        if h in cache:
            try:
                d = json.loads(cache[h])
            except Exception:
                continue
            respostas.append((cid, d.get("logradouro"), d.get("numero"), d.get("complemento"), d.get("bairro"), d.get("cidade"), d.get("uf"),
                              d.get("cep"), float(d.get("confianca") or 0.0), d.get("justificativa")))
    esquema_llm = ("cluster_id string, llm_logradouro string, llm_numero string, llm_complemento string, llm_bairro string, llm_cidade string, "
                   "llm_uf string, llm_cep string, llm_confianca double, llm_justificativa string")
    df_llm = spark.createDataFrame(respostas, esquema_llm) if respostas else spark.createDataFrame([], esquema_llm)
    df_llm = df_llm.withColumns({
        "llm_logradouro": nulo_se_vazio(norm_chave(F.col("llm_logradouro"))),
        "llm_numero": nulo_se_vazio(F.upper(F.regexp_replace(F.col("llm_numero"), r"[^0-9A-Za-z]", ""))),
        "llm_complemento": nulo_se_vazio(norm_chave(F.col("llm_complemento"))),
        "llm_bairro": nulo_se_vazio(norm_local(F.col("llm_bairro"))),
        "llm_cidade": nulo_se_vazio(norm_local(F.col("llm_cidade"))),
        "llm_uf": F.try_element_at(MAPA_UF_COL, norm_chave(F.col("llm_uf"))),
        "llm_cep": F.when(F.length(F.regexp_replace(F.coalesce(F.col("llm_cep"), F.lit("")), r"[^0-9]", "")) == 8,
                          F.regexp_replace(F.col("llm_cep"), r"[^0-9]", "")),
    })
    sug2 = sug.join(df_llm, "cluster_id", "left")
    aplica = F.col("llm_confianca") >= LIMIAR_CONFIANCA_LLM
    sug2 = sug2.withColumns({
        "logradouro_sugestao": F.when(aplica & F.col("llm_logradouro").isNotNull(), F.col("llm_logradouro")).otherwise(F.col("logradouro_sugestao")),
        "numero_sugestao": F.when(aplica & F.col("llm_numero").isNotNull(), F.col("llm_numero")).otherwise(F.col("numero_sugestao")),
        "complemento_sugestao": F.when(aplica, F.col("llm_complemento")).otherwise(F.col("complemento_sugestao")),
        "bairro_sugestao": F.when(aplica & F.col("llm_bairro").isNotNull(), F.col("llm_bairro")).otherwise(F.col("bairro_sugestao")),
        "cidade_sugestao": F.when(aplica & F.col("llm_cidade").isNotNull(), F.col("llm_cidade")).otherwise(F.col("cidade_sugestao")),
        "uf_sugestao": F.when(aplica & F.col("llm_uf").isNotNull(), F.col("llm_uf")).otherwise(F.col("uf_sugestao")),
        "cep_sugestao": F.when(aplica & F.col("llm_cep").isNotNull(), F.col("llm_cep")).otherwise(F.col("cep_sugestao")),
        "metodo_sugestao": F.when(aplica, F.lit("LLM")).otherwise(F.col("metodo_sugestao")),
        "score_confianca_sugestao": F.when(aplica, F.round(F.col("llm_confianca") * 100, 1)).otherwise(F.col("score_confianca_sugestao")),
        "flags_sugestao": F.when(aplica, F.array_union(F.col("flags_sugestao"), F.array(F.lit("LLM_APLICADO"))))
                           .when(F.col("llm_confianca").isNotNull(), F.array_union(F.col("flags_sugestao"), F.array(F.lit("LLM_BAIXA_CONFIANCA"))))
                           .otherwise(F.col("flags_sugestao")),
    })
    sug2 = sug2.withColumns({
        "cep_parte1_sugestao": F.substring(F.col("cep_sugestao"), 1, 5),
        "cep_parte2_sugestao": F.substring(F.col("cep_sugestao"), 6, 3),
    })
    sug2 = sug2.withColumn("endereco_sugestao_completo", nulo_se_vazio(F.concat_ws(", ",
        F.concat_ws(" ", F.col("logradouro_sugestao"), F.col("numero_sugestao")), F.col("complemento_sugestao"), F.col("bairro_sugestao"),
        F.concat_ws(" - ", F.col("cidade_sugestao"), F.col("uf_sugestao")),
        F.when(F.col("cep_sugestao").isNotNull(), F.concat(F.lit("CEP "), F.col("cep_parte1_sugestao"), F.lit("-"), F.col("cep_parte2_sugestao"))))))
    sug2 = grau_certeza_cluster(sug2.drop("nivel_certeza", "motivos_revisao", "grau_certeza", "requer_revisao_humana"))  # recalcula a escala após o LLM
    df_sug_llm = persistir(sug2, "06b_sugestao_cluster_llm", cluster_by="idCPF")
    if CALCULAR_CONTAGENS:
        display(df_sug_llm.filter(F.col("llm_confianca").isNotNull()).groupBy(aplica.alias("aplicado")).count())
    info(f"E8 concluída em {time.time() - t0:.0f}s")
    display(df_sug_llm.filter(F.col("llm_confianca").isNotNull())
            .select("cluster_id", "endereco_sugestao_completo", "metodo_sugestao", "score_confianca_sugestao", "llm_justificativa").limit(15))
else:
    info("E8 pulada (USAR_LLM=False ou etapa fora de ETAPAS_A_EXECUTAR)")

# ---------------------------------------------------------------------------------------------------------------
# PSEUDOCÓDIGO — versão em escala com ai_query() do Databricks SQL (Model Serving), sem loop no driver:
#
# df_alvo = obter("06_sugestao_cluster").filter(...).withColumn("prompt", <mesma montagem do prompt em Spark SQL>)
# df_alvo.createOrReplaceTempView("llm_alvo")
# spark.sql("""
#   SELECT cluster_id,
#          ai_query('databricks-claude-sonnet-4',              -- endpoint do Model Serving
#                   concat('<PROMPT_SISTEMA>', '\n', prompt),
#                   responseFormat => '{"type":"json_schema","json_schema":{"name":"endereco","schema":<ESQUEMA_JSON_LLM>,"strict":true}}')
#            AS resposta_json
#   FROM llm_alvo
# """).write.format("delta").mode("append").saveAsTable(nome_intermediario("07_llm_cache", com_lote=False))
# ---------------------------------------------------------------------------------------------------------------

## E9 — Montagem final (linhas originais + `_sugestao`), escrita e métricas

1. Relê `tb_endereco` com **o mesmo escopo** de E1 e recalcula `hash_linha`.
2. Join com `_05b_clusters_consolidados` (`idCPF`, `hash_linha` → `cluster_id`) e com a sugestão por cluster (`_06b` se o LLM rodou, senão `_06`). Como a sugestão é **por cluster**, todas as linhas de um cluster recebem exatamente os mesmos valores nas colunas `_sugestao`; só as `flag_alterou_*` variam por linha (comparam cada original com a sugestão).
   **Check de uniformidade** (impresso e gravado em `_metricas`): número de clusters com mais de uma sugestão (tem de ser 0 por construção) e número de CPFs com dois clusters distintos e sugestão idêntica (tem de ser 0 após E7b).
3. Devolve as colunas originais **com os nomes originais**, as `<coluna>_sugestao`, o endereço completo, `cluster_id`, `unidade`, `n_unidades_local`, contagens do cluster, `metodo_sugestao`, `score_confianca_sugestao`, **`grau_certeza`, `requer_revisao_humana`, `motivos_revisao`**, `flags_sugestao`, `concordancias` (JSON) e `flag_alterou_<campo>` (comparação normalizada entre original e sugestão).
   O grau vem do cluster (E7/E8) e é **rebaixado por linha** quando a sugestão contradiz esta linha: `NUMERO_ALTERADO` (a linha tinha número e a sugestão traz outro), `LOGRADOURO_ALTERADO_SEM_RESPALDO` (rua diferente sem Correios/LLM), `CEP_ALTERADO_SEM_RESPALDO` (troca o CEP por um não validado). Esses casos vão para `D_REVISAO_HUMANA`.
4. Escreve `<PREFIXO_SAIDA>_final` (Delta, `overwrite` no lote 0 / `append` nos demais, Liquid Clustering por `idCPF`) e uma linha em `<PREFIXO_SAIDA>_metricas`.
5. Se `MODO_TESTE_SINTETICO`, compara com o gabarito: pureza dos clusters, completude (um endereço verdadeiro → um cluster) e acerto por campo.

Amostra de auditoria: 30 linhas de clusters com mais de uma variante, lado a lado (original × sugestão).

In [ ]:
# ============================================================================
# E9 — montagem final, escrita e métricas
# ============================================================================
if etapa_ativa("E9"):
    t0 = time.time()
    usa_llm = USAR_LLM and ("06b_sugestao_cluster_llm" in ARTEFATOS or tabela_existe(nome_intermediario("06b_sugestao_cluster_llm")))
    sug = obter("06b_sugestao_cluster_llm" if usa_llm else "06_sugestao_cluster")
    cols_diag = ["qtd_variantes_cluster", "qtd_ocorrencias_cluster", "endereco_sugestao_completo", "metodo_sugestao",
                 "score_confianca_sugestao", "concordancias", "flags_sugestao", "nivel_certeza", "motivos_revisao", "unidade", "n_unidades_local"]
    sug = sug.select("cluster_id", *COLS_SUGESTAO, *cols_diag)
    # mapa linha -> cluster: usa a versão CONSOLIDADA de E7b (mesmo endereço eleito => mesmo cluster)
    usa_05b = "05b_clusters_consolidados" in ARTEFATOS or tabela_existe(nome_intermediario("05b_clusters_consolidados"))
    clusters = obter("05b_clusters_consolidados" if usa_05b else "05_clusters").select(*COLS_CLUSTER_LINHA)
    orig = ler_endereco_bruto()
    final = orig.join(clusters, ["idCPF", "hash_linha"], "left").join(sug, "cluster_id", "left")
    final = final.withColumns({
        "metodo_sugestao": F.coalesce(F.col("metodo_sugestao"), F.lit("SEM_SUGESTAO")),
        "score_confianca_sugestao": F.coalesce(F.col("score_confianca_sugestao"), F.lit(0.0)),
        "flags_sugestao": F.coalesce(F.col("flags_sugestao"), F.array().cast("array<string>")),
        "nivel_certeza": F.coalesce(F.col("nivel_certeza"), F.lit(4)),
        "motivos_revisao": F.coalesce(F.col("motivos_revisao"), F.array(F.lit("SEM_SUGESTAO"))),
    })
    # ---- grau de certeza no nível da LINHA: o que a sugestão muda em relação ao que ESTA linha dizia ----------------
    cep_sug_col = F.concat_ws("", F.col("cep_parte1_sugestao"), F.col("cep_parte2_sugestao"))
    num_sug_int = F.regexp_extract(F.coalesce(F.col("numero_sugestao"), F.lit("")), r"^(\d+)", 1).try_cast("bigint")
    sim_log_linha = sim_jaccard(tokens(F.col("nome_chave")), tokens(chave_tokens(F.col("logradouro_sugestao"))))
    respaldo = F.col("metodo_sugestao").isin("CORREIOS_CEP", "CORREIOS_FUZZY", "LLM")
    regras_linha = [
        ("NUMERO_ALTERADO", F.col("numero_int").isNotNull() & num_sug_int.isNotNull() & (F.col("numero_int") != num_sug_int), 4),
        ("LOGRADOURO_ALTERADO_SEM_RESPALDO", (F.coalesce(F.col("nome_chave"), F.lit("")) != "") & (sim_log_linha < 0.34) & ~respaldo, 4),
        ("CEP_ALTERADO_SEM_RESPALDO", F.col("cep8").isNotNull() & (F.nullif(cep_sug_col, F.lit("")) != F.col("cep8"))
                                      & F.array_contains(F.col("flags_sugestao"), "CEP_SUGERIDO_NAO_VALIDADO"), 4),
        # a linha não tinha número (nulo/S/N) e recebeu o número do cluster: inferência plausível, mas inferência -> no máximo B
        ("NUMERO_PREENCHIDO_PELO_CLUSTER", F.col("numero_int").isNull() & num_sug_int.isNotNull(), 2),
    ]
    # greatest(...) linear — nunca encadear when/otherwise reutilizando a expressão (explosão 2^n do plano)
    nivel_linha = F.greatest(F.col("nivel_certeza"), *[F.when(F.coalesce(cond, F.lit(False)), F.lit(minimo)).otherwise(F.lit(0)) for _, cond, minimo in regras_linha])
    final = final.withColumns({
        "nivel_certeza": nivel_linha,
        "motivos_revisao": F.array_union(F.col("motivos_revisao"), flags_array(*[(m, cond) for m, cond, _ in regras_linha])),
    })
    final = final.withColumns({
        "grau_certeza": nome_grau_expr(F.col("nivel_certeza")),
        "requer_revisao_humana": F.col("nivel_certeza") >= 4,
    })
    # flags de alteração (comparação normalizada)
    cep_orig = F.regexp_replace(F.concat_ws("", F.coalesce(F.col("cep_parte1"), F.lit("")), F.coalesce(F.col("cep_parte2"), F.lit(""))), r"[^0-9]", "")
    cep_sug = F.concat_ws("", F.col("cep_parte1_sugestao"), F.col("cep_parte2_sugestao"))
    final = final.withColumns({
        "flag_alterou_logradouro": F.col("logradouro_sugestao").isNotNull() & (norm_chave(F.col("logradouro")) != F.col("logradouro_sugestao")),
        "flag_alterou_numero": F.col("numero_sugestao").isNotNull() & (F.regexp_replace(norm_chave(F.col("numero")), r"[^0-9A-Z]", "") != F.col("numero_sugestao")),
        "flag_alterou_complemento": F.coalesce(norm_chave(F.col("complemento")), F.lit("")) != F.coalesce(F.col("complemento_sugestao"), F.lit("")),
        "flag_alterou_bairro": F.col("bairro_sugestao").isNotNull() & (norm_local(F.col("bairro")) != F.col("bairro_sugestao")),
        "flag_alterou_cidade": F.col("cidade_sugestao").isNotNull() & (norm_local(F.col("cidade")) != F.col("cidade_sugestao")),
        "flag_alterou_uf": F.col("uf_sugestao").isNotNull() & (F.coalesce(F.try_element_at(MAPA_UF_COL, norm_chave(F.col("uf"))), F.lit("")) != F.col("uf_sugestao")),
        "flag_alterou_cep": F.nullif(cep_sug, F.lit("")).isNotNull() & (cep_orig != cep_sug),
        "id_execucao": F.lit(ID_EXECUCAO),
    })
    # nomes originais de volta
    sel = []
    for interno, real in COLS_ENDERECO.items():
        if real:
            sel.append(F.col(interno).alias(real))
    if COL_DATA:
        sel.append(F.col("data_ref").alias(COL_DATA))
    for interno in CAMPOS_ENDERECO:
        real = COLS_ENDERECO.get(interno) or interno
        sel.append(F.col(f"{interno}_sugestao").alias(f"{real}_sugestao"))
    sel += [F.col(c) for c in ["endereco_sugestao_completo", "cluster_id", "unidade", "n_unidades_local", "qtd_variantes_cluster", "qtd_ocorrencias_cluster",
                               "metodo_sugestao", "score_confianca_sugestao", "grau_certeza", "requer_revisao_humana", "motivos_revisao",
                               "concordancias", "flags_sugestao"]]
    sel += [F.col(f"flag_alterou_{c}") for c in ["logradouro", "numero", "complemento", "bairro", "cidade", "uf", "cep"]]
    sel += [F.col("hash_linha"), F.col("id_execucao")]
    df_final = final.select(*sel)

    nome_final = nome_intermediario("final", com_lote=False)
    if ESCREVER_SAIDA:
        modo = "append" if (N_LOTES > 1 and LOTE_ATUAL > 0) else "overwrite"
        w = df_final.write.format("delta").mode(modo)
        if modo == "overwrite":
            w = w.option("overwriteSchema", "true")
        try:
            w = w.clusterBy("idCPF") if COLS_ENDERECO["idCPF"] == "idCPF" else w.clusterBy(COLS_ENDERECO["idCPF"])
        except Exception:
            pass
        w.saveAsTable(nome_final)
        df_final = spark.table(nome_final).filter(F.col("id_execucao") == ID_EXECUCAO) if modo == "append" else spark.table(nome_final)
        info(f"E9 — saída gravada em {nome_final} ({modo})")
    else:
        df_final = df_final.cache()
        info("E9 — ESCREVER_SAIDA=False: saída mantida em memória (df_final)")
    ARTEFATOS["final"] = df_final

    # ---- métricas -----------------------------------------------------------------------------------------------
    if CALCULAR_CONTAGENS:
        b = lambda c: F.avg(F.when(F.coalesce(c, F.lit(False)), 1.0).otherwise(0.0))
        m = df_final.agg(
            F.count(F.lit(1)).alias("linhas"), F.count_distinct(F.col(COLS_ENDERECO["idCPF"])).alias("cpfs"), F.count_distinct("cluster_id").alias("clusters"),
            F.round(F.avg("score_confianca_sugestao"), 2).alias("score_medio"),
            *[F.round(b(F.col(f"flag_alterou_{c}")), 4).alias(f"pct_alterou_{c}") for c in ["logradouro", "numero", "complemento", "bairro", "cidade", "uf", "cep"]],
        ).collect()[0].asDict()
        por_metodo = {r["metodo_sugestao"]: r["n"] for r in df_final.groupBy("metodo_sugestao").agg(F.count(F.lit(1)).alias("n")).collect()}
        por_grau = {r["grau_certeza"]: r["n"] for r in df_final.groupBy("grau_certeza").agg(F.count(F.lit(1)).alias("n")).collect()}
        info(f"E9 — linhas por grau de certeza: {json.dumps(dict(sorted(por_grau.items())), ensure_ascii=False)}")
        info("E9 — motivos de revisão mais frequentes (linhas):")
        display(df_final.select(F.explode("motivos_revisao").alias("motivo")).groupBy("motivo").count().orderBy(F.desc("count")))
        m_extra = {"por_grau": json.dumps(por_grau), "pct_revisao_humana": round(por_grau.get("D_REVISAO_HUMANA", 0) / max(1, sum(por_grau.values())), 4)}
        # ---- CHECK DE UNIFORMIDADE: (a) todas as linhas de um cluster têm a MESMA sugestão;
        #                              (b) nenhum CPF tem dois clusters com sugestão idêntica (E7b deve ter fundido)
        cols_sug_reais = [f"{(COLS_ENDERECO.get(cc) or cc)}_sugestao" for cc in CAMPOS_ENDERECO]
        com_cluster = df_final.filter(F.col("cluster_id").isNotNull())
        n_nao_uniformes = (com_cluster.groupBy("cluster_id").agg(F.count_distinct(F.struct(*cols_sug_reais)).alias("n"))
                           .filter(F.col("n") > 1).count())
        n_dup_entre_clusters = (com_cluster.select(COLS_ENDERECO["idCPF"], "cluster_id", *cols_sug_reais).distinct()
                                .groupBy(COLS_ENDERECO["idCPF"], *cols_sug_reais).agg(F.count_distinct("cluster_id").alias("n"))
                                .filter(F.col("n") > 1).count())
        info(f"E9 — CHECK DE UNIFORMIDADE: clusters com mais de uma sugestão = {n_nao_uniformes} (esperado 0) | "
             f"CPFs com clusters distintos e sugestão idêntica = {n_dup_entre_clusters} (esperado 0)")
        m.update({"check_clusters_nao_uniformes": n_nao_uniformes, "check_sugestoes_iguais_em_clusters_distintos": n_dup_entre_clusters})
        m.update(m_extra)
        m.update({"por_metodo": json.dumps(por_metodo), "id_execucao": ID_EXECUCAO, "ts": datetime.datetime.now(), "lote": LOTE_ATUAL,
                  "parametros": json.dumps({"MODO_AMOSTRA": MODO_AMOSTRA, "FRACAO_AMOSTRA": FRACAO_AMOSTRA, "N_LOTES": N_LOTES,
                                            "LIMIAR_SIM_LOGRADOURO": LIMIAR_SIM_LOGRADOURO, "LIMIAR_FUZZY_CORREIOS": LIMIAR_FUZZY_CORREIOS,
                                            "USAR_FUZZY_CORREIOS": USAR_FUZZY_CORREIOS, "USAR_LLM": USAR_LLM, "MODELO_LLM": MODELO_LLM})})
        info("E9 — métricas: " + json.dumps({k: (str(v) if not isinstance(v, (int, float)) else v) for k, v in m.items()}, ensure_ascii=False))
        if ESCREVER_SAIDA:
            spark.createDataFrame([m]).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(nome_intermediario("metricas", com_lote=False))

    # ---- avaliação contra o gabarito sintético -------------------------------------------------------------------
    if MODO_TESTE_SINTETICO:
        gab = spark.table("tb_gabarito_sintetico")
        gab = gab.withColumn("hash_linha", F.xxhash64(*[F.coalesce(F.col(c), F.lit("<NULO>")) for c in ["idCPF"] + CAMPOS_ENDERECO]))
        gab = gab.select("idCPF", "hash_linha", "id_end_verdade", "logradouro_verdade", "numero_verdade", "cep_verdade", "bairro_verdade", "cidade_verdade",
                         "uf_verdade", "complemento_verdade")
        av = df_final.join(gab, ["idCPF", "hash_linha"], "inner").join(clusters.select("idCPF", "hash_linha", "numero_int"), ["idCPF", "hash_linha"], "left")
        amb = F.array_contains(F.col("flags_sugestao"), "COMPLEMENTO_AMBIGUO")
        # FUSÕES INDEVIDAS: clusters não ambíguos em que linhas COM número misturam 2+ endereços verdadeiros (ex.: dois apartamentos) -> tem de ser 0
        fusoes = (av.filter(~amb & F.col("numero_int").isNotNull()).groupBy("cluster_id").agg(F.count_distinct("id_end_verdade").alias("n_verd"))
                  .filter(F.col("n_verd") > 1).count())
        # ANEXAÇÕES S/N ERRADAS: linhas sem número anexadas a um cluster cujo endereço verdadeiro majoritário é outro (inferência que falhou)
        verd_cl = av.filter(F.col("numero_int").isNotNull()).groupBy("cluster_id", "id_end_verdade").count() \
            .withColumn("rn", F.row_number().over(Window.partitionBy("cluster_id").orderBy(F.desc("count")))).filter("rn = 1") \
            .select("cluster_id", F.col("id_end_verdade").alias("verd_majoritaria"))
        anex_erradas = (av.filter(F.col("numero_int").isNull() & ~amb).join(verd_cl, "cluster_id", "inner")
                        .filter(F.col("id_end_verdade") != F.col("verd_majoritaria")).count())
        pur = av.filter(~amb).groupBy("cluster_id").agg(F.count_distinct("id_end_verdade").alias("n_verd")).agg(F.avg(F.when(F.col("n_verd") == 1, 1.0).otherwise(0.0)).alias("p")).collect()[0]["p"]
        comp = av.groupBy("id_end_verdade").agg(F.count_distinct("cluster_id").alias("n_cl")).agg(F.avg(F.when(F.col("n_cl") == 1, 1.0).otherwise(0.0)).alias("c")).collect()[0]["c"]
        comp_na = av.filter(~amb).groupBy("id_end_verdade").agg(F.count_distinct("cluster_id").alias("n_cl")).agg(F.avg(F.when(F.col("n_cl") == 1, 1.0).otherwise(0.0)).alias("c")).collect()[0]["c"]
        tok = lambda c: F.array_sort(tokens(norm_chave(c)))
        acertos = av.agg(
            F.avg(F.when(F.concat_ws("", F.col("cep_parte1_sugestao"), F.col("cep_parte2_sugestao")) == F.col("cep_verdade"), 1.0).otherwise(0.0)).alias("acerto_cep"),
            F.avg(F.when(F.col("numero_sugestao") == F.col("numero_verdade"), 1.0).otherwise(0.0)).alias("acerto_numero"),
            F.avg(F.when(chave_tokens(F.col("logradouro_sugestao")) == chave_tokens(F.col("logradouro_verdade")), 1.0).otherwise(0.0)).alias("acerto_logradouro"),
            F.avg(F.when(F.col("bairro_sugestao") == F.col("bairro_verdade"), 1.0).otherwise(0.0)).alias("acerto_bairro"),
            F.avg(F.when(F.col("cidade_sugestao") == F.col("cidade_verdade"), 1.0).otherwise(0.0)).alias("acerto_cidade"),
            F.avg(F.when(F.col("uf_sugestao") == F.col("uf_verdade"), 1.0).otherwise(0.0)).alias("acerto_uf"),
            F.avg(F.when(F.col("complemento_verdade").isNotNull() & ~amb, F.when(tok(F.col("complemento_sugestao")) == tok(F.col("complemento_verdade")), 1.0).otherwise(0.0))).alias("acerto_complemento_nao_ambiguo"),
            F.avg(F.when(amb, 1.0).otherwise(0.0)).alias("pct_linhas_complemento_ambiguo"),
        ).collect()[0].asDict()
        info(f"E9 — AVALIAÇÃO SINTÉTICA: fusoes_indevidas={fusoes} (tem de ser 0) anexacoes_sem_numero_erradas={anex_erradas} (informativo) "
             f"pureza_cluster={pur:.3f} completude={comp:.3f} completude_sem_ambiguos={comp_na:.3f} " + " ".join(f"{k}={v:.3f}" for k, v in acertos.items()))
        if fusoes or anex_erradas:
            info("E9 — clusters não ambíguos com 2+ endereços verdadeiros (fusões e anexações S/N erradas) — variantes envolvidas:")
            cl_fus = av.filter(~amb).groupBy("cluster_id").agg(F.count_distinct("id_end_verdade").alias("n_verd")).filter(F.col("n_verd") > 1).select("cluster_id")
            display(av.join(cl_fus, "cluster_id").orderBy("cluster_id", "id_end_verdade")
                    .select("cluster_id", "id_end_verdade", "complemento_verdade", "logradouro", "numero", "complemento", "unidade", "complemento_sugestao", "grau_certeza", "motivos_revisao").limit(30))
        info("E9 — exemplos de prédios com 2 unidades (linha original x sugestão):")
        multi = av.groupBy("idCPF", "cep_verdade", "numero_verdade").agg(F.count_distinct("id_end_verdade").alias("n")).filter(F.col("n") > 1).select("idCPF", "cep_verdade", "numero_verdade")
        display(av.join(multi, ["idCPF", "cep_verdade", "numero_verdade"]).orderBy("idCPF", "id_end_verdade")
                .select("idCPF", "id_end_verdade", "complemento_verdade", "logradouro", "numero", "complemento", "cluster_id", "complemento_sugestao", "flags_sugestao").limit(16))
        info("E9 — exemplos de erro de CEP (sugestão x verdade):")
        display(av.filter(F.concat_ws("", F.col("cep_parte1_sugestao"), F.col("cep_parte2_sugestao")) != F.col("cep_verdade"))
                .select("logradouro", "numero", "cep_parte1", "cep_parte2", "endereco_sugestao_completo", "cep_verdade", "metodo_sugestao", "flags_sugestao").limit(10))
        info("E9 — exemplos de erro de logradouro:")
        display(av.filter(chave_tokens(F.col("logradouro_sugestao")) != chave_tokens(F.col("logradouro_verdade")))
                .select("logradouro", "logradouro_sugestao", "logradouro_verdade", "metodo_sugestao").limit(10))

    info(f"E9 concluída em {time.time() - t0:.0f}s")
    info("E9 — amostra de auditoria (clusters com mais de uma variante):")
    display(df_final.filter(F.col("qtd_variantes_cluster") > 1).orderBy("cluster_id")
            .select(COLS_ENDERECO["idCPF"], "cluster_id", "logradouro", "numero", "complemento", "bairro", "cidade", "uf", "cep_parte1", "cep_parte2",
                    "endereco_sugestao_completo", "metodo_sugestao", "score_confianca_sugestao", "grau_certeza").limit(30))
    info("E9 — amostra para o time de revisão humana (grau D):")
    display(df_final.filter(F.col("requer_revisao_humana")).orderBy(F.desc("qtd_ocorrencias_cluster"), "cluster_id")
            .select(COLS_ENDERECO["idCPF"], "cluster_id", "logradouro", "numero", "complemento", "cep_parte1", "cep_parte2",
                    "endereco_sugestao_completo", "grau_certeza", "motivos_revisao").limit(20))
else:
    info("E9 pulada")

## Depois de rodar — como validar, calibrar e estender

### Validação manual (recomendado antes da base completa)
1. Na amostra de auditoria de E9, olhe 50–100 clusters com `metodo_sugestao = CONSENSO_INTERNO` e `score < 60`: são os casos em que os Correios não ajudaram. Se muitos forem "logradouro certo, CEP errado", o problema está no blocking de E5 (cidade escrita de outra forma) — amplie `ALIASES_CIDADE` ou reduza `LIMIAR_FUZZY_CORREIOS`.
2. Olhe clusters com `qtd_variantes_cluster ≥ 5` e `concordancias.numero < 0.6`: número disputado — normalmente são variantes com número embutido no logradouro que o parser leu errado; adicione o padrão em `RE_NUM_FIM` / `MARCADORES_NUMERO`.
3. Confira a lista de **tipos não mapeados** (E3) e de **tokens de complemento fora do dicionário** (E2) e estenda `MAPA_TIPOS_CORREIOS` / `MAPA_COMPL`.
4. Se aparecer `LOGRADOURO_NAO_BATE_CEP` em massa para uma cidade, verifique se `tb_correios.logradouro_correios` está vindo com o tipo na frente ou em forma reduzida (a dim `_dim_correios_logradouro` mostra `nome_log` × `nome_red_chave`).

### Pseudocódigo — um endereço por CPF (se precisar)
```python
# A saída é por cluster (um CPF pode ter 2+ endereços legítimos). Para escolher UM por CPF:
sug = spark.table(nome_intermediario("06_sugestao_cluster"))
w = Window.partitionBy("idCPF").orderBy(
    F.desc("qtd_ocorrencias_cluster"),           # mais frequente...
    F.desc("score_confianca_sugestao"),           # ...e mais confiável
    # F.desc("data_max_cluster"),                 # se houver COL_DATA: agregue max(data_max) por cluster em E7 e use aqui
)
um_por_cpf = sug.withColumn("rk", F.row_number().over(w)).filter("rk = 1").drop("rk")
```

### Reexecução parcial
- Quebrou em E6? Rode com `ETAPAS_A_EXECUTAR = "E6,E7,E8,E9"` — E6 lê `_04b_variantes_enriquecidas` persistida.
- Mudou um dicionário? Reexecute a partir de E2 (ou E3, se foi `MAPA_TIPOS_CORREIOS`).
- Base completa: `MODO_AMOSTRA=false`, `N_LOTES=10`, rode `LOTE_ATUAL=0` com `ETAPAS_A_EXECUTAR` completo (gera as dims) e os lotes 1..9 com `"E1,E2,E4,E5,E6,E7,E8,E9"` (E3 é compartilhada).

### Smoke test local (Windows/Linux, sem Databricks)
```bash
pip install "pyspark>=4.1,<4.2" rapidfuzz pandas pyarrow
# Java 17 no PATH/JAVA_HOME. Depois, em Python, com o notebook convertido em script (jupyter nbconvert --to script):
END_MODO_TESTE_SINTETICO=true END_PERSISTIR_INTERMEDIARIOS=false END_ESCREVER_SAIDA=false END_CATALOGO= END_SCHEMA_ORIGEM= END_MODO_AMOSTRA=false python sugestao_endereco_cadastro_databricks.py
```
A etapa E9 imprime `pureza_cluster`, `completude` e o acerto por campo contra o gabarito sintético.

### Ideias de extensão (não implementadas)
- Geocodificação (lat/long) como camada adicional de desambiguação entre CEPs do mesmo logradouro.
- Modelo de classificação (LightGBM) para prever "variante correta" usando os indicadores de E4 como features, treinado com a amostra auditada.
- Dicionário de bairros × município dos Correios para corrigir o campo bairro mesmo quando o CEP falha.